# AF3 to predict structure

## Prepare sequence json file

In [4]:
import pandas as pd

In [7]:
com = pd.read_csv('./tf/tf_tf/specific_filtered_1888.csv', index_col=0)
com

,motif_motif,tf_tf,cell_type,score,tf1,tf2,tf1_exp,tf2_exp,tf1_ratio,tf2_ratio
51,AHR_EBF1,AHR_EBF1,b-cell_lineage,0.033615,AHR,EBF1,2.141253,3.226498,1.023211,1.682020
143,AHR_HD/18,AHR_HOXA10,b-cell_lineage,0.028071,AHR,HOXA10,2.141253,1.228779,1.023211,1.805473
226,AHR_KAISO,AHR_ZBTB33,macrophages,0.030570,AHR,ZBTB33,2.706927,1.157291,1.320989,1.005461
227,AHR_KAISO,ARNT_ZBTB33,macrophages,0.030570,ARNT,ZBTB33,1.766330,1.157291,1.017879,1.005461
251,AHR_MEF2,AHR_MEF2D,b-cell_lineage,0.074874,AHR,MEF2D,2.141253,1.932744,1.023211,1.043751
...,...,...,...,...,...,...,...,...,...,...
59386,ZNF423_ZNF528,ZNF423_ZNF528,b-cell_lineage,0.057566,ZNF423,ZNF528,0.998403,0.998403,2.673602,1.059738
59397,ZNF431_ZNF784,ZNF431_ZNF784,b-cell_lineage,0.017117,ZNF431,ZNF784,2.023548,0.542801,1.060633,1.009132
59415,ZNF528_ZNF768,ZNF528_ZNF768,b-cell_lineage,0.140536,ZNF528,ZNF768,0.998403,0.952665,1.059738,1.295477
59425,ZNF554_ZNF784,ZNF554_ZNF784,b-cell_lineage,0.054391,ZNF554,ZNF784,0.776673,0.542801,1.277811,1.009132


In [8]:
import pandas as pd
import requests
import json
import os
import time
from tqdm import tqdm # Used to display progress bars; install via pip install tqdm first

# --- Configuration Section ---

# 1. Set the NCBI Taxonomy ID of the organism
ORGANISM_ID = "9606" # Human (Homo sapiens)

# 2. Folder name for output JSON files
OUTPUT_DIR = "alphafold3_inputs"


# --- Function Definitions ---

# Global variable to cache queried sequences to avoid repeated network requests
sequence_cache = {}

def get_protein_sequence(gene_name: str, organism_id: str) -> str | None:
    """
    Fetches protein sequence by gene name and organism ID via UniProt API.
    Prioritizes reviewed (Swiss-Prot) sequences.
    """
    if gene_name in sequence_cache:
        return sequence_cache[gene_name]

    url = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        'query': f'(gene:{gene_name}) AND (organism_id:{organism_id}) AND (reviewed:true)',
        'format': 'fasta',
        'size': 1 
    }

    try:
        time.sleep(0.5) 
        response = requests.get(url, params=params)
        response.raise_for_status()

        fasta_data = response.text
        if not fasta_data:
            print(f"Warning: Protein sequence for gene '{gene_name}' not found.")
            return None

        lines = fasta_data.strip().split('\n')
        sequence = "".join(lines[1:])
        
        if sequence:
            sequence_cache[gene_name] = sequence
            return sequence
        else:
            print(f"Warning: No valid sequence returned for gene '{gene_name}'.")
            return None

    except requests.exceptions.RequestException as e:
        print(f"Error: Network error occurred while requesting sequence for gene '{gene_name}': {e}")
        return None

# ==============================================================================
#  vvv Main Modified Function vvv
# ==============================================================================
def create_alphafold3_json(gene1: str, gene2: str, sequence1: str, sequence2: str, output_path: str):
    """
    Creates and saves the JSON file required by AlphaFold Server API based on gene names and protein sequences.
    """
    # Construct input structure compliant with AlphaFold Server API v1
    af3_input = {
        "name": f"{gene1}_{gene2}",
        "modelSeeds": [1], # Model seeds, default is [1]
        "sequences": [
            {
                "protein": {
                    "id": "A", # Chain A
                    "sequence": sequence1
                }
            },
            {
                "protein": {
                    "id": "B", # Chain B
                    "sequence": sequence2
                }
            }
        ],
        "dialect": "alphafold3",
        "version": 1
    }
    
    try:
        with open(output_path, 'w') as f:
            json.dump(af3_input, f, indent=2)
    except IOError as e:
        print(f"Error: Unable to write to file {output_path}: {e}")
# ==============================================================================
#  ^^^ Main Modified Function ^^^
# ==============================================================================


# --- Main Program ---

def main():
    # --- 1. Load your DataFrame ---
    # Replace this with your own data loading method
    # Example: com = pd.read_csv('your_data.csv')
    
    # For demonstration, we create a sample DataFrame
    # data = {
    #     'tf1': ['TP53', 'MYC', 'JUN', 'INVALID_GENE'],
    #     'tf2': ['MDM2', 'MAX', 'FOS', 'TP53']
    # }
    # com = pd.DataFrame(data)
    # print("Loaded DataFrame example:")
    # print(com)
    # print("-" * 30)

    # --- 2. Create output directory ---
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
        print(f"Creating output directory: {OUTPUT_DIR}")

    # --- 3. Iterate through DataFrame and generate files ---
    print("Starting sequence retrieval and JSON generation...")
    
    for index, row in tqdm(com.iterrows(), total=com.shape[0], desc="Processing gene pairs"):
        gene1 = row['tf1']
        gene2 = row['tf2']

        if not gene1 or not gene2 or pd.isna(gene1) or pd.isna(gene2):
            print(f"Skipping row {index+1} because gene name is empty.")
            continue

        seq1 = get_protein_sequence(gene1, ORGANISM_ID)
        seq2 = get_protein_sequence(gene2, ORGANISM_ID)

        if seq1 and seq2:
            filename = f"{gene1}_{gene2}.json"
            output_file_path = os.path.join(OUTPUT_DIR, filename)
            # Update function call, passing gene names
            create_alphafold3_json(gene1, gene2, seq1, seq2, output_file_path)
        else:
            print(f"Skipping gene pair ({gene1}, {gene2}) because sequence for one or both could not be retrieved.")
    
    print("-" * 30)
    print("Processing complete! ✅")
    print(f"Generated JSON files have been saved in the '{OUTPUT_DIR}' folder.")


if __name__ == '__main__':
    main()

创建输出目录: alphafold3_inputs
开始获取序列并生成JSON文件...


处理基因对: 100%|██████████| 1888/1888 [10:19<00:00,  3.05it/s]

------------------------------
处理完成！✅
生成的JSON文件已保存在 'alphafold3_inputs' 文件夹中。


## AF3 predict

In [ ]:
#!/bin/bash

APPDIR="/pool1/tmp/qiaolibo/beifen/af3"  # Replace with your actual path if different
ALPHAFOLD3DIR="$APPDIR/alphafold3"
#HMMER3_BINDIR="/usr/bin" # Path to HMMER binaries (**installed via OS package manager or specify your path**)
HMMER3_BINDIR="/home/qiaolibo/miniconda3/envs/Alphafold3/bin/" # Path to Conda binarys (**installed via conda**)
DB_DIR="${APPDIR}/database"
MODEL_DIR="${APPDIR}/model"
WORK_DIR=$(pwd)
OUTPUT_DIR="${WORK_DIR}/output/"
LOG_FILE="${WORK_DIR}/af3_run.log"

CUDA_VISIBLE_DEVICES=7 /home/qiaolibo/miniconda3/envs/Alphafold3/bin/python ${ALPHAFOLD3DIR}/run_alphafold.py \
    --jackhmmer_binary_path="${HMMER3_BINDIR}/jackhmmer" \
    --nhmmer_binary_path="${HMMER3_BINDIR}/nhmmer" \
    --hmmalign_binary_path="${HMMER3_BINDIR}/hmmalign" \
    --hmmsearch_binary_path="${HMMER3_BINDIR}/hmmsearch" \
    --hmmbuild_binary_path="${HMMER3_BINDIR}/hmmbuild" \
    --db_dir="${DB_DIR}" \
    --model_dir="${MODEL_DIR}" \
    --input_dir="${WORK_DIR}/json_input/" \
    --output_dir="${OUTPUT_DIR}" \
    --buckets="256,512,768,1024,1280,1536,2048,2560,3072,3584,4096,4608,5120" \
    --jax_compilation_cache_dir="$APPDIR/jax_compilation"
    2>&1 | tee -a "${LOG_FILE}"

## Check the contents of all folders

In [66]:
from pathlib import Path

# --- Please modify the main folder path you want to check here ---
# Windows Example: main_folder_path = r'C:\Users\YourUser\Documents\Projects'
# macOS/Linux Example: main_folder_path = '/home/user/data/experiments'
main_folder_path = './output/cured/'
# -----------------------------------------

# List to store details of non-compliant folders
# Each element will be a dictionary containing name, count, and list of inner folders
non_compliant_folders = []
folders_checked = 0

path = Path(main_folder_path)

print(f"--- Starting check for folder: {path} ---")

if not path.is_dir():
    print(f"Error: Path '{path}' does not exist or is not a directory.")
else:
    # Iterate through all first-level subfolders under the main folder
    for item in path.iterdir():
        if item.is_dir():
            folders_checked += 1
            
            # Use list comprehension to get names of all inner folders at once
            inner_dirs = [sub_item.name for sub_item in item.iterdir() if sub_item.is_dir()]
            
            # The count of inner folders is the length of this list
            subfolder_count = len(inner_dirs)
            
            # Check if the count is not equal to 5
            if subfolder_count != 5:
                # Record details: name, actual count, and list of inner folder names
                non_compliant_folders.append({
                    "name": item.name,
                    "count": subfolder_count,
                    "subfolders": inner_dirs
                })

    # --- After the loop, report the final detailed results ---
    print(f"\n--- Check Completed ---")
    print(f"Total subfolders checked: {folders_checked}.")

    if not non_compliant_folders:
        if folders_checked > 0:
            print("\nConclusion: ✅ All subfolders contain exactly 5 folders.")
        else:
            print("\nConclusion: ⚠️ No subfolders found in the main directory to check.")
    else:
        print(f"\nConclusion: ❌ Found {len(non_compliant_folders)} non-compliant folders (inner folder count is not 5). Details below:")
        # Iterate through the list and print details for each non-compliant folder
        for folder_info in non_compliant_folders:
            print("-" * 20)
            print(f"  Folder '{folder_info['name']}' contains {folder_info['count']} folders (Expected 5):")
            
            if folder_info['subfolders']:
                # Print names of all its inner folders
                for subfolder_name in folder_info['subfolders']:
                    print(f"    - {subfolder_name}")
            else:
                print("    (This folder does not contain any subfolders)")
        print("-" * 20)

--- 开始检查文件夹: output/cured ---

--- 检查完成 ---
总共检查了 572 个子文件夹。

结论: ✅ 所有子文件夹内的文件夹数量都是 5 个。


## Computing Interaction Interface

In [17]:
import pathlib
import os
import re
import pandas as pd
from tqdm import tqdm

# BioPython related libraries
from Bio.PDB import MMCIFParser, NeighborSearch

# --- Configuration Parameters ---
MAIN_FOLDER_PATH = pathlib.Path("./output/spe/")
DISTANCE_THRESHOLD = 5.0
OUTPUT_CSV_FILE = "spe_tf_interaction_residues_summary.csv"

# ---- Core Modification 1: Add amino acid 3-letter to 1-letter mapping dictionary ----
AMINO_ACID_MAP = {
    'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
    'GLU': 'E', 'GLN': 'Q', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
    'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
    'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'
}

def find_interface_from_cif(cif_file_path, dist_thresh):
    """
    Analyzes a single CIF file to identify interacting residues.
    Returns a tuple: (Set of Chain 1 residue strings, Set of Chain 2 residue strings).
    """
    parser = MMCIFParser(QUIET=True)
    try:
        structure = parser.get_structure("protein_complex", cif_file_path)
    except Exception:
        return set(), set()

    model = structure[0]
    chain_ids = [chain.id for chain in model]
    if len(chain_ids) < 2:
        return set(), set()
    
    chain_1_id, chain_2_id = chain_ids[0], chain_ids[1]

    chain1_atoms = [atom for atom in model[chain_1_id].get_atoms()]
    chain2_atoms = [atom for atom in model[chain_2_id].get_atoms()]

    if not chain1_atoms or not chain2_atoms:
        return set(), set()

    # Find residues in Chain 1 close to Chain 2
    ns_chain2 = NeighborSearch(chain2_atoms)
    interface_residues_chain1 = set()
    for atom1 in chain1_atoms:
        if ns_chain2.search(atom1.get_coord(), dist_thresh, 'A'):
            interface_residues_chain1.add(atom1.get_parent())

    # Find residues in Chain 2 close to Chain 1
    ns_chain1 = NeighborSearch(chain1_atoms)
    interface_residues_chain2 = set()
    for atom2 in chain2_atoms:
        if ns_chain1.search(atom2.get_coord(), dist_thresh, 'A'):
            interface_residues_chain2.add(atom2.get_parent())

    # ---- Core Modification 2: Format using the mapping dictionary ----
    # If a non-standard amino acid is encountered, use 'X' as the default value
    set_res1_formatted = {
        f"{AMINO_ACID_MAP.get(res.get_resname(), 'X')}{res.get_id()[1]}" 
        for res in interface_residues_chain1
    }
    set_res2_formatted = {
        f"{AMINO_ACID_MAP.get(res.get_resname(), 'X')}{res.get_id()[1]}" 
        for res in interface_residues_chain2
    }

    return set_res1_formatted, set_res2_formatted

def sort_key_for_seed_folders(folder_path):
    """Helper function to sort sample folders numerically."""
    match = re.search(r'sample-(\d+)$', folder_path.name)
    if match:
        return int(match.group(1))
    return -1

def main():
    if not MAIN_FOLDER_PATH.is_dir():
        print(f"Error: Directory '{MAIN_FOLDER_PATH}' does not exist.")
        return

    tf_pair_dirs = [d for d in MAIN_FOLDER_PATH.iterdir() if d.is_dir()]
    all_results = []

    print(f"Starting analysis of {len(tf_pair_dirs)} TF pairs (Outputting single-letter format)...")
    
    for tf_pair_dir in tqdm(tf_pair_dirs, desc="Processing TF Pairs"):
        # ---- Core Modification 3: Convert folder name (row index) to uppercase ----
        pair_name = tf_pair_dir.name.upper()
        
        current_row = {"TF_Pair": pair_name}
        
        # Find and sort seed directories
        seed_dirs = sorted(
            [d for d in tf_pair_dir.glob("seed-1_sample-*") if d.is_dir()],
            key=sort_key_for_seed_folders
        )
        
        # Iterate through the first 5 samples
        for i in range(5):
            col_name_tf1 = f"TF1_seed{i+1}"
            col_name_tf2 = f"TF2_seed{i+1}"

            tf1_res_str, tf2_res_str = "", ""

            if i < len(seed_dirs):
                seed_dir = seed_dirs[i]
                cif_files = list(seed_dir.glob("*.cif"))
                if cif_files:
                    set_res1, set_res2 = find_interface_from_cif(cif_files[0], DISTANCE_THRESHOLD)
                    
                    if set_res1:
                        # Sort residues numerically (e.g., A2, A10) based on position
                        res_list1_sorted = sorted(list(set_res1), key=lambda r: int(re.findall(r'\d+', r)[0]))
                        tf1_res_str = ", ".join(res_list1_sorted)
                    
                    if set_res2:
                        res_list2_sorted = sorted(list(set_res2), key=lambda r: int(re.findall(r'\d+', r)[0]))
                        tf2_res_str = ", ".join(res_list2_sorted)
                else:
                    tf1_res_str = "CIF file not found"
                    tf2_res_str = "CIF file not found"
            else:
                tf1_res_str = "Sample folder missing"
                tf2_res_str = "Sample folder missing"
            
            current_row[col_name_tf1] = tf1_res_str
            current_row[col_name_tf2] = tf2_res_str
        
        all_results.append(current_row)

    # Convert results to DataFrame
    final_df = pd.DataFrame(all_results)
    final_df = final_df.set_index("TF_Pair")
    final_df.to_csv(OUTPUT_CSV_FILE)
    
    print("\nAnalysis complete!")
    print(f"Results saved to file: '{OUTPUT_CSV_FILE}'")
    print("\nTable Preview:")
    print(final_df.head())

if __name__ == "__main__":
    main()

开始分析 1888 个转录因子对 (输出为单字母格式)...


处理转录因子对: 100%|██████████| 1888/1888 [41:50<00:00,  1.33s/it]


分析完成！
结果已保存到文件: 'spe_tf_interaction_residues_summary.csv'

表格预览:
                                                      TF1_seed1  \
TF_Pair                                                           
ARID3B_EWSR1  R198, E201, D205, K208, R238, I241, K244, I246...   
ARID3B_GMEB2  K244, S283, I284, T285, S286, F289, T290, R292...   
ARID3B_RUNX3  E99, D100, E101, D102, R198, E201, I202, D205,...   
ARID5A_HIC2   P43, E50, Q53, E57, G100, A101, E103, L104, T1...   
AHR_EBF1      M1, S3, L54, P55, F56, P57, I61, L72, Y76, A79...   

                                                      TF2_seed1  \
TF_Pair                                                           
ARID3B_EWSR1  D347, L348, G349, P350, P351, V352, D353, E356...   
ARID3B_GMEB2  D98, T138, T218, E220, D221, P222, G223, D224,...   
ARID3B_RUNX3  P72, N73, M110, A111, G112, N113, D114, E115, ...   
ARID5A_HIC2   L64, K104, R137, K140, R141, Q193, A221, G222,...   
AHR_EBF1      A26, T29, W30, M31, Q32, G33, A34, G35, E55, K..

## Count the number of transcription factors and their amino acid lengths

In [68]:
import json
import pandas as pd
from pathlib import Path

# --- Configuration Parameters ---
# Path to the input folder
INPUT_FOLDER = Path('./json_input/alphafold3_inputs_1888_spe/')

# --- Main Program ---
def main():
    """
    Main function to execute file traversal, parsing, and DataFrame creation.
    """
    if not INPUT_FOLDER.is_dir():
        print(f"Error: Input folder '{INPUT_FOLDER}' does not exist. Please check the path.")
        return

    # Get all files ending with .json in the folder
    json_files = list(INPUT_FOLDER.glob('*.json'))

    if not json_files:
        print(f"Warning: No JSON files found in folder '{INPUT_FOLDER}'.")
        return

    print(f"Found {len(json_files)} JSON files in '{INPUT_FOLDER}', starting processing...")

    # List to store all protein information
    all_proteins_data = []

    for json_file in json_files:
        try:
            # Open and load the JSON file
            with open(json_file, 'r') as f:
                data = json.load(f)

            # The root of the JSON file has a 'name' field, format like "ATF1_ATF7"
            # We use this as the basis for naming proteins
            base_name = data.get('name', 'Unknown_Pair')
            protein_names = base_name.split('_')
            
            # 'sequences' is a list containing protein objects
            sequences_list = data.get('sequences', [])

            # Iterate through the sequences list, typically containing two proteins
            for i, protein_entry in enumerate(sequences_list):
                # The 'protein' key contains id and sequence
                protein_data = protein_entry.get('protein', {})
                sequence = protein_data.get('sequence', '')
                
                # If there is no explicit protein name in JSON, infer it from the base name
                protein_name = protein_names[i] if i < len(protein_names) else f"{base_name}_p{i+1}"
                
                # Calculate sequence length
                sequence_length = len(sequence)

                # Add extracted information to the list
                all_proteins_data.append({
                    'protein_name': protein_name,
                    'sequence': sequence,
                    'length': sequence_length
                })
        
        except json.JSONDecodeError:
            print(f"Warning: File '{json_file.name}' is not a valid JSON file, skipped.")
        except Exception as e:
            print(f"Error occurred while processing file '{json_file.name}': {e}")

    # --- Create DataFrame ---
    if not all_proteins_data:
        print("Failed to extract protein data from any files.")
        return

    # Convert the list of dictionaries to a DataFrame
    df = pd.DataFrame(all_proteins_data)

    # Remove duplicates potentially caused by filename parsing errors
    df = df.drop_duplicates(subset='protein_name')

    # Set the 'protein_name' column as the index (row names)
    df = df.set_index('protein_name')

    # --- Display Results ---
    print("\n--- Processing Complete ---")
    print("Generated DataFrame Preview:")
    print(df.head())
    
    # Save the DataFrame to a CSV file
    output_filename = 'spe_protein_sequences_and_lengths.csv'
    df.to_csv(output_filename)
    print(f"\nDataFrame successfully saved to '{output_filename}'")


if __name__ == "__main__":
    main()

在 'json_input/alphafold3_inputs_1888_spe' 中找到 1888 个 JSON 文件，开始处理...

--- 处理完成 ---
生成的 DataFrame 预览:
                                                       sequence  length
protein_name                                                           
AHR           MNSSSANITYASRKRRKPVQKTVKPIPAEGIKSNPSKRHRDRLNTE...     848
EBF1          MFGIQESIQRSGSSMKEEPLGSGMNAVRTWMQGAGVLDANTAAQSG...     591
HOXA10        MSARKGYLLPSPNYPTTMSCSESPAANSFLVDSLISSGRGEAGGGG...     410
ZBTB33        MESRKLISATDIQYSGSLLNSLNEQRGHGLFCDVTVIVEDRKFRAH...     672
PRDM4         MHHRMNEMNLSPVGMEQLTSSSVSNALPVSGSHLGLAASPTHSAIP...     801

DataFrame 已成功保存到 'spe_protein_sequences_and_lengths.csv'


## Com verification analysis runx1 and RORA, REL, NFKB1

In [20]:
import os
import re
from Bio.PDB import MMCIFParser, NeighborSearch

In [31]:
CIF_DIRECTORY = "./com_validation/var/fold_runx1_t147a_rora_seed1/"
DISTANCE_THRESHOLD = 5

folder_name = os.path.basename(os.path.normpath(CIF_DIRECTORY))
OUTPUT_FILE = f"{folder_name}.txt" 

In [32]:
# --- Main Program ---
def find_interface_from_cif(cif_file_path, dist_thresh):
    """
    Analyzes a single CIF file to identify interacting residues.
    Returns a tuple: (String for printing, Set of Chain 1 residues, Set of Chain 2 residues)
    """
    parser = MMCIFParser(QUIET=True)
    try:
        structure = parser.get_structure("protein_complex", cif_file_path)
    except Exception as e:
        error_msg = f"Unable to parse file {os.path.basename(cif_file_path)}: {e}\n"
        return error_msg, set(), set()

    model = structure[0]
    
    chain_ids = [chain.id for chain in model]
    if len(chain_ids) < 2:
        warn_msg = f"Warning: File {os.path.basename(cif_file_path)} has fewer than 2 chains, cannot analyze interaction.\n"
        return warn_msg, set(), set()
    
    chain_1_id, chain_2_id = chain_ids[0], chain_ids[1]

    chain1_atoms = [atom for atom in model[chain_1_id].get_atoms()]
    chain2_atoms = [atom for atom in model[chain_2_id].get_atoms()]

    if not chain1_atoms or not chain2_atoms:
        warn_msg = f"Warning: Chain {chain_1_id} or {chain_2_id} in file {os.path.basename(cif_file_path)} has no atoms.\n"
        return warn_msg, set(), set()

    # Use NeighborSearch for efficient distance searching
    ns_chain2 = NeighborSearch(chain2_atoms)
    interface_residues_chain1 = set()
    for atom1 in chain1_atoms:
        if ns_chain2.search(atom1.get_coord(), dist_thresh, 'A'):
            interface_residues_chain1.add(atom1.get_parent())

    ns_chain1 = NeighborSearch(chain1_atoms)
    interface_residues_chain2 = set()
    for atom2 in chain2_atoms:
        if ns_chain1.search(atom2.get_coord(), dist_thresh, 'A'):
            interface_residues_chain2.add(atom2.get_parent())

    # Format residue objects into 'RESNAME+RESID' string sets for easy intersection
    set_res1_formatted = {f"{res.get_resname()}{res.get_id()[1]}" for res in interface_residues_chain1}
    set_res2_formatted = {f"{res.get_resname()}{res.get_id()[1]}" for res in interface_residues_chain2}

    # Prepare the output string for printing
    output_str = ""
    output_str += f"--- Analyzing file: {os.path.basename(cif_file_path)} ---\n"
    output_str += f"Interaction analysis based on Chain '{chain_1_id}' and Chain '{chain_2_id}' (threshold < {dist_thresh} Å)\n"
    
    if set_res1_formatted and set_res2_formatted:
        # Sort residues numerically for cleaner output (e.g., extracting numbers from strings)
        res_list1_sorted = sorted(list(set_res1_formatted), key=lambda r: int(re.findall(r'\d+', r)[0]))
        res_list2_sorted = sorted(list(set_res2_formatted), key=lambda r: int(re.findall(r'\d+', r)[0]))
        
        output_str += f"  Interacting residues on Chain {chain_1_id} ({len(res_list1_sorted)} found): {', '.join(res_list1_sorted)}\n"
        output_str += f"  Interacting residues on Chain {chain_2_id} ({len(res_list2_sorted)} found): {', '.join(res_list2_sorted)}\n"
    else:
        output_str += "  No interacting residues found under this threshold.\n"
    
    return output_str, set_res1_formatted, set_res2_formatted


if __name__ == "__main__":
    if not os.path.isdir(CIF_DIRECTORY):
        print(f"Error: Directory '{CIF_DIRECTORY}' does not exist. Please modify the CIF_DIRECTORY variable in the script.")
    else:
        # List to store all analysis report strings
        full_analysis_report = []
        
        # Variables for calculating intersection
        core_intersection_sites = None
        processed_files_for_intersection = []

        # 1. Sort files by name
        cif_files = sorted([f for f in os.listdir(CIF_DIRECTORY) if f.endswith(".cif")])

        print(f"Found {len(cif_files)} CIF files, starting analysis in filename order...")

        # 2. Loop through and process files
        for filename in cif_files:
            cif_path = os.path.join(CIF_DIRECTORY, filename)
            
            # Analyze file and retrieve results
            result_string, sites_chain1, sites_chain2 = find_interface_from_cif(cif_path, DISTANCE_THRESHOLD)
            
            # Store single file analysis report
            full_analysis_report.append(result_string)

            # Update intersection
            if sites_chain1: # Calculate intersection only for files where interaction sites were found
                processed_files_for_intersection.append(filename)
                if core_intersection_sites is None:
                    # If it's the first valid file, assign directly
                    core_intersection_sites = sites_chain1
                else:
                    # Otherwise, update intersection
                    core_intersection_sites.intersection_update(sites_chain1)

        # 3. Generate and append the final intersection report
        summary_header = "\n" + "="*50 + "\n" + "=== Core Interaction Site Analysis Summary ===\n" + "="*50 + "\n"
        
        summary_report = ""
        if core_intersection_sites is not None and core_intersection_sites:
            num_files = len(processed_files_for_intersection)
            summary_report += f"In all {num_files} complexes where interactions were detected, the common core interaction sites on Chain 1 (intersection) are:\n"
            
            # Sort the final intersection results
            sorted_core_sites = sorted(list(core_intersection_sites), key=lambda r: int(re.findall(r'\d+', r)[0]))
            
            summary_report += f"  Core residues ({len(sorted_core_sites)} count): {', '.join(sorted_core_sites)}\n\n"
            summary_report += f"List of files involved in intersection calculation ({num_files} count):\n"
            summary_report += "\n".join([f" - {fname}" for fname in processed_files_for_intersection]) + "\n"
            
        elif not processed_files_for_intersection:
            summary_report += "No valid interaction sites detected in any files, cannot calculate intersection.\n"
        else:
            summary_report += "No common interacting residues on Chain 1 across all complexes with detected interactions (intersection is empty).\n"
            summary_report += f"List of files involved in intersection calculation ({len(processed_files_for_intersection)} count):\n"
            summary_report += "\n".join([f" - {fname}" for fname in processed_files_for_intersection]) + "\n"

        full_analysis_report.append(summary_header)
        full_analysis_report.append(summary_report)

        # 4. Write all results to file and print
        final_report_str = "\n".join(full_analysis_report)
        print(final_report_str)
        
        with open(OUTPUT_FILE, 'w') as f_out:
            f_out.write(final_report_str)
            
        print(f"\nAnalysis complete! All results saved to file: {OUTPUT_FILE}")

找到 5 个CIF文件，开始按文件名顺序分析...
--- 分析文件: fold_runx1_t147a_rora_seed1_new_model_0.cif ---
互作分析基于 链 'A' 和 链 'B' (阈值 < 5 Å)
  链 A 上的互作残基 (79个): MET1, ARG2, ILE3, VAL5, ALA7, SER10, ARG11, ARG12, PHE13, THR14, PRO15, PRO16, THR18, LEU20, MET25, ALA28, LEU29, LEU31, GLY37, ALA38, LEU40, ALA41, PRO59, GLU61, TRP79, ARG80, CYS81, ASN82, LYS83, THR84, LEU85, PRO86, ILE87, ALA88, GLU116, ARG118, ARG130, ASN132, ASP133, ARG135, VAL137, ARG139, ARG142, VAL170, ASP171, ARG174, GLU175, ARG177, ASP185, SER191, LEU192, SER193, PHE194, SER195, ARG197, LEU198, SER199, GLU202, THR352, TYR353, LEU354, PRO355, PRO356, TYR358, SER372, SER373, PRO374, SER375, SER442, ARG444, LEU445, GLU446, GLU447, ALA448, VAL449, TRP450, ARG451, PRO452, TYR453
  链 B 上的互作残基 (97个): SER139, ASP141, LYS144, PHE145, GLY146, ARG147, MET148, SER149, LYS150, GLN152, ARG153, SER228, PRO229, ASP230, GLN231, LYS282, LEU285, GLU286, THR287, CYS288, GLN289, TYR290, LEU291, GLU294, LEU295, GLN296, GLN297, ILE298, TRP300, GLN301, GLU309, ASN3

## Confirm the identity and intersection of protein sequences used

In [169]:
tf_seq = pd.read_csv('./output/all_protein_sequences_and_lengths.csv')
tf_seq

,protein_name,sequence,length
0,AHR,MNSSSANITYASRKRRKPVQKTVKPIPAEGIKSNPSKRHRDRLNTE...,848
1,EBF1,MFGIQESIQRSGSSMKEEPLGSGMNAVRTWMQGAGVLDANTAAQSG...,591
2,HOXA10,MSARKGYLLPSPNYPTTMSCSESPAANSFLVDSLISSGRGEAGGGG...,410
3,ZBTB33,MESRKLISATDIQYSGSLLNSLNEQRGHGLFCDVTVIVEDRKFRAH...,672
4,PRDM4,MHHRMNEMNLSPVGMEQLTSSSVSNALPVSGSHLGLAASPTHSAIP...,801
...,...,...,...
390,TEAD2,MGEPRAGAALDDGSGWTGSEEGSEEGTGGSEGAGGDGGPDAEGVWS...,447
391,ZNF554,MVTCAHLGRRARLPAAQPSACPGTCFSQEERMAAGYLPRWSQELVT...,538
392,ZNF324,MAFEDVAVYFSQEEWGLLDTAQRALYRRVMLDNFALVASLGLSTSR...,553
393,SREBF1,MDEPPFSEAALEQALGEPCDLDAALLTDIEDMLQLINNQDSDFPGL...,1147


In [171]:
tf_seq.replace('ARNTL', 'BMAL1', inplace=True)

In [172]:
tf_seq['formatted_sequence'] = tf_seq['sequence'].apply(
    lambda seq: ','.join([f"{i+1}-{aa}" for i, aa in enumerate(seq)])
)

In [173]:
tf_seq = tf_seq.sort_values('protein_name')
tf_seq

,protein_name,sequence,length,formatted_sequence
0,AHR,MNSSSANITYASRKRRKPVQKTVKPIPAEGIKSNPSKRHRDRLNTE...,848,"1-M,2-N,3-S,4-S,5-S,6-A,7-N,8-I,9-T,10-Y,11-A,..."
15,AIRE,MATDAALRRLLRLHRTEIAVAVDSAFPLLHALADHDVVPEDKFQET...,545,"1-M,2-A,3-T,4-D,5-A,6-A,7-L,8-R,9-R,10-L,11-L,..."
289,AR,MEVQLGLGRVYPRPPSKTYRGAFQNLFQSVREVIQNPGPRHPEAAS...,920,"1-M,2-E,3-V,4-Q,5-L,6-G,7-L,8-G,9-R,10-V,11-Y,..."
26,ARID3A,MKLQAVMETLLQRQQRARQELEARQQLPPDPPAAPPGRARAAPDED...,593,"1-M,2-K,3-L,4-Q,5-A,6-V,7-M,8-E,9-T,10-L,11-L,..."
198,ARID3B,MEPLQQQQQQQQQQQKQPHLAPLQMDAREKQGQQMREAQFLYAQKL...,561,"1-M,2-E,3-P,4-L,5-Q,6-Q,7-Q,8-Q,9-Q,10-Q,11-Q,..."
...,...,...,...,...
372,ZNF827,MPRRKQEQPKRLPSHVSRQEEAEGELSEGEHWYGNSSETPSEASYG...,1081,"1-M,2-P,3-R,4-R,5-K,6-Q,7-E,8-Q,9-P,10-K,11-R,..."
134,ZNF85,MGPLTFRDVAIEFSLKEWQCLDTAQRNLYRNVMLENYRNLVFLGIT...,595,"1-M,2-G,3-P,4-L,5-T,6-F,7-R,8-D,9-V,10-A,11-I,..."
127,ZSCAN16,MTTALEPEDQKGLLIIKAEDHYWGQDSSSQKCSPHRRELYRQHFRK...,348,"1-M,2-T,3-T,4-A,5-L,6-E,7-P,8-E,9-D,10-Q,11-K,..."
338,ZSCAN21,MMTKVLGMAPVLGPRPPQEQVGPLMVKVEEKEEKGKYLPSLEMFRQ...,473,"1-M,2-M,3-T,4-K,5-V,6-L,7-G,8-M,9-A,10-P,11-V,..."


In [174]:
import pandas as pd

# --- Step 1: Preprocess data for efficient calculation ---

# a. In tf_seq, convert 'formatted_sequence' strings into sets
#    This significantly speeds up subsequent intersection calculations
tf_seq['sequence_set'] = tf_seq['formatted_sequence'].apply(lambda x: set(x.split(',')))

# b. In gene_summary, similarly convert 'MutationList' strings into sets
gene_summary['mutation_set'] = gene_summary['MutationSiteAndAA_MutationList'].apply(lambda x: set(x.split(',')))


# --- Step 2: Map information from gene_summary to tf_seq ---

# a. Create a mapping dictionary from GeneSymbol to mutation set for fast lookup
gene_to_mutations_map = gene_summary.set_index('GeneSymbol')['mutation_set'].to_dict()

# b. Apply this mapping to the 'protein_name' column in tf_seq
#    This finds the corresponding mutation set for each row in tf_seq
#    If the gene does not exist in gene_summary, .get() returns an empty set()
tf_seq['mapped_mutation_set'] = tf_seq['protein_name'].apply(lambda name: gene_to_mutations_map.get(name, set()))


# --- Step 3: Calculate intersection counts and create a new column ---

# a. Define a function to calculate the size of the intersection between two sets
def calculate_intersection_count(row):
    return len(row['sequence_set'].intersection(row['mapped_mutation_set']))

# b. Calculate the intersection count within tf_seq
tf_seq['intersection_count'] = tf_seq.apply(calculate_intersection_count, axis=1)


# --- Step 4: Add intersection counts back to gene_summary ---

# a. Create a mapping dictionary from protein_name to intersection count
gene_to_intersection_count_map = tf_seq.set_index('protein_name')['intersection_count'].to_dict()

# b. Apply this mapping to the 'GeneSymbol' column in gene_summary
gene_summary['intersection_count'] = gene_summary['GeneSymbol'].map(gene_to_intersection_count_map).fillna(0).astype(int)


# --- Cleanup and Output ---

# Remove temporary set columns to keep the final result clean
tf_seq.drop(columns=['sequence_set', 'mapped_mutation_set'], inplace=True)
gene_summary.drop(columns=['mutation_set'], inplace=True)


print("Updated tf_seq DataFrame:")
print(tf_seq)

print("\n" + "="*40 + "\n")

print("Updated gene_summary DataFrame:")
print(gene_summary)

Updated tf_seq DataFrame:
    protein_name                                           sequence  length  \
0            AHR  MNSSSANITYASRKRRKPVQKTVKPIPAEGIKSNPSKRHRDRLNTE...     848   
15          AIRE  MATDAALRRLLRLHRTEIAVAVDSAFPLLHALADHDVVPEDKFQET...     545   
289           AR  MEVQLGLGRVYPRPPSKTYRGAFQNLFQSVREVIQNPGPRHPEAAS...     920   
26        ARID3A  MKLQAVMETLLQRQQRARQELEARQQLPPDPPAAPPGRARAAPDED...     593   
198       ARID3B  MEPLQQQQQQQQQQQKQPHLAPLQMDAREKQGQQMREAQFLYAQKL...     561   
..           ...                                                ...     ...   
372       ZNF827  MPRRKQEQPKRLPSHVSRQEEAEGELSEGEHWYGNSSETPSEASYG...    1081   
134        ZNF85  MGPLTFRDVAIEFSLKEWQCLDTAQRNLYRNVMLENYRNLVFLGIT...     595   
127      ZSCAN16  MTTALEPEDQKGLLIIKAEDHYWGQDSSSQKCSPHRRELYRQHFRK...     348   
338      ZSCAN21  MMTKVLGMAPVLGPRPPQEQVGPLMVKVEEKEEKGKYLPSLEMFRQ...     473   
71       ZSCAN31  MASTEEQYDLKIVKVEEDPIWDQETHLRGNNFSGQEASRQLFRQFC...     406   

                         

In [179]:
gene_summary['MutationSiteAndAA_MutationCount']

0      326
1      428
2      464
3       91
4       58
      ... 
390    109
391     78
392     34
393     65
394     32
Name: MutationSiteAndAA_MutationCount, Length: 395, dtype: int64

In [153]:
tf_seq['MutationSite'] = gene_summary['MutationSiteAndAA_MutationList'].tolist()
tf_seq['MutationCount'] = gene_summary['MutationSiteAndAA_MutationCount'].tolist()

In [154]:
del tf_seq['sequence']

In [155]:
tf_seq

,protein_name,formatted_sequence,intersection_count,MutationSite,MutationCount
0,AHR,"1-M,2-N,3-S,4-S,5-S,6-A,7-N,8-I,9-T,10-Y,11-A,...",326,"621-Q,570-V,396-R,257-S,466-S,467-I,480-P,487-...",326
15,AIRE,"1-M,2-A,3-T,4-D,5-A,6-A,7-L,8-R,9-R,10-L,11-L,...",427,"257-R,83-K,139-R,228-G,1-M,80-V,401-P,432-A,44...",428
289,AR,"1-M,2-E,3-V,4-Q,5-L,6-G,7-L,8-G,9-R,10-V,11-Y,...",463,"775-R,719-W,867-V,797-W,591-K,648-S,764-Y,884-...",464
26,ARID3A,"1-M,2-K,3-L,4-Q,5-A,6-V,7-M,8-E,9-T,10-L,11-L,...",91,"79-G,447-E,486-A,522-S,551-G,511-T,43-P,197-R,...",91
47,ARID3B,"1-M,2-E,3-P,4-L,5-Q,6-Q,7-Q,8-Q,9-Q,10-Q,11-Q,...",58,"238-R,263-V,509-L,350-G,402-R,491-I,166-P,147-...",58
...,...,...,...,...,...
372,ZNF827,"1-M,2-P,3-R,4-R,5-K,6-Q,7-E,8-Q,9-P,10-K,11-R,...",109,"772-P,906-T,770-Q,226-K,591-N,223-L,499-G,821-...",109
134,ZNF85,"1-M,2-G,3-P,4-L,5-T,6-F,7-R,8-D,9-V,10-A,11-I,...",78,"231-K,250-K,318-E,447-I,278-I,329-L,385-L,425-...",78
127,ZSCAN16,"1-M,2-T,3-T,4-A,5-L,6-E,7-P,8-E,9-D,10-Q,11-K,...",34,"196-K,102-R,62-L,305-T,326-G,118-D,336-I,159-H...",34
338,ZSCAN21,"1-M,2-M,3-T,4-K,5-V,6-L,7-G,8-M,9-A,10-P,11-V,...",65,"36-K,45-R,184-G,297-R,289-S,168-T,147-K,46-Q,6...",65


In [184]:
tf_seq.to_csv('./tf_var_sites.csv')

### Determine whether all mutations are in the sequence

In [227]:
seq = 'MEQKPSKVECGSDPEENSARSPDGKRKRKNGQCSLKTSMSGYIPSYLDKDEQCVVCGDKATGYHYRCITCEGCKGFFRRTIQKNLHPTYSCKYDSCCVIDKITRNQCQLCRFKKCIAVGMAMDLVLDDSKRVAKRKLIEQNRERRRKEEMIRSLQQRPEPTPEEWDLIHIATEAHRSTNAQGSHWKQRRKFLPDDIGQSPIVSMPDGDKVDLEAFSEFTKIITPAITRVVDFAKKLPMFSELPCEDQIILLKGCCMEIMSLRAAVRYDPESDTLTLSGEMAVKREQLKNGGLGVVSDAIFELGKSLSAFNLDDTEVALLQAVLLMSTDRSGLLCVDKIEKSQEAYLLAFEHYVNHRKHNIPHFWPKLLMKVTDLRMIGACHASRFLHMKVECPTELFPPLFLEVFEDQEV'

In [228]:
mut = tf_seq[tf_seq['protein_name'] == 'THRA']['MutationSite'].item()
mut

KeyError: 'protein_name'

In [229]:
set(mut.split(',')) - set(','.join([f"{i+1}-{aa}" for i, aa in enumerate(seq)]).split(','))

{'372-R', '377-S', '384-A', '417-R', '462-D'}

In [230]:
len(set(mut.split(',')) - set(','.join([f"{i+1}-{aa}" for i, aa in enumerate(seq)]).split(',')))

5

### Adjust part of the protein sequence and re-predict it

In [284]:
import pandas as pd

In [285]:
tf_seq = pd.read_csv('./result/new_seq.csv')
tf_seq

,gene,seq
0,ETS1,MSYFVDSAGSSPVPYSAPRPAVVRQGPSNTYEDPRMNCGFQSNYHQ...
1,MXI1,MGKRGRPRKEARCEGAGLAPAAPPAVPPAVAAPQPPALPEDPAGAK...
2,MGA,MEEKQQIILANQDGGTVAGAAPTFFVILKQPGNGKTDQGILVTNQD...
3,ZNF250,MAAARLLPVPAGPQAKLTFEDVAVLLSQDEWDRLCPAQRGLYRNVM...
4,CREM,MSKCARKKYIKTNPRQMTMETVESQHDGSITASLTESKSAHVQTQT...
5,PPARG,MVDTEMPFWPTNFGISSVDLSVMEDHSHSFDIKPFTTVDFSSISTP...
6,DDIT3,MAAESLPFSFGTLSSWELEAWYEDLQEVLSSDENGGTYVSPPGNEE...
7,POU2F1,MADGGAASQDESSAAAAAAADSRMNNPSETSKPSMESGDGNTGTQT...
8,ZNF423,MSRRKQAKPRSVKVEEGEASDFSLAWDSSVTAAGGLEGEPECDQKT...
9,RORB,MRAQIEVIPCKICGDKSSGIHYGVITCEGCKGFFRRSQQNNASYSC...


In [286]:
import os

In [287]:
all = os.listdir('./json_input/all/')

In [288]:
all

['AHR_EBF1.json',
 'AHR_HOXA10.json',
 'AHR_ZBTB33.json',
 'PRDM4_ZNF770.json',
 'AHR_MEF2D.json',
 'AHR_MEF2C.json',
 'AHR_MEF2A.json',
 'AHR_MYBL1.json',
 'AHR_VDR.json',
 'AHR_RFX5.json',
 'AHR_TFCP2.json',
 'AHR_ZNF41.json',
 'AHR_ZNF528.json',
 'AIRE_CREB5.json',
 'AIRE_ATF7.json',
 'AIRE_ZNF121.json',
 'AIRE_ZNF134.json',
 'AIRE_ZNF232.json',
 'BATF_E2F4.json',
 'E2F4_NFE2L1.json',
 'NFE2L1_TCF3.json',
 'JUN_TCF3.json',
 'ARID3A_FOS.json',
 'ARID3A_JDP2.json',
 'ARID3A_ATF3.json',
 'ARID3A_FOSL2.json',
 'ARID3A_FOSL1.json',
 'ARID3A_FOSB.json',
 'ARID3A_JUNB.json',
 'ARID3A_NFE2L1.json',
 'ARID3A_JUND.json',
 'ARID3A_BATF.json',
 'ARID3A_JUN.json',
 'BATF_NRF1.json',
 'NFE2L1_NRF1.json',
 'NFE2L1_TP53.json',
 'JUN_PRDM1.json',
 'JUNB_PRDM1.json',
 'BATF_PRDM1.json',
 'FOSB_PRDM1.json',
 'JUND_PRDM4.json',
 'JDP2_PRDM4.json',
 'NFE2_PRDM4.json',
 'FOSB_PRDM4.json',
 'FOS_PRDM4.json',
 'ATF3_KDM2B.json',
 'FOSL1_RBPJ.json',
 'JDP2_RBPJ.json',
 'JUN_KDM2B.json',
 'ATF3_RBPJ.json',
 

In [289]:
def is_gene_pair_match(filename, genes_to_check):
    """
    Checks if either GENE1 or GENE2 from the filename 'GENE1_GENE2.json' 
    exists in the genes_to_check set.
    """
    # a. Safely remove the .json suffix to retrieve the 'GENE1_GENE2' portion
    base_name = os.path.splitext(filename)[0]
    
    # b. Split by underscore
    parts = base_name.split('_')
    
    # c. Check if the format is correct (must consist of exactly two parts)
    if len(parts) != 2:
        return False
        
    gene1, gene2 = parts
    
    # d. Exact match: Check if gene1 or gene2 exists in the set
    if gene1 in genes_to_check or gene2 in genes_to_check:
        return True
    
    return False

# 3. Use list comprehension and this new function to perform filtering
filtered_list = [filename for filename in all if is_gene_pair_match(filename, set(tf_seq['gene']))]

In [290]:
len(filtered_list)

561

In [291]:
import shutil

In [292]:
import os
import shutil

# --- 3. Iterate through the list and copy files ---

# Counters to track successful and failed file copies
success_count = 0
fail_count = 0

print("\nStarting file copy process...")

for filename in filtered_list:
    # a. Construct the full source file path
    source_path = os.path.join('./json_input/all/', filename)
    
    # b. Construct the full destination file path
    destination_path = os.path.join('./json_input/cured_seq', filename)
    
    try:
        # c. Check if the source file exists
        if os.path.exists(source_path):
            # d. Execute the copy operation
            # shutil.copy2() copies both file content and metadata (e.g., modification time)
            # If you only want to copy the content, you can use shutil.copy()
            shutil.copy2(source_path, destination_path)
            print(f"  [Success] Copied '{filename}' to the destination folder.")
            success_count += 1
        else:
            # Note: Ensure 'source_folder' is defined in your previous code or replace with the hardcoded path
            print(f"  [Warning] Source file '{filename}' does not exist in source folder, skipped.")
            fail_count += 1
            
    except Exception as e:
        # Catch other possible errors, such as permission issues
        print(f"  [Failure] Error occurred while copying '{filename}': {e}")
        fail_count += 1

# --- 4. Print summary report ---

print("\nCopy operation completed.")
print(f"Total: Successfully copied {success_count} files, Failed/Skipped {fail_count} files.")


开始复制文件...
  [成功] 已将 'AHR_MEF2A.json' 复制到目标文件夹。
  [成功] 已将 'AHR_ZNF41.json' 复制到目标文件夹。
  [成功] 已将 'NFE2L1_TCF3.json' 复制到目标文件夹。
  [成功] 已将 'JUN_TCF3.json' 复制到目标文件夹。
  [成功] 已将 'FOSL1_RBPJ.json' 复制到目标文件夹。
  [成功] 已将 'JDP2_RBPJ.json' 复制到目标文件夹。
  [成功] 已将 'ATF3_RBPJ.json' 复制到目标文件夹。
  [成功] 已将 'FOS_RBPJ.json' 复制到目标文件夹。
  [成功] 已将 'NFE2L1_RBPJ.json' 复制到目标文件夹。
  [成功] 已将 'BATF_RBPJ.json' 复制到目标文件夹。
  [成功] 已将 'JUND_RBPJ.json' 复制到目标文件夹。
  [成功] 已将 'JUNB_RBPJ.json' 复制到目标文件夹。
  [成功] 已将 'FOSL2_RBPJ.json' 复制到目标文件夹。
  [成功] 已将 'FOSB_RBPJ.json' 复制到目标文件夹。
  [成功] 已将 'JUN_RBPJ.json' 复制到目标文件夹。
  [成功] 已将 'NFE2_ZFP57.json' 复制到目标文件夹。
  [成功] 已将 'FOSB_ZFP57.json' 复制到目标文件夹。
  [成功] 已将 'BATF_ZFP57.json' 复制到目标文件夹。
  [成功] 已将 'FOSL1_ZFP57.json' 复制到目标文件夹。
  [成功] 已将 'NFE2L1_ZFP57.json' 复制到目标文件夹。
  [成功] 已将 'FOSL2_ZFP57.json' 复制到目标文件夹。
  [成功] 已将 'MAF_ZBTB33.json' 复制到目标文件夹。
  [成功] 已将 'BACH1_MEF2B.json' 复制到目标文件夹。
  [成功] 已将 'MAF_MEF2C.json' 复制到目标文件夹。
  [成功] 已将 'MAF_MEF2A.json' 复制到目标文件夹。
  [成功] 已将 'MEF2B_NFE2L2.json' 复制到目标文件夹。
  [成功] 已

In [293]:
import os
import json
import pandas as pd

# b. Define the folder path containing JSON files
json_folder_path = './json_input/cured_seq' # Assuming script is in the same directory as cured_seq folder

# --- 2. Create a mapping dictionary from gene name to sequence ---
#    This will significantly improve lookup efficiency
gene_to_sequence_map = tf_seq.set_index('gene')['seq'].to_dict()

print("Example of the created mapping dictionary:")
print(list(gene_to_sequence_map.items())[:2]) # Print the first two key-value pairs


# --- 3. Ensure the target directory exists ---
if not os.path.isdir(json_folder_path):
    print(f"Error: Folder '{json_folder_path}' does not exist. Please check the path.")
    exit()

# --- 4. Get the list of all JSON files ---
try:
    json_files = [f for f in os.listdir(json_folder_path) if f.endswith('.json')]
    if not json_files:
        print(f"Warning: No .json files found in '{json_folder_path}'.")
        exit()
    print(f"\nFound {len(json_files)} JSON files in the folder. Starting processing...")
except Exception as e:
    print(f"Error: Unable to read folder '{json_folder_path}': {e}")
    exit()

# --- 5. Loop through and modify each JSON file ---

# Counters for success and failure
success_count = 0
fail_count = 0
not_found_genes = set() # Use a set to store missing gene names to avoid duplicate printing

for filename in json_files:
    file_path = os.path.join(json_folder_path, filename)
    
    try:
        # a. Read the JSON file
        with open(file_path, 'r') as f:
            data = json.load(f)

        # Flag to record if the file has been modified
        file_modified = False

        # b. Parse gene names from the 'name' field
        #    e.g., "AHR_MEF2A" -> ['AHR', 'MEF2A']
        genes_in_file = data.get('name', '').split('_')

        if not genes_in_file or not data.get('sequences'):
            print(f"  [Warning] File '{filename}' format is incorrect (missing 'name' or 'sequences' fields), skipped.")
            fail_count += 1
            continue

        # c. Iterate through the sequences list in JSON and perform replacement
        for i, seq_entry in enumerate(data['sequences']):
            if i < len(genes_in_file):
                gene_name_to_replace = genes_in_file[i]
                
                # d. Look up the new sequence from the mapping dictionary
                if gene_name_to_replace in gene_to_sequence_map:
                    new_sequence = gene_to_sequence_map[gene_name_to_replace]
                    
                    # Check if the sequence actually needs updating
                    if seq_entry['protein']['sequence'] != new_sequence:
                        seq_entry['protein']['sequence'] = new_sequence
                        file_modified = True
                else:
                    # If this gene is not found in our DataFrame, record it
                    not_found_genes.add(gene_name_to_replace)

        # e. If the file content was modified, write back to the file
        if file_modified:
            with open(file_path, 'w') as f:
                # indent=2 formats the JSON file for easier reading
                json.dump(data, f, indent=2)
            print(f"  [Success] Updated file '{filename}'.")
            success_count += 1
        else:
            print(f"  [Info] File '{filename}' content is already up-to-date, no update needed.")
            # This case can also be counted as success
            success_count += 1

    except json.JSONDecodeError:
        print(f"  [Failure] File '{filename}' is not a valid JSON file, skipped.")
        fail_count += 1
    except Exception as e:
        print(f"  [Failure] Unknown error occurred while processing file '{filename}': {e}")
        fail_count += 1

# --- 6. Print summary report ---
print("\nProcessing complete.")
print(f"Total: {success_count} files processed successfully, {fail_count} files failed or skipped.")

if not_found_genes:
    print("\nWarning: Sequences for the following genes were not found in your tf_seq DataFrame:")
    for gene in sorted(list(not_found_genes)):
        print(f"  - {gene}")

创建的映射字典示例:
[('ETS1', 'MSYFVDSAGSSPVPYSAPRPAVVRQGPSNTYEDPRMNCGFQSNYHQQRPCYPFWDEMATQEVPTGLEHCVSDMECADVPLLTPSSKEMMSQALKATFSGFTKEQQRLGIPKDPRQWTETHVRDWVMWAVNEFSLKGVDFQKFCMNGAALCALGKDCFLELAPDFVGDILWEHLEILQKEDVKPYQVNGVNPAYPESRYTSDYFISYGIEHAQCVPPSEFSEPSFITESYQTLHPISSEELLSLKYENDYPSVILRDPLQTDTLQNDYFAIKQEVVTPDNMCMGRTSRGKLGGQDSFESIESYDSCDRLTQSWSSQSSFNSLQRVPSYDSFDSEDYPAALPNHKPKGTFKDYVRDRADLNKDKPVIPAAALAGYTGSGPIQLWQFLLELLTDKSCQSFISWTGDGWEFKLSDPDEVARRWGKRKNKPKMNYEKLSRGLRYYYDKNIIHKTAGKRYVYRFVCDLQSLLGYTPEELHAMLDVKPDADE'), ('MXI1', 'MGKRGRPRKEARCEGAGLAPAAPPAVPPAVAAPQPPALPEDPAGAKPRCPFSDIFNTSENSMEKHINTFLQNVQILLEAASYLEQIEKENKKCEHGYASSFPSMPSPRLQHSKPPRRLSRAQKHSSGSSNTSTANRSTHNELEKNRRAHLRLCLERLKVLIPLGPDCTRHTTLGLLNKAKAHIKKLEEAERKSQHQLENLEREQRFLKWRLEQLQGPQEMERIRMDSIGSTISSDRSDSEREEIEVDVESTEFSHGEVDNISTTSISDIDDHSSLPSIGSDEGYSSASVKLSFTS')]

在文件夹中找到 561 个JSON文件。开始处理...
  [成功] 已更新文件 'AHR_MEF2A.json'。
  [成功] 已更新文件 'AHR_ZNF41.json'。
  [成功] 已更新文件 'NFE2L1_TCF3.json'。
  [成功] 已更新文件 'JUN_TCF3.json'。
  [成功] 已更新文件 'FOSL1_RBPJ.j

### Adjusted mutation correspondence rate

In [186]:
gene_summary = pd.read_csv('./clinvar/tf_var.csv', index_col=0)
gene_summary

,GeneSymbol,MutationSiteAndAA_MutationList,MutationSiteAndAA_MutationCount
0,AHR,"621-Q,570-V,396-R,257-S,466-S,467-I,480-P,487-...",326
1,AIRE,"257-R,83-K,139-R,228-G,1-M,80-V,401-P,432-A,44...",428
2,AR,"775-R,719-W,867-V,797-W,591-K,648-S,764-Y,884-...",464
3,ARID3A,"79-G,447-E,486-A,522-S,551-G,511-T,43-P,197-R,...",91
4,ARID3B,"238-R,263-V,509-L,350-G,402-R,491-I,166-P,147-...",58
...,...,...,...
390,ZNF827,"772-P,906-T,770-Q,226-K,591-N,223-L,499-G,821-...",109
391,ZNF85,"231-K,250-K,318-E,447-I,278-I,329-L,385-L,425-...",78
392,ZSCAN16,"196-K,102-R,62-L,305-T,326-G,118-D,336-I,159-H...",34
393,ZSCAN21,"36-K,45-R,184-G,297-R,289-S,168-T,147-K,46-Q,6...",65


In [187]:
tf_seq = pd.read_csv('./result/old/all_protein_sequences_and_lengths.csv')[['protein_name', 'sequence']]
tf_seq

,protein_name,sequence
0,AHR,MNSSSANITYASRKRRKPVQKTVKPIPAEGIKSNPSKRHRDRLNTE...
1,EBF1,MFGIQESIQRSGSSMKEEPLGSGMNAVRTWMQGAGVLDANTAAQSG...
2,HOXA10,MSARKGYLLPSPNYPTTMSCSESPAANSFLVDSLISSGRGEAGGGG...
3,ZBTB33,MESRKLISATDIQYSGSLLNSLNEQRGHGLFCDVTVIVEDRKFRAH...
4,PRDM4,MHHRMNEMNLSPVGMEQLTSSSVSNALPVSGSHLGLAASPTHSAIP...
...,...,...
390,TEAD2,MGEPRAGAALDDGSGWTGSEEGSEEGTGGSEGAGGDGGPDAEGVWS...
391,ZNF554,MVTCAHLGRRARLPAAQPSACPGTCFSQEERMAAGYLPRWSQELVT...
392,ZNF324,MAFEDVAVYFSQEEWGLLDTAQRALYRRVMLDNFALVASLGLSTSR...
393,SREBF1,MDEPPFSEAALEQALGEPCDLDAALLTDIEDMLQLINNQDSDFPGL...


In [188]:
tf_seq.replace('ARNTL', 'BMAL1', inplace=True)

In [189]:
new_seq = pd.read_csv('./result/new_seq.csv')
new_seq.columns = ['protein_name', 'sequence']
new_seq

,protein_name,sequence
0,ETS1,MSYFVDSAGSSPVPYSAPRPAVVRQGPSNTYEDPRMNCGFQSNYHQ...
1,MXI1,MGKRGRPRKEARCEGAGLAPAAPPAVPPAVAAPQPPALPEDPAGAK...
2,MGA,MEEKQQIILANQDGGTVAGAAPTFFVILKQPGNGKTDQGILVTNQD...
3,ZNF250,MAAARLLPVPAGPQAKLTFEDVAVLLSQDEWDRLCPAQRGLYRNVM...
4,CREM,MSKCARKKYIKTNPRQMTMETVESQHDGSITASLTESKSAHVQTQT...
5,PPARG,MVDTEMPFWPTNFGISSVDLSVMEDHSHSFDIKPFTTVDFSSISTP...
6,DDIT3,MAAESLPFSFGTLSSWELEAWYEDLQEVLSSDENGGTYVSPPGNEE...
7,POU2F1,MADGGAASQDESSAAAAAAADSRMNNPSETSKPSMESGDGNTGTQT...
8,ZNF423,MSRRKQAKPRSVKVEEGEASDFSLAWDSSVTAAGGLEGEPECDQKT...
9,RORB,MRAQIEVIPCKICGDKSSGIHYGVITCEGCKGFFRRSQQNNASYSC...


In [190]:
tf_seq = pd.concat([tf_seq[~tf_seq['protein_name'].isin(new_seq['protein_name'])], new_seq])
tf_seq

,protein_name,sequence
0,AHR,MNSSSANITYASRKRRKPVQKTVKPIPAEGIKSNPSKRHRDRLNTE...
1,EBF1,MFGIQESIQRSGSSMKEEPLGSGMNAVRTWMQGAGVLDANTAAQSG...
2,HOXA10,MSARKGYLLPSPNYPTTMSCSESPAANSFLVDSLISSGRGEAGGGG...
3,ZBTB33,MESRKLISATDIQYSGSLLNSLNEQRGHGLFCDVTVIVEDRKFRAH...
4,PRDM4,MHHRMNEMNLSPVGMEQLTSSSVSNALPVSGSHLGLAASPTHSAIP...
...,...,...
45,NFIB,MMYSPICLTQDEFHPFIEALLPHVRAIAYTWFNLQARKRKYFKKHE...
46,THRA,MEQKPSKVECGSDPEENSARSPDGKRKRKNGQCSLKTSMSGYIPSY...
47,ARID3B,MEPLQQQQQQQQQQQKQPHLAPLQMDAREKQGQQMREAQFLYAQKL...
48,KLF4,MRQPPGESDMAVSDALLPSFSTFASGPAGREKTLRQAGAPNNRWRE...


In [191]:
tf_seq['formatted_sequence'] = tf_seq['sequence'].apply(
    lambda seq: ','.join([f"{i+1}-{aa}" for i, aa in enumerate(seq)])
)

In [192]:
tf_seq

,protein_name,sequence,formatted_sequence
0,AHR,MNSSSANITYASRKRRKPVQKTVKPIPAEGIKSNPSKRHRDRLNTE...,"1-M,2-N,3-S,4-S,5-S,6-A,7-N,8-I,9-T,10-Y,11-A,..."
1,EBF1,MFGIQESIQRSGSSMKEEPLGSGMNAVRTWMQGAGVLDANTAAQSG...,"1-M,2-F,3-G,4-I,5-Q,6-E,7-S,8-I,9-Q,10-R,11-S,..."
2,HOXA10,MSARKGYLLPSPNYPTTMSCSESPAANSFLVDSLISSGRGEAGGGG...,"1-M,2-S,3-A,4-R,5-K,6-G,7-Y,8-L,9-L,10-P,11-S,..."
3,ZBTB33,MESRKLISATDIQYSGSLLNSLNEQRGHGLFCDVTVIVEDRKFRAH...,"1-M,2-E,3-S,4-R,5-K,6-L,7-I,8-S,9-A,10-T,11-D,..."
4,PRDM4,MHHRMNEMNLSPVGMEQLTSSSVSNALPVSGSHLGLAASPTHSAIP...,"1-M,2-H,3-H,4-R,5-M,6-N,7-E,8-M,9-N,10-L,11-S,..."
...,...,...,...
45,NFIB,MMYSPICLTQDEFHPFIEALLPHVRAIAYTWFNLQARKRKYFKKHE...,"1-M,2-M,3-Y,4-S,5-P,6-I,7-C,8-L,9-T,10-Q,11-D,..."
46,THRA,MEQKPSKVECGSDPEENSARSPDGKRKRKNGQCSLKTSMSGYIPSY...,"1-M,2-E,3-Q,4-K,5-P,6-S,7-K,8-V,9-E,10-C,11-G,..."
47,ARID3B,MEPLQQQQQQQQQQQKQPHLAPLQMDAREKQGQQMREAQFLYAQKL...,"1-M,2-E,3-P,4-L,5-Q,6-Q,7-Q,8-Q,9-Q,10-Q,11-Q,..."
48,KLF4,MRQPPGESDMAVSDALLPSFSTFASGPAGREKTLRQAGAPNNRWRE...,"1-M,2-R,3-Q,4-P,5-P,6-G,7-E,8-S,9-D,10-M,11-A,..."


In [193]:
tf_seq = tf_seq.sort_values('protein_name')
tf_seq

,protein_name,sequence,formatted_sequence
0,AHR,MNSSSANITYASRKRRKPVQKTVKPIPAEGIKSNPSKRHRDRLNTE...,"1-M,2-N,3-S,4-S,5-S,6-A,7-N,8-I,9-T,10-Y,11-A,..."
15,AIRE,MATDAALRRLLRLHRTEIAVAVDSAFPLLHALADHDVVPEDKFQET...,"1-M,2-A,3-T,4-D,5-A,6-A,7-L,8-R,9-R,10-L,11-L,..."
289,AR,MEVQLGLGRVYPRPPSKTYRGAFQNLFQSVREVIQNPGPRHPEAAS...,"1-M,2-E,3-V,4-Q,5-L,6-G,7-L,8-G,9-R,10-V,11-Y,..."
26,ARID3A,MKLQAVMETLLQRQQRARQELEARQQLPPDPPAAPPGRARAAPDED...,"1-M,2-K,3-L,4-Q,5-A,6-V,7-M,8-E,9-T,10-L,11-L,..."
47,ARID3B,MEPLQQQQQQQQQQQKQPHLAPLQMDAREKQGQQMREAQFLYAQKL...,"1-M,2-E,3-P,4-L,5-Q,6-Q,7-Q,8-Q,9-Q,10-Q,11-Q,..."
...,...,...,...
372,ZNF827,MPRRKQEQPKRLPSHVSRQEEAEGELSEGEHWYGNSSETPSEASYG...,"1-M,2-P,3-R,4-R,5-K,6-Q,7-E,8-Q,9-P,10-K,11-R,..."
134,ZNF85,MGPLTFRDVAIEFSLKEWQCLDTAQRNLYRNVMLENYRNLVFLGIT...,"1-M,2-G,3-P,4-L,5-T,6-F,7-R,8-D,9-V,10-A,11-I,..."
127,ZSCAN16,MTTALEPEDQKGLLIIKAEDHYWGQDSSSQKCSPHRRELYRQHFRK...,"1-M,2-T,3-T,4-A,5-L,6-E,7-P,8-E,9-D,10-Q,11-K,..."
338,ZSCAN21,MMTKVLGMAPVLGPRPPQEQVGPLMVKVEEKEEKGKYLPSLEMFRQ...,"1-M,2-M,3-T,4-K,5-V,6-L,7-G,8-M,9-A,10-P,11-V,..."


In [194]:
import pandas as pd

# --- Step 1: Preprocess data for efficient calculation ---

# a. In tf_seq, convert 'formatted_sequence' strings into sets
#    This significantly speeds up subsequent intersection calculations
tf_seq['sequence_set'] = tf_seq['formatted_sequence'].apply(lambda x: set(x.split(',')))

# b. In gene_summary, similarly convert 'MutationList' strings into sets
gene_summary['mutation_set'] = gene_summary['MutationSiteAndAA_MutationList'].apply(lambda x: set(x.split(',')))


# --- Step 2: Map information from gene_summary to tf_seq ---

# a. Create a mapping dictionary from GeneSymbol to mutation set for fast lookup
gene_to_mutations_map = gene_summary.set_index('GeneSymbol')['mutation_set'].to_dict()

# b. Apply this mapping to the 'protein_name' column in tf_seq
#    This finds the corresponding mutation set for each row in tf_seq
#    If the gene does not exist in gene_summary, .get() returns an empty set()
tf_seq['mapped_mutation_set'] = tf_seq['protein_name'].apply(lambda name: gene_to_mutations_map.get(name, set()))


# --- Step 3: Calculate intersection counts and create a new column ---

# a. Define a function to calculate the size of the intersection between two sets
def calculate_intersection_count(row):
    return len(row['sequence_set'].intersection(row['mapped_mutation_set']))

# b. Calculate the intersection count within tf_seq
tf_seq['intersection_count'] = tf_seq.apply(calculate_intersection_count, axis=1)


# --- Step 4: Add intersection counts back to gene_summary ---

# a. Create a mapping dictionary from protein_name to intersection count
gene_to_intersection_count_map = tf_seq.set_index('protein_name')['intersection_count'].to_dict()

# b. Apply this mapping to the 'GeneSymbol' column in gene_summary
gene_summary['intersection_count'] = gene_summary['GeneSymbol'].map(gene_to_intersection_count_map).fillna(0).astype(int)


# --- Cleanup and Output ---

# Remove temporary set columns to keep the final result clean
tf_seq.drop(columns=['sequence_set', 'mapped_mutation_set'], inplace=True)
gene_summary.drop(columns=['mutation_set'], inplace=True)


print("Updated tf_seq DataFrame:")
print(tf_seq)

print("\n" + "="*40 + "\n")

print("Updated gene_summary DataFrame:")
print(gene_summary)

Updated tf_seq DataFrame:
    protein_name                                           sequence  \
0            AHR  MNSSSANITYASRKRRKPVQKTVKPIPAEGIKSNPSKRHRDRLNTE...   
15          AIRE  MATDAALRRLLRLHRTEIAVAVDSAFPLLHALADHDVVPEDKFQET...   
289           AR  MEVQLGLGRVYPRPPSKTYRGAFQNLFQSVREVIQNPGPRHPEAAS...   
26        ARID3A  MKLQAVMETLLQRQQRARQELEARQQLPPDPPAAPPGRARAAPDED...   
47        ARID3B  MEPLQQQQQQQQQQQKQPHLAPLQMDAREKQGQQMREAQFLYAQKL...   
..           ...                                                ...   
372       ZNF827  MPRRKQEQPKRLPSHVSRQEEAEGELSEGEHWYGNSSETPSEASYG...   
134        ZNF85  MGPLTFRDVAIEFSLKEWQCLDTAQRNLYRNVMLENYRNLVFLGIT...   
127      ZSCAN16  MTTALEPEDQKGLLIIKAEDHYWGQDSSSQKCSPHRRELYRQHFRK...   
338      ZSCAN21  MMTKVLGMAPVLGPRPPQEQVGPLMVKVEEKEEKGKYLPSLEMFRQ...   
71       ZSCAN31  MASTEEQYDLKIVKVEEDPIWDQETHLRGNNFSGQEASRQLFRQFC...   

                                    formatted_sequence  intersection_count  
0    1-M,2-N,3-S,4-S,5-S,6-A,7-N,8-I,9-T,10-

In [195]:
gene_summary

,GeneSymbol,MutationSiteAndAA_MutationList,MutationSiteAndAA_MutationCount,intersection_count
0,AHR,"621-Q,570-V,396-R,257-S,466-S,467-I,480-P,487-...",326,326
1,AIRE,"257-R,83-K,139-R,228-G,1-M,80-V,401-P,432-A,44...",428,427
2,AR,"775-R,719-W,867-V,797-W,591-K,648-S,764-Y,884-...",464,463
3,ARID3A,"79-G,447-E,486-A,522-S,551-G,511-T,43-P,197-R,...",91,91
4,ARID3B,"238-R,263-V,509-L,350-G,402-R,491-I,166-P,147-...",58,58
...,...,...,...,...
390,ZNF827,"772-P,906-T,770-Q,226-K,591-N,223-L,499-G,821-...",109,109
391,ZNF85,"231-K,250-K,318-E,447-I,278-I,329-L,385-L,425-...",78,78
392,ZSCAN16,"196-K,102-R,62-L,305-T,326-G,118-D,336-I,159-H...",34,34
393,ZSCAN21,"36-K,45-R,184-G,297-R,289-S,168-T,147-K,46-Q,6...",65,65


In [196]:
gene_summary['ratio'] = gene_summary['intersection_count']/gene_summary['MutationSiteAndAA_MutationCount']

In [197]:
gene_summary.to_csv('./result/var_seq_ratio.csv')

## Reorganize the interaction interface

In [294]:
import pathlib
import os
import re
import pandas as pd
from tqdm import tqdm

# BioPython related libraries
from Bio.PDB import MMCIFParser, NeighborSearch

# --- Configuration Parameters ---
MAIN_FOLDER_PATH = pathlib.Path("./output/cured/")
DISTANCE_THRESHOLD = 5.0
OUTPUT_CSV_FILE = "spe_tf_interaction_residues_summary.csv"

# ---- Core Modification 1: Add amino acid 3-letter to 1-letter mapping dictionary ----
AMINO_ACID_MAP = {
    'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
    'GLU': 'E', 'GLN': 'Q', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
    'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
    'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'
}

def find_interface_from_cif(cif_file_path, dist_thresh):
    """
    Analyzes a single CIF file to identify interacting residues.
    Returns a tuple: (Set of Chain 1 residue strings, Set of Chain 2 residue strings).
    """
    parser = MMCIFParser(QUIET=True)
    try:
        structure = parser.get_structure("protein_complex", cif_file_path)
    except Exception:
        return set(), set()

    model = structure[0]
    chain_ids = [chain.id for chain in model]
    if len(chain_ids) < 2:
        return set(), set()
    
    chain_1_id, chain_2_id = chain_ids[0], chain_ids[1]

    chain1_atoms = [atom for atom in model[chain_1_id].get_atoms()]
    chain2_atoms = [atom for atom in model[chain_2_id].get_atoms()]

    if not chain1_atoms or not chain2_atoms:
        return set(), set()

    # Find residues in Chain 1 that are close to Chain 2
    ns_chain2 = NeighborSearch(chain2_atoms)
    interface_residues_chain1 = set()
    for atom1 in chain1_atoms:
        if ns_chain2.search(atom1.get_coord(), dist_thresh, 'A'):
            interface_residues_chain1.add(atom1.get_parent())

    # Find residues in Chain 2 that are close to Chain 1
    ns_chain1 = NeighborSearch(chain1_atoms)
    interface_residues_chain2 = set()
    for atom2 in chain2_atoms:
        if ns_chain1.search(atom2.get_coord(), dist_thresh, 'A'):
            interface_residues_chain2.add(atom2.get_parent())

    # ---- Core Modification 2: Format using the mapping dictionary ----
    # If a non-standard amino acid is encountered, use 'X' as the default value
    set_res1_formatted = {
        f"{AMINO_ACID_MAP.get(res.get_resname(), 'X')}{res.get_id()[1]}" 
        for res in interface_residues_chain1
    }
    set_res2_formatted = {
        f"{AMINO_ACID_MAP.get(res.get_resname(), 'X')}{res.get_id()[1]}" 
        for res in interface_residues_chain2
    }

    return set_res1_formatted, set_res2_formatted

def sort_key_for_seed_folders(folder_path):
    """Helper function to sort sample folders numerically."""
    match = re.search(r'sample-(\d+)$', folder_path.name)
    if match:
        return int(match.group(1))
    return -1

def main():
    if not MAIN_FOLDER_PATH.is_dir():
        print(f"Error: Directory '{MAIN_FOLDER_PATH}' does not exist.")
        return

    tf_pair_dirs = [d for d in MAIN_FOLDER_PATH.iterdir() if d.is_dir()]
    all_results = []

    print(f"Starting analysis of {len(tf_pair_dirs)} TF pairs (Outputting single-letter format)...")
    
    for tf_pair_dir in tqdm(tf_pair_dirs, desc="Processing TF Pairs"):
        # ---- Core Modification 3: Convert folder name (row index) to uppercase ----
        pair_name = tf_pair_dir.name.upper()
        
        current_row = {"TF_Pair": pair_name}
        
        # Find and sort seed directories
        seed_dirs = sorted(
            [d for d in tf_pair_dir.glob("seed-1_sample-*") if d.is_dir()],
            key=sort_key_for_seed_folders
        )
        
        for i in range(5):
            col_name_tf1 = f"TF1_seed{i+1}"
            col_name_tf2 = f"TF2_seed{i+1}"

            tf1_res_str, tf2_res_str = "", ""

            if i < len(seed_dirs):
                seed_dir = seed_dirs[i]
                cif_files = list(seed_dir.glob("*.cif"))
                if cif_files:
                    set_res1, set_res2 = find_interface_from_cif(cif_files[0], DISTANCE_THRESHOLD)
                    
                    if set_res1:
                        # Sorting logic remains unchanged (sorting by residue number)
                        res_list1_sorted = sorted(list(set_res1), key=lambda r: int(re.findall(r'\d+', r)[0]))
                        tf1_res_str = ", ".join(res_list1_sorted)
                    
                    if set_res2:
                        res_list2_sorted = sorted(list(set_res2), key=lambda r: int(re.findall(r'\d+', r)[0]))
                        tf2_res_str = ", ".join(res_list2_sorted)
                else:
                    tf1_res_str = "CIF file not found"
                    tf2_res_str = "CIF file not found"
            else:
                tf1_res_str = "Sample folder missing"
                tf2_res_str = "Sample folder missing"
            
            current_row[col_name_tf1] = tf1_res_str
            current_row[col_name_tf2] = tf2_res_str
        
        all_results.append(current_row)

    final_df = pd.DataFrame(all_results)
    final_df = final_df.set_index("TF_Pair")
    final_df.to_csv(OUTPUT_CSV_FILE)
    
    print("\nAnalysis complete!")
    print(f"Results saved to file: '{OUTPUT_CSV_FILE}'")
    print("\nTable Preview:")
    print(final_df.head())

if __name__ == "__main__":
    main()

开始分析 561 个转录因子对 (输出为单字母格式)...


处理转录因子对: 100%|██████████| 561/561 [12:38<00:00,  1.35s/it]


分析完成！
结果已保存到文件: 'spe_tf_interaction_residues_summary.csv'

表格预览:
                                                       TF1_seed1  \
TF_Pair                                                            
AHR_MEF2A      P18, K63, E165, R172, R225, C226, L227, D229, ...   
ARID3A_NFYC    M54, Q55, A57, Q58, M59, A60, A61, L62, A64, M...   
ARID3A_IRF5    M1, K2, L3, Q4, M7, E8, L10, L11, Q58, M59, L6...   
ARID3A_POU2F2  L3, Q4, V6, M7, T9, L10, R13, Q58, M59, L62, M...   
AHR_ZNF41      K32, S33, N34, P35, S36, R38, H39, R40, R42, L...   

                                                       TF2_seed1  \
TF_Pair                                                            
AHR_MEF2A      I46, K53, L54, Q56, D61, M62, D63, L66, L67, E...   
ARID3A_NFYC    W23, M27, I30, R31, N32, L33, T34, V35, F38, R...   
ARID3A_IRF5    R14, H49, T51, R52, K73, A84, A88, N89, R91, C...   
ARID3A_POU2F2  E217, S220, L222, E223, E226, R230, K233, L239...   
AHR_ZNF41      L137, N152, L155, K159, K163, K303

### replace original result

In [296]:
cured = pd.read_csv('./result/cured_tf_interaction_residues_summary.csv')
cured

,TF_Pair,TF1_seed1,TF2_seed1,TF1_seed2,TF2_seed2,TF1_seed3,TF2_seed3,TF1_seed4,TF2_seed4,TF1_seed5,TF2_seed5
0,AHR_MEF2A,"P18, K63, E165, R172, R225, C226, L227, D229, ...","I46, K53, L54, Q56, D61, M62, D63, L66, L67, E...","E160, I162, T164, R167, R225, L227, F234, L293...","R15, Q18, T22, K53, D61, D63, K64, L67, E71, R...","H39, L43, K63, L64, D65, K66, L67, F148, Q149,...","K4, Q7, T9, R10, I11, M12, D13, E14, R17, S50,...","I8, T9, Y10, A11, S12, R13, K14, R15, R16, K17...","M1, G2, R3, K4, D13, R15, N16, Q18, V19, T20, ...","I8, Y10, R16, S36, H39, R40, L43, H163, T164, ...","S51, N52, K53, D61, D63, K64, L67, Y69, T70, N..."
1,ARID3A_NFYC,"M54, Q55, A57, Q58, M59, A60, A61, L62, A64, M...","W23, M27, I30, R31, N32, L33, T34, V35, F38, R...","M1, K2, L3, Q4, M7, L10, Q14, R18, L21, M54, Q...","W23, M27, I30, R31, N32, L33, T34, V35, F38, R...","M1, K2, L3, Q4, V6, M7, L10, R13, Q14, R18, L2...","W23, M27, I30, R31, N32, L33, T34, V35, F38, R...","M1, K2, L3, Q4, M7, L10, Q14, R18, L21, M54, Q...","W23, M27, I30, R31, N32, L33, T34, V35, F38, R...","M1, K2, L3, V6, M7, L10, R13, Q14, A17, R18, L...","W23, P24, M27, I30, R31, N32, L33, T34, V35, F..."
2,ARID3A_IRF5,"M1, K2, L3, Q4, M7, E8, L10, L11, Q58, M59, L6...","R14, H49, T51, R52, K73, A84, A88, N89, R91, C...","L3, L10, R13, Q14, Q58, M65, P125, W127, E128,...","R14, R91, K96, R108, G247, E248, L250, L251, D...","E142, R255, P258, N260, R261, T308, S309, F312...","G247, E248, L250, L251, D253, L254, L255, I256...","M1, K2, L3, Q4, A5, V6, M7, L10, L11, R13, Q14...","M153, L154, P155, S156, L157, S158, L159, G247...","W127, E130, E131, M132, E134, D135, L136, E138...","R14, H49, T51, R52, K73, A84, A88, N89, R91, C..."
3,ARID3A_POU2F2,"L3, Q4, V6, M7, T9, L10, R13, Q58, M59, L62, M...","E217, S220, L222, E223, E226, R230, K233, L239...","K2, L3, L136, D139, E140, E141, E142, E143, E1...","L160, L161, S259, Q260, T261, T262, S264, R265...","E134, L136, G137, E138, D139, E140, E141, E142...","M6, E218, L222, E223, E226, R230, K233, R265, ...","M1, K2, L3, E90, E91, D92, A93, A94, W127, E12...","L128, Q132, V135, T188, Q189, A190, V191, E217...","V6, L10, R13, D139, E142, E143, E144, E145, D1...","Q120, D124, I125, L131, Q132, S259, Q260, T261..."
4,AHR_ZNF41,"K32, S33, N34, P35, S36, R38, H39, R40, R42, L...","L137, N152, L155, K159, K163, K303, S470, D471...","N34, P35, S36, H39, R40, K63, Q149, D152, R225...","M105, E415, C416, G417, K418, A419, H429, H433...","K32, P35, R38, H39, R40, R42, E46, R49, D65, K...","E40, D128, L130, I133, E136, L137, D140, N141,...","N2, S3, S4, A6, N7, I8, Y10, A11, R13, K14, R1...","D45, A47, Q48, R50, L51, D54, V55, E58, L130, ...","M1, N2, S3, A6, N7, I8, T9, Y10, A11, S12, R13...","M1, S21, S22, C23, E24, S26, V27, S28, D31, V3..."
...,...,...,...,...,...,...,...,...,...,...,...
556,THRA_TP53,"L154, E159, W165, I168, H169, T172, E173, R176...","P12, P13, L14, S15, Q16, E17, T18, F19, L22, W...","S45, Y46, Y65, R66, C67, V118, G119, M120, A12...","L14, Q16, E17, T18, F19, S20, L22, W23, K24, L...","E148, I151, Q155, R157, E159, W165, D166, I168...","Q16, E17, T18, F19, S20, L22, W23, K24, L26, P...","Y46, E51, Y63, R66, D123, L124, L126, K130, A1...","P12, L14, Q16, E17, T18, F19, S20, L22, W23, K...","R144, K147, E148, M150, I151, L154, P205, D206...","P12, L14, Q16, E17, T18, F19, S20, L22, W23, K..."
557,THRA_ZNF282,"Y42, I43, P44, S45, Y46, L47, D48, K49, E51, T...","W151, R166, E169, N170, N173, W181, R184, P186...","Y42, P44, S45, Y46, T61, Y63, Y65, R66, E71, K...","I96, S97, L98, W99, V102, A103, I105, Q106, E1...","T61, E71, G72, K74, G75, F76, R78, R79, Q82, I...","E169, N170, N173, L174, R176, N177, F180, W181...","Y46, Y63, R66, D123, L124, L126, K134, L137, M...","N170, N173, L174, N177, F180, W181, L183, R184...","M39, S40, Y42, I43, P44, S45, Y46, T61, Y63, H...","W25, W27, A30, L31, P32, E34, R125, T126, T128..."
558,THRA_ZNF713,"K49, R145, L154, T161, P162, E163, E164, D166,...","F8, E18, E30, S31, L32, D36, V37, A3

In [298]:
spe = pd.read_csv('./result/old/spe_tf_interaction_residues_summary.csv')
spe

,TF_Pair,TF1_seed1,TF2_seed1,TF1_seed2,TF2_seed2,TF1_seed3,TF2_seed3,TF1_seed4,TF2_seed4,TF1_seed5,TF2_seed5
0,ARID3B_EWSR1,"R198, E201, D205, K208, R238, I241, K244, I246...","D347, L348, G349, P350, P351, V352, D353, E356...","R238, P240, I241, A243, K244, S283, I284, T285...","D353, D355, E356, D357, D359, K382, Q383, C384...","D105, E106, V107, A108, E109, R198, M229, R232...","T120, Q121, P122, Q199, L346, D347, L348, G349...","T234, R238, P240, I241, A243, K244, S286, F289...","D353, E356, C384, G385, V386, E418, D419, P421...","R198, E201, D205, K208, R232, T234, R238, I241...","D347, L348, G349, P350, P351, V352, D353, P354..."
1,ARID3B_GMEB2,"K244, S283, I284, T285, S286, F289, T290, R292...","D98, T138, T218, E220, D221, P222, G223, D224,...","Q173, N175, W176, L178, D179, Q181, L182, G196...","M1, P211, A212, T213, T218, T226, I229, T235, ...","W190, G197, R198, R200, E201, R204, K231, P235...","D50, T54, A57, A58, A61, F65, T66, A67, S68, S...","E201, G233, T234, R238, I241, M242, A243, S286...","N114, T213, E217, T218, D221, P222, G223, D224...","E93, D95, G96, L98, L182, N185, W190, D192, D1...","C96, G97, D98, K116, E122, H123, E129, H132, A..."
2,ARID3B_RUNX3,"E99, D100, E101, D102, R198, E201, I202, D205,...","P72, N73, M110, A111, G112, N113, D114, E115, ...","W190, A194, D195, R198, R232, G233, T234, R238...","M55, V56, R68, D70, S71, P72, A111, G112, N113...","R198, R232, G233, T234, R238, K269, W271, I284...","R68, D70, M110, A111, G112, N113, D114, E115, ...","D195, R198, R232, G233, T234, N237, R238, K269...","R68, T69, D70, S71, P72, N73, M110, A111, G112...","D104, V107, S191, D195, R198, E201, G233, T234...","R49, V52, D70, S71, P72, N73, F74, L98, G99, D..."
3,ARID5A_HIC2,"P43, E50, Q53, E57, G100, A101, E103, L104, T1...","L64, K104, R137, K140, R141, Q193, A221, G222,...","K10, Q11, E14, L18, A22, S23, K25, P26, K29, Q...","M1, V2, S3, P5, L6, A7, L8, R9, W10, W13, R16,...","T13, E14, A17, L18, P21, A22, P24, K25, P26, K...","V2, S3, L6, R9, W10, W13, A14, G15, R16, D18, ...","E40, R78, V79, P80, H81, L82, G83, F84, K112, ...","C361, P362, G363, E364, E365, G366, E367, G368...","H81, G83, F84, R109, G123, S124, T125, S126, T...","M1, F44, H77, S90, F93, Q94, R164, T167, V170,..."
4,AHR_EBF1,"M1, S3, L54, P55, F56, P57, I61, L72, Y76, A79...","A26, T29, W30, M31, Q32, G33, A34, G35, E55, K...","R40, N44, D48, P55, P57, D59, I61, N62, K63, D...","G21, G23, M24, N25, A26, V27, R28, T29, N40, A...","R15, R16, K17, V19, Q20, T22, K24, H39, R40, R...","L20, G21, M24, N25, R28, G46, R51, Y75, R77, Q...","S12, R13, K14, R15, R16, K17, V19, H39, R40, L...","L49, A50, R51, H53, E55, A73, Y75, R77, Q78, G...","H39, R42, L43, E46, L47, R49, L50, L53, L54, P...","E55, Q80, P81, E83, D184, R185, F186, F187, E2..."
...,...,...,...,...,...,...,...,...,...,...,...
1883,MGA_ZNF784,"M1, A2, K4, Q105, P106, P107, T108, V125, S126...","M1, A2, A3, A4, R5, P6, E7, A8, S10, F65, V77,...","P107, P121, G123, A124, V125, S126, V127, W129...","M1, A2, A3, A4, R5, E7, P23, L24, D25, L26, C6...","P106, P107, T108, W119, V125, S126, W129, Q272...","M1, A2, A3, A4, R5, E7, P57, F65, V77, V105, C...","P121, G123, T291, N293, N295, Y385, D429, W492...","M1, A2, A3, A4, R5, P6, E7, A8, Q9, S10, G63, ...","D104, Q105, P106, P107, T108, K109, W119, P121...","A4, R5, P6, A8, Q9, S10, R11, S12, S13, P14, T..."
1884,PRDM5_ZNF528,"D8, S11, L12, K13, R16, V17, Q18, D19, A26, R2...","M1, A2, L8, D12, V13, A14, I15, F17, E20, C24,...","M1, L2, G3, M4, Y5, P7, D8, R9, F10, S11, L12,...","Q5, L8, M11, D12, V13, A14, I15, E16, F17, E20...","L2, S14, S15, R16, V17, Q18, D19, G20, M21, M4...","Q5, G6, P7, L8, D12, V13, A14, I15, F17, L25, ...","Q18, D19, P45, E46, D47, D49, E50, N51, M52, D...","Q5, G6, P7, L8, D12, V13, I15, E16, F17, S18, ...","M1, L2, G3, M4, Y5, L12, K13, S14, S15, D19, G...","L3, Q5, M11, D12, A14, I15, F17, E21, C24, L25..."
1885,MGA_NFIB,"D104, P106, P107, V125, S126, Q272, W280, K281...","M1, Y3, R85, Q86, E87, Y88, D91,

In [299]:
com = pd.read_csv('./result/old/com_tf_interaction_residues_summary.csv')
com

,TF_Pair,TF1_seed1,TF2_seed1,TF1_seed2,TF2_seed2,TF1_seed3,TF2_seed3,TF1_seed4,TF2_seed4,TF1_seed5,TF2_seed5
0,JDP2_JUND,"G14, S15, L16, P17, K93, R96, T97, L100, Q101,...","F25, A26, S27, P28, L290, R292, I293, L296, E2...","R90, K93, R96, T97, L100, Q101, E103, S104, L1...","K289, L290, R292, I293, L296, E297, K299, V300...","K93, R96, T97, L100, Q101, E103, S104, L107, E...","L290, R292, I293, L296, E297, K299, V300, L303...","K93, R96, T97, L100, Q101, E103, S104, L107, E...","F136, G139, F140, K142, A143, L144, D146, L147...","T12, G14, K93, R96, T97, L100, Q101, E103, S10...","L54, S55, E56, S100, L103, I107, E133, F136, A..."
1,EGR1_TBX21,"M39, L100, T101, A102, E103, F247, V249, P250,...","P128, T160, R164, R165, F167, R184, F186, D188...","Y32, L35, M39, V249, P250, M251, I252, P253, D...","L131, R138, A140, N143, T160, Q162, R164, R165...","Q15, V249, P250, M251, I252, P253, Y255, L256,...","G14, T15, I158, R164, R184, F186, D188, V190, ...","Y32, L35, M39, F247, V249, P250, M251, I252, P...","L131, R138, A140, N143, I158, T160, Q162, G163...","Y32, L35, E36, M38, M39, N43, E219, V249, P250...","Y125, L127, P128, L131, R138, A140, T160, R164..."
2,EGR1_THAP1,"R371, I372, C373, I387, T389, H390, T391, G392","M1, V2, Q3, Y14, F22, K24, K49, C67, N69, K70,...","R371, I372, C373, M374, R388, T389, H390, T391...","M1, V2, Q3, Y14, F22, K24, N69, L71","L307, S310, Y311, Q312, S313, Q314, L315, I316...","Q3, Y14, D15, K16, K18, P19, V20, S21, F22, H2...","Y311, P367, F368, Q369, R371, I372, C373, M374...","M1, Q3, Y14, K18, P19, V20, S21, F22, H23, K24...","Y311, Q314, L315, C373, M374, A397, D399, I400...","K24, K49, Y50, E66, C67, N68, N69, L71, R169, ..."
3,EGR1_ZNF263,"L304, N308, Y311, Q312, L315, I316, P318, R320...","M74, T76, E78, E82, E103, L104, H105, E107, E1...","S319, R320, M321, R322, Y324, I361, K366, R371...","Q95, M210, P213, L215, P216, E248, T606, L607,...","H334, E335, P337, Y338, S350, R351, S352, D353...","P72, E73, M74, R75, T76, K77, E78, L437, E438,...","M321, E343, D346, Q369, C373, M374, R375, N376","L379, C380, P381, L382, C383, G384, T447, Y490...","P318, S319, R320, M321, K323, Y324, P325, N326...","L28, H67, P72, E73, M74, R75, T76, L198, F199,..."
4,EGR1_ZNF341,"H25, S26, P27, T28, C192, A193, V194, P195, K3...","S203, L204, G205, P206, P207, N211, P212, G213...","T281, Q282, Q283, P284, S285, L286, P288, T291...","Q3, A4, F6, E7, A8, L9, E10, D48, D49, D51, F5...","S183, A184, S185, Q186, S187, P188, N326, P328...","L9, L65, P66, G132, N133, I134, L135, V136, S1...","L256, S313, I316, P318, M321, N326, P332, P333...","D48, D49, E50, D51, F53, L54, C55, G56, L65, F...","L116, V117, E118, T119, K305, A306, L307, N308...","Q21, L24, D25, I44, L47, D48, D49, E50, D51, L..."
...,...,...,...,...,...,...,...,...,...,...,...
334,KLF6_ZNF770,"D2, H17, E18, T19, G20, Y21, A24, L25, S27, E6...","K34, Y46, H49, T50, Q52, C60, K62, F64, H69, R...","M1, D2, V3, L4, P5, C7, Q11, E12, I15, G20, Y2...","C32, K34, H45, L47, I48, H49, T50, G51, K53, R...","D2, L4, P5, M6, C7, S8, F10, Q11, E12, Q14, I1...","N6, C32, K34, K41, R44, H45, I48, H49, G51, K5...","M6, C7, F10, Q11, L13, Q14, H17, E18, T19, G20...","R25, I31, C32, K34, I48, H49, G51, K53, K62, T...","G20, S23, A24, L25, P26, S27, E29, E30, Q34, E...","C32, K34, S40, R44, H45, I48, H49, K53, T63, F..."
335,PATZ1_TAF1,"F52, A54, H55, A61, Y65, S68, L104, E105, T108...","E141, D142, D143, Y144, D145, A146, D147, E149...","M1, E2, R3, S12, G13, C14, Y15, T16, Y17, Q18,...","M1, G2, P3, C5, L7, L8, L9, R10, T11, A12, A13...","M1, E2, R3, V4, N5, S8, C9, G10, P11, S12, Y15...","G99, W100, R102, S103, T104, K445, R634, E707,...","M1, E2, R3, V4, D6, A7, S8, G10, P11, S12, G13...","E118, E120, S121, R123, Y124, D441, V442, I631...","E2, R3, D6, A7, C9, G10, P11, G13, Y15, R21, E...","R634, E707, R918, K920, D921, A922, G923, Y924..."
336,RREB1_ZNF263,"K29, T31, E32, N33, G34, G35, S36, P37, Q38, G...","M1, A2, S3, G4, Q8, R10, G12, L13, L14, I15, V...","N18,

In [300]:
spe_cured = cured[cured['TF_Pair'].isin(list(set(cured['TF_Pair']) & set(spe['TF_Pair'])))]
spe_cured

,TF_Pair,TF1_seed1,TF2_seed1,TF1_seed2,TF2_seed2,TF1_seed3,TF2_seed3,TF1_seed4,TF2_seed4,TF1_seed5,TF2_seed5
0,AHR_MEF2A,"P18, K63, E165, R172, R225, C226, L227, D229, ...","I46, K53, L54, Q56, D61, M62, D63, L66, L67, E...","E160, I162, T164, R167, R225, L227, F234, L293...","R15, Q18, T22, K53, D61, D63, K64, L67, E71, R...","H39, L43, K63, L64, D65, K66, L67, F148, Q149,...","K4, Q7, T9, R10, I11, M12, D13, E14, R17, S50,...","I8, T9, Y10, A11, S12, R13, K14, R15, R16, K17...","M1, G2, R3, K4, D13, R15, N16, Q18, V19, T20, ...","I8, Y10, R16, S36, H39, R40, L43, H163, T164, ...","S51, N52, K53, D61, D63, K64, L67, Y69, T70, N..."
1,ARID3A_NFYC,"M54, Q55, A57, Q58, M59, A60, A61, L62, A64, M...","W23, M27, I30, R31, N32, L33, T34, V35, F38, R...","M1, K2, L3, Q4, M7, L10, Q14, R18, L21, M54, Q...","W23, M27, I30, R31, N32, L33, T34, V35, F38, R...","M1, K2, L3, Q4, V6, M7, L10, R13, Q14, R18, L2...","W23, M27, I30, R31, N32, L33, T34, V35, F38, R...","M1, K2, L3, Q4, M7, L10, Q14, R18, L21, M54, Q...","W23, M27, I30, R31, N32, L33, T34, V35, F38, R...","M1, K2, L3, V6, M7, L10, R13, Q14, A17, R18, L...","W23, P24, M27, I30, R31, N32, L33, T34, V35, F..."
2,ARID3A_IRF5,"M1, K2, L3, Q4, M7, E8, L10, L11, Q58, M59, L6...","R14, H49, T51, R52, K73, A84, A88, N89, R91, C...","L3, L10, R13, Q14, Q58, M65, P125, W127, E128,...","R14, R91, K96, R108, G247, E248, L250, L251, D...","E142, R255, P258, N260, R261, T308, S309, F312...","G247, E248, L250, L251, D253, L254, L255, I256...","M1, K2, L3, Q4, A5, V6, M7, L10, L11, R13, Q14...","M153, L154, P155, S156, L157, S158, L159, G247...","W127, E130, E131, M132, E134, D135, L136, E138...","R14, H49, T51, R52, K73, A84, A88, N89, R91, C..."
3,ARID3A_POU2F2,"L3, Q4, V6, M7, T9, L10, R13, Q58, M59, L62, M...","E217, S220, L222, E223, E226, R230, K233, L239...","K2, L3, L136, D139, E140, E141, E142, E143, E1...","L160, L161, S259, Q260, T261, T262, S264, R265...","E134, L136, G137, E138, D139, E140, E141, E142...","M6, E218, L222, E223, E226, R230, K233, R265, ...","M1, K2, L3, E90, E91, D92, A93, A94, W127, E12...","L128, Q132, V135, T188, Q189, A190, V191, E217...","V6, L10, R13, D139, E142, E143, E144, E145, D1...","Q120, D124, I125, L131, Q132, S259, Q260, T261..."
4,AHR_ZNF41,"K32, S33, N34, P35, S36, R38, H39, R40, R42, L...","L137, N152, L155, K159, K163, K303, S470, D471...","N34, P35, S36, H39, R40, K63, Q149, D152, R225...","M105, E415, C416, G417, K418, A419, H429, H433...","K32, P35, R38, H39, R40, R42, E46, R49, D65, K...","E40, D128, L130, I133, E136, L137, D140, N141,...","N2, S3, S4, A6, N7, I8, Y10, A11, R13, K14, R1...","D45, A47, Q48, R50, L51, D54, V55, E58, L130, ...","M1, N2, S3, A6, N7, I8, T9, Y10, A11, S12, R13...","M1, S21, S22, C23, E24, S26, V27, S28, D31, V3..."
...,...,...,...,...,...,...,...,...,...,...,...
555,SOX5_THRA,"L139, V142, V143, L146, K147, K150, M151, E153...","Y42, P44, S45, Y46, D48, E51, Y63, H64, Y65, R...","L139, V142, V143, L146, K147, R149, K150, E153...","M39, Y42, Y65, E71, K74, R78, L126, R131, V132...","V143, L146, K147, K150, E153, L154, T162, P163...","M39, Y42, P44, S45, Y46, D48, K49, T61, Y63, Y...","L139, V142, V143, L146, K147, K150, E153, L154...","Y42, P44, S45, Y46, D48, K49, Y63, Y65, R66, R...","L139, V142, V143, L146, K147, R149, K150, E153...","S45, Y46, L47, D48, K49, D50, Y65, E71, G72, C..."
557,THRA_ZNF282,"Y42, I43, P44, S45, Y46, L47, D48, K49, E51, T...","W151, R166, E169, N170, N173, W181, R184, P186...","Y42, P44, S45, Y46, T61, Y63, Y65, R66, E71, K...","I96, S97, L98, W99, V102, A103, I105, Q106, E1...","T61, E71, G72, K74, G75, F76, R78, R79, Q82, I...","E169, N170, N173, L174, R176, N177, F180, W181...","Y46, Y63, R66, D123, L124, L126, K134, L137, M...","N170, N173, L174, N177, F180, W181, L183, R184...","M39, S40, Y42, I43, P44, S45, Y46, T61, Y63, H...","W25, W27, A30, L31, P32, E34, R125, T126, T128..."
558,THRA_ZNF713,"K49, R145, L154, T161, P162, E163, E164, D166,...","F8, E18, E30, S31, L32, D36, V37, A3

In [301]:
com_cured = cured[cured['TF_Pair'].isin(list(set(cured['TF_Pair']) & set(com['TF_Pair'])))]
com_cured

,TF_Pair,TF1_seed1,TF2_seed1,TF1_seed2,TF2_seed2,TF1_seed3,TF2_seed3,TF1_seed4,TF2_seed4,TF1_seed5,TF2_seed5
17,ATF1_XBP1,"I25, A29, V32, S33, L35, S36, E37, E39, E40, D...","L74, K77, L78, R81, Q85, R88, K91, K92, R94, M...","A54, H55, L58, R60, R61, P62, S63, Y64, R65, K...","K91, R94, M95, S96, L98, E99, Q101, V102, V103...","Q42, D43, S44, S45, D46, K53, R61, Y64, R219, ...","R90, K91, K92, R94, M95, L98, E99, Q101, V102,...","S38, E39, S41, Q42, D43, S44, S45, D46, S50, S...","N80, R81, A83, A84, A87, R88, K91, K92, R94, M...","P16, G17, S18, A19, E37, R231, K233, K234, K23...","K77, K91, R94, M95, L98, E99, Q101, V102, L105..."
18,ATF2_XBP1,"W17, N38, E39, D40, H41, L76, L84, F85, L88, R...","L78, T86, R90, K91, R94, M95, L98, E99, Q101, ...","M1, L5, Q12, L16, M19, K23, Q34, R35, F36, T37...","R63, L74, K77, L78, N80, R81, A83, A84, A87, R...","H47, K54, P72, T73, L76, E80, A89, E93, F96, P...","P41, A42, Q43, R44, K77, R81, R88, K91, K92, R...","W17, D21, D22, F25, R35, F36, T37, N38, E39, D...","T14, V17, G29, A32, G33, L36, R44, A46, S47, P...","D21, R35, T37, N38, D40, H41, E80, L84, F85, N...","P31, G33, Q34, A35, L36, P37, L38, M39, V40, P..."
24,ATF3_XBP1,"L29, V30, D33, L37, P39, F40, V41, E43, L45, K...","Q62, L64, L67, E70, L74, R75, L78, K91, M95, L...","S28, L29, V30, F31, F34, K42, E43, L45, R46, I...","K91, R94, M95, L98, E99, Q101, V102, L105, E10...","P25, P26, G27, S28, L29, F31, D33, F34, A35, L...","Q62, R63, L64, S68, E70, E71, L74, R75, K77, L...","R104, K107, K108, K110, T111, L114, Q115, E117...","R88, K91, K92, R94, M95, L98, E99, Q101, V102,...","K89, R92, R93, N96, K97, A100, R104, K107, K10...","R88, K91, K92, M95, L98, E99, Q101, V102, L105..."
28,BACH2_MAF,"E60, W63, Q64, K98, R117, M118, H119, N120, L1...","P13, T14, S15, P16, L17, A18, Y21, V22, D26, L...","N22, L25, G26, D29, A97, K98, L99, L100, E122,...","E4, L5, A6, M7, S8, N9, S10, D11, L12, P13, S1...","P10, M11, Y12, V13, Y14, E15, S16, T17, V18, T...","S15, L17, A18, E20, Y21, N23, D24, F25, D26, K...","E15, T17, F128, R144, D146, Y234, E592, S593, ...","L101, P103, L106, F108, P110, E111, A113, V114...","M11, Y12, V13, E15, T17, K98, L99, L100, L101,...","M7, D11, L12, P13, S15, L17, A18, E20, Y21, V2..."
29,BACH1_MAF,"L3, N6, S7, V8, F9, A10, Y11, E12, S14, V15, F...","L5, M7, D11, P13, T14, S15, L17, A18, Y21, V22...","E12, S14, V15, S17, T18, N19, L22, D26, A54, K...","T14, E20, Y21, N23, F25, D26, M28, F30, E31, V...","Y11, E12, S13, S14, V15, T18, N19, L22, T93, A...","N9, L12, P13, T14, S15, L17, A18, M19, E20, Y2...","E12, K95, E119, C122, Q124, F125, L126, K127, ...","L5, M7, D11, L12, P13, S15, L17, Y21, V22, D26...","L157, D161, D164, L165, T167, D168, E171, S387...","L5, M7, D11, L12, P13, S15, L17, A18, Y21, V22..."
...,...,...,...,...,...,...,...,...,...,...,...
550,PPARA_THRA,"A97, L98, N99, I100, E101, C102, R103, I104, C...","S40, G41, Y42, I43, P44, S45, Y46, L47, D48, K...","P94, S95, G96, A97, L98, N99, I100, E101, R103...","M39, G41, Y42, I43, P44, S45, Y46, L47, A60, T...","L72, P74, A75, S76, S77, P78, H193, D194, E196...","R66, I81, N84, M122, D123, L124, V125, L126, D...","S110, G111, Y112, H113, H117, E120, K123, R127...","Y42, S45, Y46, D48, E51, Q52, V54, Y63, R66, C...","R103, V116, L162, S163, V164, G165, M166, S167...","S45, Y46, L47, E51, V54, Y63, R66, C67, I68, V..."
552,RARA_THRA,"Y19, Y24, F26, F28, N163, K165, E176, S177, T1...","S40, Y42, R146, E149, Q156, R157, P158, K220, ...","P79, P80, P82, R83, I84, Y85, K94, G97, Y98, Y...","M39, S40, G41, Y42, P44, S45, Y46, Y63, H64, Y...","Y19, P22, Y24, F26, F27, F28, P80, L81, R83, Y...","M39, Y42, P44, S45, Y46, L47, D48, E51, Y63, R...","E176, S177, T179, T181, P182, E183, Q315, I332...","R146, M150, S153, L154, Q156, R157, P158, V322...","Y24, Q117, K118, N119, M120, V121, Y122, T123,...","K147, M150, S153, L154, Q156, R157, P158, E159..."
553,RORA_THRA,"K61, S66, Q67, I68, S81, G82, I83, H84, E91, K...","V132, R135, K136, E139, Q140, E143, R144, 

In [302]:
spe_old = spe[~spe['TF_Pair'].isin(list(set(cured['TF_Pair']) & set(spe['TF_Pair'])))]
spe_old

,TF_Pair,TF1_seed1,TF2_seed1,TF1_seed2,TF2_seed2,TF1_seed3,TF2_seed3,TF1_seed4,TF2_seed4,TF1_seed5,TF2_seed5
3,ARID5A_HIC2,"P43, E50, Q53, E57, G100, A101, E103, L104, T1...","L64, K104, R137, K140, R141, Q193, A221, G222,...","K10, Q11, E14, L18, A22, S23, K25, P26, K29, Q...","M1, V2, S3, P5, L6, A7, L8, R9, W10, W13, R16,...","T13, E14, A17, L18, P21, A22, P24, K25, P26, K...","V2, S3, L6, R9, W10, W13, A14, G15, R16, D18, ...","E40, R78, V79, P80, H81, L82, G83, F84, K112, ...","C361, P362, G363, E364, E365, G366, E367, G368...","H81, G83, F84, R109, G123, S124, T125, S126, T...","M1, F44, H77, S90, F93, Q94, R164, T167, V170,..."
4,AHR_EBF1,"M1, S3, L54, P55, F56, P57, I61, L72, Y76, A79...","A26, T29, W30, M31, Q32, G33, A34, G35, E55, K...","R40, N44, D48, P55, P57, D59, I61, N62, K63, D...","G21, G23, M24, N25, A26, V27, R28, T29, N40, A...","R15, R16, K17, V19, Q20, T22, K24, H39, R40, R...","L20, G21, M24, N25, R28, G46, R51, Y75, R77, Q...","S12, R13, K14, R15, R16, K17, V19, H39, R40, L...","L49, A50, R51, H53, E55, A73, Y75, R77, Q78, G...","H39, R42, L43, E46, L47, R49, L50, L53, L54, P...","E55, Q80, P81, E83, D184, R185, F186, F187, E2..."
5,AHR_HOXA10,"S36, R40, D152, R225, D229, N230, S231, S232, ...","N27, L30, L34, I35, G38, A328, R378, K381, R39...","S36, R40, L67, L70, V74, D152, T194, L196, R22...","L30, V31, S33, L34, Y57, Y58, K323, N326, A327...","K21, V23, I26, D65, L67, S68, R71, L134, D152,...","N13, P15, T17, L252, P253, P254, A255, L256, K...","S36, H39, R40, L43, L47, K63, D65, K66, L67, L...","F29, L30, L34, S156, S158, F159, A160, N162, I...","N7, Y10, K63, L64, D65, L67, S68, R71, D132, L...","L65, P66, L71, Y73, G74, L75, Q76, C78, F81, P..."
7,AHR_MYBL1,"M1, S4, S5, I8, T9, A11, S12, R15, R16, K17, P...","Q15, D18, H19, E22, V23, Q25, Q26, L29, F71, H...","M1, N2, S3, S5, A6, N7, I8, T9, Y10, S12, R13,...","E10, D11, L14, Q15, D18, D20, Y21, V23, P24, Q...","M1, N2, S3, A6, T9, Y10, S12, R13, K14, R16, K...","D11, L14, Q15, D18, Y21, E22, Q25, N159, K177,...","M1, N2, S3, S4, S5, A6, I8, Y10, V23, K24, I26...","K3, R6, S7, E10, D13, L14, Y16, A17, D20, P24,...","M1, N2, S3, S4, S5, A6, N7, I8, Y10, A11, R13,...","D41, D44, K47, K48, E51, Q52, P83, E84, L85, I..."
8,AHR_MEF2D,"Y10, A11, S12, R13, K14, R15, R16, K21, K24, N...","D63, L66, L67, E71, N73, E74, P75, H76, E77, S...","N2, S3, A6, N7, Y10, K14, K17, Q20, K21, K24, ...","M1, G2, R3, R15, N16, V19, T20, K23, D102, G10...","Y10, R15, K21, T22, K24, P25, I26, P27, A28, E...","R3, L35, C39, Y72, E74, P75, H76, E77, S78, R7...","I8, T9, S12, R13, R15, R16, V19, Q20, V23, I31...","L54, Q56, D61, D63, L66, L67, Y69, T70, E71, Y...","S4, S5, A6, N7, I8, T9, Y10, A11, S12, R13, K1...","M1, G2, R3, K4, T22, K23, F26, K30, K31, Y33, ..."
...,...,...,...,...,...,...,...,...,...,...,...
1878,AR_ZNF354C,"R13, Y19, R20, F23, Q24, L26, F27, V30, R31, I...","E10, D16, V17, V19, F20, F21, S22, E25, H28, L...","R13, S16, K17, T18, Y19, R20, F23, Q24, L26, F...","M1, A2, V3, L5, L6, V12, D16, V17, A18, V19, F...","L26, F27, V30, R31, V33, I34, Q35, N36, P37, P...","S7, Q9, E10, V12, D16, V17, V19, F20, F21, E25...","R13, K17, Y19, A22, F23, N25, L26, S29, V30, V...","Q9, V12, R15, D16, V17, A18, F21, E25, H28, L2...","R20, F23, Q24, F27, V30, R31, I34, Q35, I232, ...","V12, R15, D16, V17, A18, V19, F20, F21, S22, E..."
1879,PRDM4_ZNF467,"E360, N365, M369, A370, T371, L372, F373, T374...","P82, V83, G84, T85, C86, P87, G88, E89, E90, W...","N89, Y90, T91, L92, P93, P95, P98, H99, L100, ...","E108, V109, W111, P112, Q113, H114, L115, S116...","L109, A370, T371, F373, I375, P385, D387, F396...","P87, G88, E89, E90, W91, M92, I93, R94, K95, V...","F105, R405, L408, S409, P411, R436, C438, G440...","G88, E89, E90, W91, M92, I93, R94, K95, E99, E...","E87, L92, P93, L109, L113, V345, S347, D348, S...","K97, E104, A105, E106, E108, E110, W111, P112,..."
1880,PRDM4_ZNF770,"L85, P86, E87, R88, N89, Y90, T91, L92, P93, P...","N19, L21, N24, P26, Y27, V28, C29, N

In [303]:
com_old = com[~com['TF_Pair'].isin(list(set(cured['TF_Pair']) & set(com['TF_Pair'])))]
com_old

,TF_Pair,TF1_seed1,TF2_seed1,TF1_seed2,TF2_seed2,TF1_seed3,TF2_seed3,TF1_seed4,TF2_seed4,TF1_seed5,TF2_seed5
0,JDP2_JUND,"G14, S15, L16, P17, K93, R96, T97, L100, Q101,...","F25, A26, S27, P28, L290, R292, I293, L296, E2...","R90, K93, R96, T97, L100, Q101, E103, S104, L1...","K289, L290, R292, I293, L296, E297, K299, V300...","K93, R96, T97, L100, Q101, E103, S104, L107, E...","L290, R292, I293, L296, E297, K299, V300, L303...","K93, R96, T97, L100, Q101, E103, S104, L107, E...","F136, G139, F140, K142, A143, L144, D146, L147...","T12, G14, K93, R96, T97, L100, Q101, E103, S10...","L54, S55, E56, S100, L103, I107, E133, F136, A..."
1,EGR1_TBX21,"M39, L100, T101, A102, E103, F247, V249, P250,...","P128, T160, R164, R165, F167, R184, F186, D188...","Y32, L35, M39, V249, P250, M251, I252, P253, D...","L131, R138, A140, N143, T160, Q162, R164, R165...","Q15, V249, P250, M251, I252, P253, Y255, L256,...","G14, T15, I158, R164, R184, F186, D188, V190, ...","Y32, L35, M39, F247, V249, P250, M251, I252, P...","L131, R138, A140, N143, I158, T160, Q162, G163...","Y32, L35, E36, M38, M39, N43, E219, V249, P250...","Y125, L127, P128, L131, R138, A140, T160, R164..."
2,EGR1_THAP1,"R371, I372, C373, I387, T389, H390, T391, G392","M1, V2, Q3, Y14, F22, K24, K49, C67, N69, K70,...","R371, I372, C373, M374, R388, T389, H390, T391...","M1, V2, Q3, Y14, F22, K24, N69, L71","L307, S310, Y311, Q312, S313, Q314, L315, I316...","Q3, Y14, D15, K16, K18, P19, V20, S21, F22, H2...","Y311, P367, F368, Q369, R371, I372, C373, M374...","M1, Q3, Y14, K18, P19, V20, S21, F22, H23, K24...","Y311, Q314, L315, C373, M374, A397, D399, I400...","K24, K49, Y50, E66, C67, N68, N69, L71, R169, ..."
3,EGR1_ZNF263,"L304, N308, Y311, Q312, L315, I316, P318, R320...","M74, T76, E78, E82, E103, L104, H105, E107, E1...","S319, R320, M321, R322, Y324, I361, K366, R371...","Q95, M210, P213, L215, P216, E248, T606, L607,...","H334, E335, P337, Y338, S350, R351, S352, D353...","P72, E73, M74, R75, T76, K77, E78, L437, E438,...","M321, E343, D346, Q369, C373, M374, R375, N376","L379, C380, P381, L382, C383, G384, T447, Y490...","P318, S319, R320, M321, K323, Y324, P325, N326...","L28, H67, P72, E73, M74, R75, T76, L198, F199,..."
4,EGR1_ZNF341,"H25, S26, P27, T28, C192, A193, V194, P195, K3...","S203, L204, G205, P206, P207, N211, P212, G213...","T281, Q282, Q283, P284, S285, L286, P288, T291...","Q3, A4, F6, E7, A8, L9, E10, D48, D49, D51, F5...","S183, A184, S185, Q186, S187, P188, N326, P328...","L9, L65, P66, G132, N133, I134, L135, V136, S1...","L256, S313, I316, P318, M321, N326, P332, P333...","D48, D49, E50, D51, F53, L54, C55, G56, L65, F...","L116, V117, E118, T119, K305, A306, L307, N308...","Q21, L24, D25, I44, L47, D48, D49, E50, D51, L..."
...,...,...,...,...,...,...,...,...,...,...,...
329,KLF6_MAZ,"D2, E29, S45, E46, P47, F57, D58, Q60, E61, D6...","I191, C192, A193, L194, C195, A196, E198, T268...","M1, D2, H17, E18, T19, G20, S83, P85, S233, L2...","I191, C192, A193, L194, C195, A196, K200, H212...","C7, Q11, Q14, H17, E18, Q60, E61, W64, R71, E7...","G45, L48, Q49, R51, F52, F53, Q64, C339, S340,...","S8, E12, T19, Y21, A24, L25, P26, S233, L247, ...","K305, Q314, R321, P336, Y337, N338, S340, L352...","Y21, S233, L247, T248, F251, R252, T255, A257,...","R364, P393, C394, H395, V396, C397, G398, M400..."
331,KLF6_THAP1,"M1, D2, P5, M6, S8, I9, E12, L13, V16, Y21, A2...","M1, V2, S4, S6, A7, Y8, R13, K16, R42, F45, T4...","S192, D194, R252, K253, H254, G256, A257, K258...","K16, D17, E66, C67, N68, N69, F186, E189","M1, D2, P5, M6, I9, E12, L13, V16, Y21, F22, L...","T28, K64, E66, C67, N68, N69, L71, L72, D141, ...","M1, D2, P5, M6, S8, I9, E12, L13, V16, T19, Y2...","Q3, Y14, K16, K18, V20, S21, F22, H23, E66, N6...","E12, I15, V16, T19, G20, Y21, S23, A24, L25, P...","M1, V2, Q3, Y14, F22, K49, Y50, N68, N69, Y137..."
332,KLF6_ZNF263,"M1, D2, P5, C7, S8, I9, Q11, E12, Q14, I15, V1...","L14, V16, L18, W24, P33, G34, S36, P37, E38, A...","V3, H17, G20, Y21

In [304]:
spe = pd.concat([spe_old, spe_cured])
spe

,TF_Pair,TF1_seed1,TF2_seed1,TF1_seed2,TF2_seed2,TF1_seed3,TF2_seed3,TF1_seed4,TF2_seed4,TF1_seed5,TF2_seed5
3,ARID5A_HIC2,"P43, E50, Q53, E57, G100, A101, E103, L104, T1...","L64, K104, R137, K140, R141, Q193, A221, G222,...","K10, Q11, E14, L18, A22, S23, K25, P26, K29, Q...","M1, V2, S3, P5, L6, A7, L8, R9, W10, W13, R16,...","T13, E14, A17, L18, P21, A22, P24, K25, P26, K...","V2, S3, L6, R9, W10, W13, A14, G15, R16, D18, ...","E40, R78, V79, P80, H81, L82, G83, F84, K112, ...","C361, P362, G363, E364, E365, G366, E367, G368...","H81, G83, F84, R109, G123, S124, T125, S126, T...","M1, F44, H77, S90, F93, Q94, R164, T167, V170,..."
4,AHR_EBF1,"M1, S3, L54, P55, F56, P57, I61, L72, Y76, A79...","A26, T29, W30, M31, Q32, G33, A34, G35, E55, K...","R40, N44, D48, P55, P57, D59, I61, N62, K63, D...","G21, G23, M24, N25, A26, V27, R28, T29, N40, A...","R15, R16, K17, V19, Q20, T22, K24, H39, R40, R...","L20, G21, M24, N25, R28, G46, R51, Y75, R77, Q...","S12, R13, K14, R15, R16, K17, V19, H39, R40, L...","L49, A50, R51, H53, E55, A73, Y75, R77, Q78, G...","H39, R42, L43, E46, L47, R49, L50, L53, L54, P...","E55, Q80, P81, E83, D184, R185, F186, F187, E2..."
5,AHR_HOXA10,"S36, R40, D152, R225, D229, N230, S231, S232, ...","N27, L30, L34, I35, G38, A328, R378, K381, R39...","S36, R40, L67, L70, V74, D152, T194, L196, R22...","L30, V31, S33, L34, Y57, Y58, K323, N326, A327...","K21, V23, I26, D65, L67, S68, R71, L134, D152,...","N13, P15, T17, L252, P253, P254, A255, L256, K...","S36, H39, R40, L43, L47, K63, D65, K66, L67, L...","F29, L30, L34, S156, S158, F159, A160, N162, I...","N7, Y10, K63, L64, D65, L67, S68, R71, D132, L...","L65, P66, L71, Y73, G74, L75, Q76, C78, F81, P..."
7,AHR_MYBL1,"M1, S4, S5, I8, T9, A11, S12, R15, R16, K17, P...","Q15, D18, H19, E22, V23, Q25, Q26, L29, F71, H...","M1, N2, S3, S5, A6, N7, I8, T9, Y10, S12, R13,...","E10, D11, L14, Q15, D18, D20, Y21, V23, P24, Q...","M1, N2, S3, A6, T9, Y10, S12, R13, K14, R16, K...","D11, L14, Q15, D18, Y21, E22, Q25, N159, K177,...","M1, N2, S3, S4, S5, A6, I8, Y10, V23, K24, I26...","K3, R6, S7, E10, D13, L14, Y16, A17, D20, P24,...","M1, N2, S3, S4, S5, A6, N7, I8, Y10, A11, R13,...","D41, D44, K47, K48, E51, Q52, P83, E84, L85, I..."
8,AHR_MEF2D,"Y10, A11, S12, R13, K14, R15, R16, K21, K24, N...","D63, L66, L67, E71, N73, E74, P75, H76, E77, S...","N2, S3, A6, N7, Y10, K14, K17, Q20, K21, K24, ...","M1, G2, R3, R15, N16, V19, T20, K23, D102, G10...","Y10, R15, K21, T22, K24, P25, I26, P27, A28, E...","R3, L35, C39, Y72, E74, P75, H76, E77, S78, R7...","I8, T9, S12, R13, R15, R16, V19, Q20, V23, I31...","L54, Q56, D61, D63, L66, L67, Y69, T70, E71, Y...","S4, S5, A6, N7, I8, T9, Y10, A11, S12, R13, K1...","M1, G2, R3, K4, T22, K23, F26, K30, K31, Y33, ..."
...,...,...,...,...,...,...,...,...,...,...,...
555,SOX5_THRA,"L139, V142, V143, L146, K147, K150, M151, E153...","Y42, P44, S45, Y46, D48, E51, Y63, H64, Y65, R...","L139, V142, V143, L146, K147, R149, K150, E153...","M39, Y42, Y65, E71, K74, R78, L126, R131, V132...","V143, L146, K147, K150, E153, L154, T162, P163...","M39, Y42, P44, S45, Y46, D48, K49, T61, Y63, Y...","L139, V142, V143, L146, K147, K150, E153, L154...","Y42, P44, S45, Y46, D48, K49, Y63, Y65, R66, R...","L139, V142, V143, L146, K147, R149, K150, E153...","S45, Y46, L47, D48, K49, D50, Y65, E71, G72, C..."
557,THRA_ZNF282,"Y42, I43, P44, S45, Y46, L47, D48, K49, E51, T...","W151, R166, E169, N170, N173, W181, R184, P186...","Y42, P44, S45, Y46, T61, Y63, Y65, R66, E71, K...","I96, S97, L98, W99, V102, A103, I105, Q106, E1...","T61, E71, G72, K74, G75, F76, R78, R79, Q82, I...","E169, N170, N173, L174, R176, N177, F180, W181...","Y46, Y63, R66, D123, L124, L126, K134, L137, M...","N170, N173, L174, N177, F180, W181, L183, R184...","M39, S40, Y42, I43, P44, S45, Y46, T61, Y63, H...","W25, W27, A30, L31, P32, E34, R125, T126, T128..."
558,THRA_ZNF713,"K49, R145, L154, T161, P162, E163, E164, D166,...","F8, E18, E30, S31, L32, D36, V37, A38, V39

In [305]:
spe.columns

Index(['TF_Pair', 'TF1_seed1', 'TF2_seed1', 'TF1_seed2', 'TF2_seed2',
       'TF1_seed3', 'TF2_seed3', 'TF1_seed4', 'TF2_seed4', 'TF1_seed5',
       'TF2_seed5'],
      dtype='object')

In [306]:
import pandas as pd

# Assuming 'spe' is the DataFrame you have already created
# spe = pd.concat([spe_old, spe_cured])

# --- Step 1: Dynamically identify relevant columns for TF1 and TF2 ---
# Filter all columns starting with 'TF1_seed'
tf1_cols = [col for col in spe.columns if col.startswith('TF1_seed')]
# Filter all columns starting with 'TF2_seed'
tf2_cols = [col for col in spe.columns if col.startswith('TF2_seed')]

print("Identified TF1 related columns:", tf1_cols)
print("Identified TF2 related columns:", tf2_cols)
print("\n")


# --- Step 2: Define helper functions to process strings and perform set operations ---
def get_sites_from_string(site_string):
    """Converts a string like 'P43, E50, Q53' into a set {'P43', 'E50', 'Q53'}."""
    # First check if it is a valid string to prevent issues with null values (NaN)
    if not isinstance(site_string, str):
        return set() # Return an empty set if the value is null
    # Split the string and strip whitespace around each element
    return set(site.strip() for site in site_string.split(','))

def calculate_set_operation(row, target_cols, operation='union'):
    """
    Calculates the union or intersection of sets for specified columns in a DataFrame row.
    
    :param row: A row of the DataFrame (a Series object)
    :param target_cols: List of column names to process (e.g., tf1_cols)
    :param operation: 'union' or 'intersection'
    """
    # Extract data from all target columns in this row and convert them to a list of sets using the function above
    list_of_sets = [get_sites_from_string(row[col]) for col in target_cols]
    
    if not list_of_sets:
        return "" # Return empty string if no target columns

    if operation == 'union':
        # set.union(*list_of_sets) calculates the union of multiple sets
        result_set = set.union(*list_of_sets)
    elif operation == 'intersection':
        # set.intersection(*list_of_sets) calculates the intersection of multiple sets
        result_set = set.intersection(*list_of_sets)
    else:
        return ""

    # Convert the final set back to a sorted, comma-separated string for easy viewing
    return ', '.join(sorted(list(result_set)))


# --- Step 3: Apply the function to each row of the DataFrame to calculate new columns ---
print("Calculating unions and intersections, this may take some time...")

# axis=1 means applying the function row-wise
spe['TF1_Union'] = spe.apply(lambda row: calculate_set_operation(row, tf1_cols, 'union'), axis=1)
spe['TF1_Intersection'] = spe.apply(lambda row: calculate_set_operation(row, tf1_cols, 'intersection'), axis=1)

spe['TF2_Union'] = spe.apply(lambda row: calculate_set_operation(row, tf2_cols, 'union'), axis=1)
spe['TF2_Intersection'] = spe.apply(lambda row: calculate_set_operation(row, tf2_cols, 'intersection'), axis=1)

print("Calculation complete!")


# --- Step 4: Display final results ---
print("\nPreview of DataFrame with four new feature columns added:")

# For better display, select only key columns for preview
display_cols = ['TF_Pair', 'TF1_seed1', 'TF1_seed2', 'TF1_Union', 'TF1_Intersection']
print(spe[display_cols].head())

识别到的TF1相关列: ['TF1_seed1', 'TF1_seed2', 'TF1_seed3', 'TF1_seed4', 'TF1_seed5']
识别到的TF2相关列: ['TF2_seed1', 'TF2_seed2', 'TF2_seed3', 'TF2_seed4', 'TF2_seed5']


正在计算并集和交集，这可能需要一些时间...
计算完成！

添加了四列新特征后的 DataFrame 预览:
       TF_Pair                                          TF1_seed1  \
3  ARID5A_HIC2  P43, E50, Q53, E57, G100, A101, E103, L104, T1...   
4     AHR_EBF1  M1, S3, L54, P55, F56, P57, I61, L72, Y76, A79...   
5   AHR_HOXA10  S36, R40, D152, R225, D229, N230, S231, S232, ...   
7    AHR_MYBL1  M1, S4, S5, I8, T9, A11, S12, R15, R16, K17, P...   
8    AHR_MEF2D  Y10, A11, S12, R13, K14, R15, R16, K21, K24, N...   

                                           TF1_seed2  \
3  K10, Q11, E14, L18, A22, S23, K25, P26, K29, Q...   
4  R40, N44, D48, P55, P57, D59, I61, N62, K63, D...   
5  S36, R40, L67, L70, V74, D152, T194, L196, R22...   
7  M1, N2, S3, S5, A6, N7, I8, T9, Y10, S12, R13,...   
8  N2, S3, A6, N7, Y10, K14, K17, Q20, K21, K24, ...   

                                   

In [307]:
spe

,TF_Pair,TF1_seed1,TF2_seed1,TF1_seed2,TF2_seed2,TF1_seed3,TF2_seed3,TF1_seed4,TF2_seed4,TF1_seed5,TF2_seed5,TF1_Union,TF1_Intersection,TF2_Union,TF2_Intersection
3,ARID5A_HIC2,"P43, E50, Q53, E57, G100, A101, E103, L104, T1...","L64, K104, R137, K140, R141, Q193, A221, G222,...","K10, Q11, E14, L18, A22, S23, K25, P26, K29, Q...","M1, V2, S3, P5, L6, A7, L8, R9, W10, W13, R16,...","T13, E14, A17, L18, P21, A22, P24, K25, P26, K...","V2, S3, L6, R9, W10, W13, A14, G15, R16, D18, ...","E40, R78, V79, P80, H81, L82, G83, F84, K112, ...","C361, P362, G363, E364, E365, G366, E367, G368...","H81, G83, F84, R109, G123, S124, T125, S126, T...","M1, F44, H77, S90, F93, Q94, R164, T167, V170,...","A101, A17, A182, A22, A265, A272, A284, A288, ...",,"A14, A221, A270, A271, A346, A379, A382, A405,...","D564, E565, M580"
4,AHR_EBF1,"M1, S3, L54, P55, F56, P57, I61, L72, Y76, A79...","A26, T29, W30, M31, Q32, G33, A34, G35, E55, K...","R40, N44, D48, P55, P57, D59, I61, N62, K63, D...","G21, G23, M24, N25, A26, V27, R28, T29, N40, A...","R15, R16, K17, V19, Q20, T22, K24, H39, R40, R...","L20, G21, M24, N25, R28, G46, R51, Y75, R77, Q...","S12, R13, K14, R15, R16, K17, V19, H39, R40, L...","L49, A50, R51, H53, E55, A73, Y75, R77, Q78, G...","H39, R42, L43, E46, L47, R49, L50, L53, L54, P...","E55, Q80, P81, E83, D184, R185, F186, F187, E2...","A104, A121, A193, A267, A411, A79, A86, C219, ...","F83, K80, L87, N111, P55, Y76","A227, A26, A34, A346, A382, A383, A388, A404, ...","A382, E387, I400, L359, L379, L385, L389, Q397..."
5,AHR_HOXA10,"S36, R40, D152, R225, D229, N230, S231, S232, ...","N27, L30, L34, I35, G38, A328, R378, K381, R39...","S36, R40, L67, L70, V74, D152, T194, L196, R22...","L30, V31, S33, L34, Y57, Y58, K323, N326, A327...","K21, V23, I26, D65, L67, S68, R71, L134, D152,...","N13, P15, T17, L252, P253, P254, A255, L256, K...","S36, H39, R40, L43, L47, K63, D65, K66, L67, L...","F29, L30, L34, S156, S158, F159, A160, N162, I...","N7, Y10, K63, L64, D65, L67, S68, R71, D132, L...","L65, P66, L71, Y73, G74, L75, Q76, C78, F81, P...","A193, C226, D132, D152, D229, D65, E160, E279,...","D152, R225","A160, A255, A327, A328, A405, C170, C280, C78,...",
7,AHR_MYBL1,"M1, S4, S5, I8, T9, A11, S12, R15, R16, K17, P...","Q15, D18, H19, E22, V23, Q25, Q26, L29, F71, H...","M1, N2, S3, S5, A6, N7, I8, T9, Y10, S12, R13,...","E10, D11, L14, Q15, D18, D20, Y21, V23, P24, Q...","M1, N2, S3, A6, T9, Y10, S12, R13, K14, R16, K...","D11, L14, Q15, D18, Y21, E22, Q25, N159, K177,...","M1, N2, S3, S4, S5, A6, I8, Y10, V23, K24, I26...","K3, R6, S7, E10, D13, L14, Y16, A17, D20, P24,...","M1, N2, S3, S4, S5, A6, N7, I8, Y10, A11, R13,...","D41, D44, K47, K48, E51, Q52, P83, E84, L85, I...","A104, A11, A121, A168, A236, A28, A302, A327, ...","D144, D229, D59, E345, E390, F234, F404, G147,...","A162, A17, A213, A278, A384, A385, A404, A445,...","D586, D725, E584, E731, I728, L591, M729, P542..."
8,AHR_MEF2D,"Y10, A11, S12, R13, K14, R15, R16, K21, K24, N...","D63, L66, L67, E71, N73, E74, P75, H76, E77, S...","N2, S3, A6, N7, Y10, K14, K17, Q20, K21, K24, ...","M1, G2, R3, R15, N16, V19, T20, K23, D102, G10...","Y10, R15, K21, T22, K24, P25, I26, P27, A28, E...","R3, L35, C39, Y72, E74, P75, H76, E77, S78, R7...","I8, T9, S12, R13, R15, R16, V19, Q20, V23, I31...","L54, Q56, D61, D63, L66, L67, Y69, T70, E71, Y...","S4, S5, A6, N7, I8, T9, Y10, A11, S12, R13, K1...","M1, G2, R3, K4, T22, K23, F26, K30, K31, Y33, ...","A11, A193, A267, A28, A441, A6, A766, A768, A7...","D152, E160, F234, H39, L43, P35, R225","A120, A202, A228, A295, A342, A349, A363, A426...",K521
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
555,SOX5_THRA,"L139, V142, V143, L146, K147, K150, M151, E153...","Y42, P44, S45, Y46, D48, E51, Y63, H64, Y65, R...","L139, V142, V143, L146, K147, R149, K150, E153...","M39, Y42, Y65, E71, K74, R78, L126, R131, V132...","V143, L146, K147, K150, E153, L154, T162, P163...","M39, Y42, P44, S45, Y46, D48, K49, T61, Y63, Y...

In [309]:
spe.to_csv('./result/spe_1888_sites.csv')

In [310]:
com = pd.concat([com_old, com_cured])
com

,TF_Pair,TF1_seed1,TF2_seed1,TF1_seed2,TF2_seed2,TF1_seed3,TF2_seed3,TF1_seed4,TF2_seed4,TF1_seed5,TF2_seed5
0,JDP2_JUND,"G14, S15, L16, P17, K93, R96, T97, L100, Q101,...","F25, A26, S27, P28, L290, R292, I293, L296, E2...","R90, K93, R96, T97, L100, Q101, E103, S104, L1...","K289, L290, R292, I293, L296, E297, K299, V300...","K93, R96, T97, L100, Q101, E103, S104, L107, E...","L290, R292, I293, L296, E297, K299, V300, L303...","K93, R96, T97, L100, Q101, E103, S104, L107, E...","F136, G139, F140, K142, A143, L144, D146, L147...","T12, G14, K93, R96, T97, L100, Q101, E103, S10...","L54, S55, E56, S100, L103, I107, E133, F136, A..."
1,EGR1_TBX21,"M39, L100, T101, A102, E103, F247, V249, P250,...","P128, T160, R164, R165, F167, R184, F186, D188...","Y32, L35, M39, V249, P250, M251, I252, P253, D...","L131, R138, A140, N143, T160, Q162, R164, R165...","Q15, V249, P250, M251, I252, P253, Y255, L256,...","G14, T15, I158, R164, R184, F186, D188, V190, ...","Y32, L35, M39, F247, V249, P250, M251, I252, P...","L131, R138, A140, N143, I158, T160, Q162, G163...","Y32, L35, E36, M38, M39, N43, E219, V249, P250...","Y125, L127, P128, L131, R138, A140, T160, R164..."
2,EGR1_THAP1,"R371, I372, C373, I387, T389, H390, T391, G392","M1, V2, Q3, Y14, F22, K24, K49, C67, N69, K70,...","R371, I372, C373, M374, R388, T389, H390, T391...","M1, V2, Q3, Y14, F22, K24, N69, L71","L307, S310, Y311, Q312, S313, Q314, L315, I316...","Q3, Y14, D15, K16, K18, P19, V20, S21, F22, H2...","Y311, P367, F368, Q369, R371, I372, C373, M374...","M1, Q3, Y14, K18, P19, V20, S21, F22, H23, K24...","Y311, Q314, L315, C373, M374, A397, D399, I400...","K24, K49, Y50, E66, C67, N68, N69, L71, R169, ..."
3,EGR1_ZNF263,"L304, N308, Y311, Q312, L315, I316, P318, R320...","M74, T76, E78, E82, E103, L104, H105, E107, E1...","S319, R320, M321, R322, Y324, I361, K366, R371...","Q95, M210, P213, L215, P216, E248, T606, L607,...","H334, E335, P337, Y338, S350, R351, S352, D353...","P72, E73, M74, R75, T76, K77, E78, L437, E438,...","M321, E343, D346, Q369, C373, M374, R375, N376","L379, C380, P381, L382, C383, G384, T447, Y490...","P318, S319, R320, M321, K323, Y324, P325, N326...","L28, H67, P72, E73, M74, R75, T76, L198, F199,..."
4,EGR1_ZNF341,"H25, S26, P27, T28, C192, A193, V194, P195, K3...","S203, L204, G205, P206, P207, N211, P212, G213...","T281, Q282, Q283, P284, S285, L286, P288, T291...","Q3, A4, F6, E7, A8, L9, E10, D48, D49, D51, F5...","S183, A184, S185, Q186, S187, P188, N326, P328...","L9, L65, P66, G132, N133, I134, L135, V136, S1...","L256, S313, I316, P318, M321, N326, P332, P333...","D48, D49, E50, D51, F53, L54, C55, G56, L65, F...","L116, V117, E118, T119, K305, A306, L307, N308...","Q21, L24, D25, I44, L47, D48, D49, E50, D51, L..."
...,...,...,...,...,...,...,...,...,...,...,...
550,PPARA_THRA,"A97, L98, N99, I100, E101, C102, R103, I104, C...","S40, G41, Y42, I43, P44, S45, Y46, L47, D48, K...","P94, S95, G96, A97, L98, N99, I100, E101, R103...","M39, G41, Y42, I43, P44, S45, Y46, L47, A60, T...","L72, P74, A75, S76, S77, P78, H193, D194, E196...","R66, I81, N84, M122, D123, L124, V125, L126, D...","S110, G111, Y112, H113, H117, E120, K123, R127...","Y42, S45, Y46, D48, E51, Q52, V54, Y63, R66, C...","R103, V116, L162, S163, V164, G165, M166, S167...","S45, Y46, L47, E51, V54, Y63, R66, C67, I68, V..."
552,RARA_THRA,"Y19, Y24, F26, F28, N163, K165, E176, S177, T1...","S40, Y42, R146, E149, Q156, R157, P158, K220, ...","P79, P80, P82, R83, I84, Y85, K94, G97, Y98, Y...","M39, S40, G41, Y42, P44, S45, Y46, Y63, H64, Y...","Y19, P22, Y24, F26, F27, F28, P80, L81, R83, Y...","M39, Y42, P44, S45, Y46, L47, D48, E51, Y63, R...","E176, S177, T179, T181, P182, E183, Q315, I332...","R146, M150, S153, L154, Q156, R157, P158, V322...","Y24, Q117, K118, N119, M120, V121, Y122, T123,...","K147, M150, S153, L154, Q156, R157, P158, E159..."
553,RORA_THRA,"K61, S66, Q67, I68, S81, G82, I83, H84, E91, K...","V132, R135, K136, E139, Q140, E143, R144, K147...","R98, T120

In [311]:
import pandas as pd

# Assuming 'com' is the DataFrame you have already created (renamed from spe in previous context)
# spe = pd.concat([spe_old, spe_cured])

# --- Step 1: Dynamically identify relevant columns for TF1 and TF2 ---
# Filter all columns starting with 'TF1_seed'
tf1_cols = [col for col in com.columns if col.startswith('TF1_seed')]
# Filter all columns starting with 'TF2_seed'
tf2_cols = [col for col in com.columns if col.startswith('TF2_seed')]

print("Identified TF1 related columns:", tf1_cols)
print("Identified TF2 related columns:", tf2_cols)
print("\n")


# --- Step 2: Define helper functions to process strings and perform set operations ---
def get_sites_from_string(site_string):
    """Converts a string like 'P43, E50, Q53' into a set {'P43', 'E50', 'Q53'}."""
    # First check if it is a valid string to prevent issues with null values (NaN)
    if not isinstance(site_string, str):
        return set() # Return an empty set if the value is null
    # Split the string and strip whitespace around each element
    return set(site.strip() for site in site_string.split(','))

def calculate_set_operation(row, target_cols, operation='union'):
    """
    Calculates the union or intersection of sets for specified columns in a DataFrame row.
    
    :param row: A row of the DataFrame (a Series object)
    :param target_cols: List of column names to process (e.g., tf1_cols)
    :param operation: 'union' or 'intersection'
    """
    # Extract data from all target columns in this row and convert them to a list of sets using the function above
    list_of_sets = [get_sites_from_string(row[col]) for col in target_cols]
    
    if not list_of_sets:
        return "" # Return empty string if no target columns

    if operation == 'union':
        # set.union(*list_of_sets) calculates the union of multiple sets
        result_set = set.union(*list_of_sets)
    elif operation == 'intersection':
        # set.intersection(*list_of_sets) calculates the intersection of multiple sets
        result_set = set.intersection(*list_of_sets)
    else:
        return ""

    # Convert the final set back to a sorted, comma-separated string for easy viewing
    return ', '.join(sorted(list(result_set)))


# --- Step 3: Apply the function to each row of the DataFrame to calculate new columns ---
print("Calculating unions and intersections, this may take some time...")

# axis=1 means applying the function row-wise
com['TF1_Union'] = com.apply(lambda row: calculate_set_operation(row, tf1_cols, 'union'), axis=1)
com['TF1_Intersection'] = com.apply(lambda row: calculate_set_operation(row, tf1_cols, 'intersection'), axis=1)

com['TF2_Union'] = com.apply(lambda row: calculate_set_operation(row, tf2_cols, 'union'), axis=1)
com['TF2_Intersection'] = com.apply(lambda row: calculate_set_operation(row, tf2_cols, 'intersection'), axis=1)

print("Calculation complete!")


# --- Step 4: Display final results ---
print("\nPreview of DataFrame with four new feature columns added:")

# For better display, select only key columns for preview
display_cols = ['TF_Pair', 'TF1_seed1', 'TF1_seed2', 'TF1_Union', 'TF1_Intersection']
print(com[display_cols].head())

识别到的TF1相关列: ['TF1_seed1', 'TF1_seed2', 'TF1_seed3', 'TF1_seed4', 'TF1_seed5']
识别到的TF2相关列: ['TF2_seed1', 'TF2_seed2', 'TF2_seed3', 'TF2_seed4', 'TF2_seed5']


正在计算并集和交集，这可能需要一些时间...
计算完成！

添加了四列新特征后的 DataFrame 预览:
       TF_Pair                                          TF1_seed1  \
0    JDP2_JUND  G14, S15, L16, P17, K93, R96, T97, L100, Q101,...   
1   EGR1_TBX21  M39, L100, T101, A102, E103, F247, V249, P250,...   
2   EGR1_THAP1     R371, I372, C373, I387, T389, H390, T391, G392   
3  EGR1_ZNF263  L304, N308, Y311, Q312, L315, I316, P318, R320...   
4  EGR1_ZNF341  H25, S26, P27, T28, C192, A193, V194, P195, K3...   

                                           TF1_seed2  \
0  R90, K93, R96, T97, L100, Q101, E103, S104, L1...   
1  Y32, L35, M39, V249, P250, M251, I252, P253, D...   
2  R371, I372, C373, M374, R388, T389, H390, T391...   
3  S319, R320, M321, R322, Y324, I361, K366, R371...   
4  T281, Q282, Q283, P284, S285, L286, P288, T291...   

                                   

In [312]:
com

,TF_Pair,TF1_seed1,TF2_seed1,TF1_seed2,TF2_seed2,TF1_seed3,TF2_seed3,TF1_seed4,TF2_seed4,TF1_seed5,TF2_seed5,TF1_Union,TF1_Intersection,TF2_Union,TF2_Intersection
0,JDP2_JUND,"G14, S15, L16, P17, K93, R96, T97, L100, Q101,...","F25, A26, S27, P28, L290, R292, I293, L296, E2...","R90, K93, R96, T97, L100, Q101, E103, S104, L1...","K289, L290, R292, I293, L296, E297, K299, V300...","K93, R96, T97, L100, Q101, E103, S104, L107, E...","L290, R292, I293, L296, E297, K299, V300, L303...","K93, R96, T97, L100, Q101, E103, S104, L107, E...","F136, G139, F140, K142, A143, L144, D146, L147...","T12, G14, K93, R96, T97, L100, Q101, E103, S10...","L54, S55, E56, S100, L103, I107, E133, F136, A...","C139, D144, E103, E108, E124, E152, G14, H135,...","C139, E103, E108, E124, H135, I118, I129, I140...","A137, A143, A26, A311, A314, C336, D146, E133,...","A311, A314, C336, E297, G335, H331, I293, K299..."
1,EGR1_TBX21,"M39, L100, T101, A102, E103, F247, V249, P250,...","P128, T160, R164, R165, F167, R184, F186, D188...","Y32, L35, M39, V249, P250, M251, I252, P253, D...","L131, R138, A140, N143, T160, Q162, R164, R165...","Q15, V249, P250, M251, I252, P253, Y255, L256,...","G14, T15, I158, R164, R184, F186, D188, V190, ...","Y32, L35, M39, F247, V249, P250, M251, I252, P...","L131, R138, A140, N143, I158, T160, Q162, G163...","Y32, L35, E36, M38, M39, N43, E219, V249, P250...","Y125, L127, P128, L131, R138, A140, T160, R164...","A102, A397, A462, C373, D254, D399, E103, E219...","C373, E393, F257, F396, H390, I252, I372, I387...","A140, A175, A230, A282, A304, A308, A321, A435...","A321, C284, D188, E281, F186, F324, G216, H271..."
2,EGR1_THAP1,"R371, I372, C373, I387, T389, H390, T391, G392","M1, V2, Q3, Y14, F22, K24, K49, C67, N69, K70,...","R371, I372, C373, M374, R388, T389, H390, T391...","M1, V2, Q3, Y14, F22, K24, N69, L71","L307, S310, Y311, Q312, S313, Q314, L315, I316...","Q3, Y14, D15, K16, K18, P19, V20, S21, F22, H2...","Y311, P367, F368, Q369, R371, I372, C373, M374...","M1, Q3, Y14, K18, P19, V20, S21, F22, H23, K24...","Y311, Q314, L315, C373, M374, A397, D399, I400...","K24, K49, Y50, E66, C67, N68, N69, L71, R169, ...","A397, C370, C373, C401, D399, F368, G364, G392...",C373,"C67, D15, E178, E182, E66, F22, H23, K16, K179...","K24, L71, N69"
3,EGR1_ZNF263,"L304, N308, Y311, Q312, L315, I316, P318, R320...","M74, T76, E78, E82, E103, L104, H105, E107, E1...","S319, R320, M321, R322, Y324, I361, K366, R371...","Q95, M210, P213, L215, P216, E248, T606, L607,...","H334, E335, P337, Y338, S350, R351, S352, D353...","P72, E73, M74, R75, T76, K77, E78, L437, E438,...","M321, E343, D346, Q369, C373, M374, R375, N376","L379, C380, P381, L382, C383, G384, T447, Y490...","P318, S319, R320, M321, K323, Y324, P325, N326...","L28, H67, P72, E73, M74, R75, T76, L198, F199,...","C373, C543, D346, D353, E335, E343, E410, E541...",,"C380, C383, C608, C636, D118, D638, E103, E107...",
4,EGR1_ZNF341,"H25, S26, P27, T28, C192, A193, V194, P195, K3...","S203, L204, G205, P206, P207, N211, P212, G213...","T281, Q282, Q283, P284, S285, L286, P288, T291...","Q3, A4, F6, E7, A8, L9, E10, D48, D49, D51, F5...","S183, A184, S185, Q186, S187, P188, N326, P328...","L9, L65, P66, G132, N133, I134, L135, V136, S1...","L256, S313, I316, P318, M321, N326, P332, P333...","D48, D49, E50, D51, F53, L54, C55, G56, L65, F...","L116, V117, E118, T119, K305, A306, L307, N308...","Q21, L24, D25, I44, L47, D48, D49, E50, D51, L...","A184, A193, A306, A339, A508, C192, C340, C345...","C401, D399, D423, E335, E393, G402, H334, H390...","A144, A358, A360, A4, A554, A672, A8, A852, C3...","D635, F603, H607, H612, H683, I615, K601, K671..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
550,PPARA_THRA,"A97, L98, N99, I100, E101, C102, R103, I104, C...","S40, G41, Y42, I43, P44, S45, Y46, L47, D48, K...","P94, S95, G96, A97, L98, N99, I100, E101, R103...","M39, G41, Y42, I43, P44, S45, Y46, L47, A60, T...","L72, P74, A75, S76, S77, P78, H193, D194, E196..

In [314]:
com.to_csv('./result/com_339_sites.csv')

# Sorting out TF pathogenic variants

## Overlaps with clinvar

In [25]:
import pandas as pd

In [26]:
# '/pool1/tmp/lixuanzhao/Clinvar/clinvar_info.csv'

In [27]:
data = pd.read_table(
        './clinvar/variant_summary.txt.gz'
    )

/tmp/ipykernel_1492626/449638274.py:1: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_table(


In [28]:
len(data)

7859994

In [29]:
data.head()

,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),...,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification
0,15041,Indel,NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTA...,9907,AP5Z1,HGNC:22197,Pathogenic/Likely pathogenic,1,"Dec 17, 2024",397704705,...,TGCTGTAAACTGTAACTGTAAA,-,-,-,-,-,-,SCV001451119|SCV005622007|SCV005909190,-,-
1,15041,Indel,NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTA...,9907,AP5Z1,HGNC:22197,Pathogenic/Likely pathogenic,1,"Dec 17, 2024",397704705,...,TGCTGTAAACTGTAACTGTAAA,-,-,-,-,-,-,SCV001451119|SCV005622007|SCV005909190,-,-
2,15042,Deletion,NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs),9907,AP5Z1,HGNC:22197,Pathogenic,1,"Jun 29, 2010",397704709,...,G,-,-,-,-,-,-,SCV000020156,-,-
3,15042,Deletion,NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs),9907,AP5Z1,HGNC:22197,Pathogenic,1,"Jun 29, 2010",397704709,...,G,-,-,-,-,-,-,SCV000020156,-,-
4,15043,single nucleotide variant,NM_014630.3(ZNF592):c.3136G>A (p.Gly1046Arg),9640,ZNF592,HGNC:28986,Uncertain significance,0,"Jun 29, 2015",150829393,...,A,-,-,-,-,-,-,SCV000020157,-,-


In [30]:
data.columns

Index(['#AlleleID', 'Type', 'Name', 'GeneID', 'GeneSymbol', 'HGNC_ID',
       'ClinicalSignificance', 'ClinSigSimple', 'LastEvaluated', 'RS# (dbSNP)',
       'nsv/esv (dbVar)', 'RCVaccession', 'PhenotypeIDS', 'PhenotypeList',
       'Origin', 'OriginSimple', 'Assembly', 'ChromosomeAccession',
       'Chromosome', 'Start', 'Stop', 'ReferenceAllele', 'AlternateAllele',
       'Cytogenetic', 'ReviewStatus', 'NumberSubmitters', 'Guidelines',
       'TestedInGTR', 'OtherIDs', 'SubmitterCategories', 'VariationID',
       'PositionVCF', 'ReferenceAlleleVCF', 'AlternateAlleleVCF',
       'SomaticClinicalImpact', 'SomaticClinicalImpactLastEvaluated',
       'ReviewStatusClinicalImpact', 'Oncogenicity',
       'OncogenicityLastEvaluated', 'ReviewStatusOncogenicity',
       'SCVsForAggregateGermlineClassification',
       'SCVsForAggregateSomaticClinicalImpact',
       'SCVsForAggregateOncogenicityClassification'],
      dtype='object')

### Extract SNVs that cause protein changes 

In [31]:
protein_changes = data[data['Name'].str.contains(r'\(p\.', na=False)]
protein_changes_snv = protein_changes[protein_changes['Type'] == 'single nucleotide variant']
protein_changes_snv.head()

,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),...,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification
4,15043,single nucleotide variant,NM_014630.3(ZNF592):c.3136G>A (p.Gly1046Arg),9640,ZNF592,HGNC:28986,Uncertain significance,0,"Jun 29, 2015",150829393,...,A,-,-,-,-,-,-,SCV000020157,-,-
5,15043,single nucleotide variant,NM_014630.3(ZNF592):c.3136G>A (p.Gly1046Arg),9640,ZNF592,HGNC:28986,Uncertain significance,0,"Jun 29, 2015",150829393,...,A,-,-,-,-,-,-,SCV000020157,-,-
6,15044,single nucleotide variant,NM_017547.4(FOXRED1):c.694C>T (p.Gln232Ter),55572,FOXRED1,HGNC:26927,Pathogenic,1,"Aug 17, 2025",267606829,...,T,-,-,-,-,-,-,SCV000680696|SCV001363290|SCV002793147|SCV0029...,-,-
7,15044,single nucleotide variant,NM_017547.4(FOXRED1):c.694C>T (p.Gln232Ter),55572,FOXRED1,HGNC:26927,Pathogenic,1,"Aug 17, 2025",267606829,...,T,-,-,-,-,-,-,SCV000680696|SCV001363290|SCV002793147|SCV0029...,-,-
8,15045,single nucleotide variant,NM_017547.4(FOXRED1):c.1289A>G (p.Asn430Ser),55572,FOXRED1,HGNC:26927,Likely pathogenic,1,"Jun 06, 2024",267606830,...,G,-,-,-,-,-,-,SCV005680614,-,-


In [32]:
len(protein_changes_snv)

5983599

### Extract SNVs of corresponding proteins

In [33]:
import pandas as pd

In [34]:
tf = pd.read_csv('./output/old_result/all_protein_sequences_and_lengths.csv')
tf

,protein_name,sequence,length
0,AHR,MNSSSANITYASRKRRKPVQKTVKPIPAEGIKSNPSKRHRDRLNTE...,848
1,EBF1,MFGIQESIQRSGSSMKEEPLGSGMNAVRTWMQGAGVLDANTAAQSG...,591
2,HOXA10,MSARKGYLLPSPNYPTTMSCSESPAANSFLVDSLISSGRGEAGGGG...,410
3,ZBTB33,MESRKLISATDIQYSGSLLNSLNEQRGHGLFCDVTVIVEDRKFRAH...,672
4,PRDM4,MHHRMNEMNLSPVGMEQLTSSSVSNALPVSGSHLGLAASPTHSAIP...,801
...,...,...,...
390,TEAD2,MGEPRAGAALDDGSGWTGSEEGSEEGTGGSEGAGGDGGPDAEGVWS...,447
391,ZNF554,MVTCAHLGRRARLPAAQPSACPGTCFSQEERMAAGYLPRWSQELVT...,538
392,ZNF324,MAFEDVAVYFSQEEWGLLDTAQRALYRRVMLDNFALVASLGLSTSR...,553
393,SREBF1,MDEPPFSEAALEQALGEPCDLDAALLTDIEDMLQLINNQDSDFPGL...,1147


In [35]:
set(tf['protein_name']) - set(protein_changes_snv['GeneSymbol'])

{'ARNTL'}

In [36]:
set(['BMAL1']) & set(protein_changes_snv['GeneSymbol'])

{'BMAL1'}

In [37]:
# 替换ARNTL为BMAL1
tf.replace('ARNTL', 'BMAL1', inplace=True)

In [38]:
set(tf['protein_name']) - set(protein_changes_snv['GeneSymbol'])

set()

In [39]:
tf_pc_snv = protein_changes_snv[protein_changes_snv['GeneSymbol'].isin(tf['protein_name'])]
tf_pc_snv = tf_pc_snv[tf_pc_snv['Assembly'] == 'GRCh38']
tf_pc_snv

,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),...,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification
1352,15754,single nucleotide variant,NM_001109809.2(ZFP57):c.783C>A (p.Cys261Ter),346171,ZFP57,HGNC:18791,Pathogenic,1,"Aug 01, 2008",61730328,...,T,-,-,-,-,-,-,SCV000020901,-,-
1358,15757,single nucleotide variant,NM_001109809.5(ZFP57):c.1372C>G (p.His458Asp),346171,ZFP57,HGNC:18791,Pathogenic,1,"Aug 01, 2008",79020217,...,C,-,-,-,-,-,-,SCV000020904,-,-
1360,15758,single nucleotide variant,NM_001109809.5(ZFP57):c.743G>A (p.Arg248His),346171,ZFP57,HGNC:18791,Pathogenic,1,"Aug 01, 2008",77625743,...,T,-,-,-,-,-,-,SCV000020905,-,-
1362,15759,single nucleotide variant,NM_001109809.5(ZFP57):c.829C>A (p.His277Asn),346171,ZFP57,HGNC:18791,Pathogenic,1,"Aug 01, 2008",78378398,...,T,-,-,-,-,-,-,SCV000020906,-,-
3070,16685,single nucleotide variant,NM_018105.3(THAP1):c.241T>C (p.Phe81Leu),55145,THAP1,HGNC:20856,Uncertain significance,1,"Oct 17, 2024",118204013,...,G,-,-,-,-,-,-,SCV001233480,-,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7859259,4391604,single nucleotide variant,NM_000546.6(TP53):c.391A>C (p.Asn131His),7157,TP53,HGNC:11998,Likely pathogenic,1,"Jul 30, 2025",-1,...,G,-,-,-,-,-,-,SCV006554497,-,-
7859281,4391615,single nucleotide variant,NM_006060.6(IKZF1):c.266G>A (p.Gly89Glu),10320,IKZF1,HGNC:13176,Likely pathogenic,1,"Apr 08, 2025",-1,...,A,-,-,-,-,-,-,SCV006554580,-,-
7859317,4391633,single nucleotide variant,NM_052854.4(CREB3L1):c.1042C>T (p.Gln348Ter),90993,CREB3L1,HGNC:18856,Likely pathogenic,1,-,-1,...,T,-,-,-,-,-,-,SCV006333454,-,-
7859787,4392078,single nucleotide variant,NM_006186.4(NR4A2):c.235T>C (p.Tyr79His),4929,NR4A2,HGNC:7981,Uncertain significance,0,"May 27, 2025",-1,...,G,-,-,-,-,-,-,SCV006555130,-,-


In [40]:
tf_pc_snv.head().to_csv('./clinvar/var_sample.csv')

In [41]:
tf_pc_snv.columns

Index(['#AlleleID', 'Type', 'Name', 'GeneID', 'GeneSymbol', 'HGNC_ID',
       'ClinicalSignificance', 'ClinSigSimple', 'LastEvaluated', 'RS# (dbSNP)',
       'nsv/esv (dbVar)', 'RCVaccession', 'PhenotypeIDS', 'PhenotypeList',
       'Origin', 'OriginSimple', 'Assembly', 'ChromosomeAccession',
       'Chromosome', 'Start', 'Stop', 'ReferenceAllele', 'AlternateAllele',
       'Cytogenetic', 'ReviewStatus', 'NumberSubmitters', 'Guidelines',
       'TestedInGTR', 'OtherIDs', 'SubmitterCategories', 'VariationID',
       'PositionVCF', 'ReferenceAlleleVCF', 'AlternateAlleleVCF',
       'SomaticClinicalImpact', 'SomaticClinicalImpactLastEvaluated',
       'ReviewStatusClinicalImpact', 'Oncogenicity',
       'OncogenicityLastEvaluated', 'ReviewStatusOncogenicity',
       'SCVsForAggregateGermlineClassification',
       'SCVsForAggregateSomaticClinicalImpact',
       'SCVsForAggregateOncogenicityClassification'],
      dtype='object')

In [42]:
tf_pc_snv['GeneSymbol'].value_counts()

GeneSymbol
NPAT      2207
TP53      1968
EPAS1     1605
SMAD4     1546
WT1       1405
          ... 
NFYB        10
MEF2B        7
ZBTB7B       6
BATF         6
ATF7         1
Name: count, Length: 395, dtype: int64

In [43]:
tf_pc_snv['ClinicalSignificance'].value_counts()

ClinicalSignificance
Uncertain significance                                     41150
Likely benign                                              14230
Conflicting classifications of pathogenicity                2141
Benign                                                      1889
Pathogenic                                                  1755
Benign/Likely benign                                        1232
Likely pathogenic                                           1208
Pathogenic/Likely pathogenic                                 393
-                                                             45
not provided                                                  44
risk factor                                                    7
Affects                                                        4
Likely pathogenic; association                                 4
other                                                          4
Uncertain significance; association                            2
Path

In [44]:
tf_pc_snv['PhenotypeList'].value_counts()

PhenotypeList
not specified                                                                                                  17789
not provided                                                                                                   12092
Inborn genetic diseases                                                                                         6758
not provided|Inborn genetic diseases                                                                             753
Polyglandular autoimmune syndrome, type 1                                                                        601
                                                                                                               ...  
not specified|not provided|Neonatal diabetes mellitus with congenital hypothyroidism|Monogenic diabetes            1
not specified|not provided|Neonatal diabetes mellitus with congenital hypothyroidism|GLIS3-related disorder        1
not specified|TWIST1-related craniosynostosis     

In [45]:
tf_pc_snv[['Name','GeneSymbol','ClinicalSignificance','PhenotypeIDS','PhenotypeList']]

,Name,GeneSymbol,ClinicalSignificance,PhenotypeIDS,PhenotypeList
1352,NM_001109809.2(ZFP57):c.783C>A (p.Cys261Ter),ZFP57,Pathogenic,"MONDO:MONDO:0011073,MedGen:C1832386,OMIM:60141...","Diabetes mellitus, transient neonatal, 1"
1358,NM_001109809.5(ZFP57):c.1372C>G (p.His458Asp),ZFP57,Pathogenic,"MONDO:MONDO:0011073,MedGen:C1832386,OMIM:60141...","Diabetes mellitus, transient neonatal, 1"
1360,NM_001109809.5(ZFP57):c.743G>A (p.Arg248His),ZFP57,Pathogenic,"MONDO:MONDO:0011073,MedGen:C1832386,OMIM:60141...","Diabetes mellitus, transient neonatal, 1"
1362,NM_001109809.5(ZFP57):c.829C>A (p.His277Asn),ZFP57,Pathogenic,"MONDO:MONDO:0011073,MedGen:C1832386,OMIM:60141...","Diabetes mellitus, transient neonatal, 1"
3070,NM_018105.3(THAP1):c.241T>C (p.Phe81Leu),THAP1,Uncertain significance,"MONDO:MONDO:0011264,MedGen:C1414216,OMIM:60262...",Torsion dystonia 6
...,...,...,...,...,...
7859259,NM_000546.6(TP53):c.391A>C (p.Asn131His),TP53,Likely pathogenic,"MONDO:MONDO:0011450,MedGen:C2676676,OMIM:60437...","Breast-ovarian cancer, familial, susceptibilit..."
7859281,NM_006060.6(IKZF1):c.266G>A (p.Gly89Glu),IKZF1,Likely pathogenic,"MONDO:MONDO:0014810,MedGen:C4225173,OMIM:61687...",Pancytopenia due to IKZF1 mutations
7859317,NM_052854.4(CREB3L1):c.1042C>T (p.Gln348Ter),CREB3L1,Likely pathogenic,"MONDO:MONDO:0014544,MedGen:C4015610,OMIM:61622...",Osteogenesis imperfecta type 16
7859787,NM_006186.4(NR4A2):c.235T>C (p.Tyr79His),NR4A2,Uncertain significance,"MONDO:MONDO:0859257,MedGen:C5677001,OMIM:61991...",Intellectual developmental disorder with langu...


### Summary variant sites

In [46]:
import pandas as pd
import re

# --- Preparation ---

# 1. Amino acid conversion dictionary (Three-letter to One-letter)
aa_three_to_one = {
    'Ala': 'A', 'Arg': 'R', 'Asn': 'N', 'Asp': 'D', 'Cys': 'C',
    'Glu': 'E', 'Gln': 'Q', 'Gly': 'G', 'His': 'H', 'Ile': 'I',
    'Leu': 'L', 'Lys': 'K', 'Met': 'M', 'Phe': 'F', 'Pro': 'P',
    'Ser': 'S', 'Thr': 'T', 'Trp': 'W', 'Tyr': 'Y', 'Val': 'V',
    'Ter': '*', 'Stop': '*'
}

# --- Step 1: Data Preprocessing ---

def parse_mutation_info(name_string):
    """
    Parses the HGVS nomenclature string and returns a tuple containing 
    (Gene Symbol, "Position-AminoAcid").
    """
    if not isinstance(name_string, str):
        return (None, None)

    # Corrected regular expression to handle hyphens/dashes in gene names
    gene_match = re.search(r'\(([A-Z0-9_.-]+)\)', name_string)
    protein_match = re.search(r'\(p\.([A-Za-z0-9*=_]+)\)', name_string)

    if not gene_match or not protein_match:
        return (None, None)

    gene_symbol = gene_match.group(1)
    p_change = protein_match.group(1)

    # Match the 3-letter amino acid code and the position number
    original_aa_match = re.match(r'([A-Z][a-z]{2})([0-9]+)', p_change)
    if original_aa_match:
        ref_aa_3 = original_aa_match.group(1)
        position = original_aa_match.group(2)
        ref_aa_1 = aa_three_to_one.get(ref_aa_3, '?')
        # Return a tuple for easier splitting later
        return (gene_symbol, f"{position}-{ref_aa_1}")
    
    return (None, None)

# Apply the function and split the result into two columns
# (Assuming 'tf_pc_snv' is your input DataFrame)
parsed_info = tf_pc_snv['Name'].apply(parse_mutation_info).apply(pd.Series)
parsed_info.columns = ['GeneSymbol', 'MutationSiteAndAA']

# Merge parsed results with the original DataFrame (if other columns are needed) or use directly
# Here, we use the parsed new DataFrame directly
temp_df = parsed_info.dropna()


# --- Core Step: Grouping and Multiple Aggregation (groupby.agg) ---

# Define aggregation functions
# 'nunique' is a built-in pandas function used to count the number of unique values
aggregation_functions = {
    'MutationSiteAndAA': [
        ('MutationList', lambda x: ','.join(x.unique())),
        ('MutationCount', 'nunique')
    ]
}

# Perform grouping and aggregation
# as_index=False keeps the grouping key (GeneSymbol) as a column instead of setting it as the index
gene_summary = temp_df.groupby('GeneSymbol', as_index=False).agg(aggregation_functions)

# The DataFrame column names become a MultiIndex; we need to flatten them
gene_summary.columns = ['_'.join(col).strip('_') for col in gene_summary.columns.values]


# --- Output Results ---
print("Summarized mutation list and count by gene:")
print(gene_summary)

Summarized mutation list and count by gene:
    GeneSymbol                     MutationSiteAndAA_MutationList  \
0          AHR  621-Q,570-V,396-R,257-S,466-S,467-I,480-P,487-...   
1         AIRE  257-R,83-K,139-R,228-G,1-M,80-V,401-P,432-A,44...   
2           AR  775-R,719-W,867-V,797-W,591-K,648-S,764-Y,884-...   
3       ARID3A  79-G,447-E,486-A,522-S,551-G,511-T,43-P,197-R,...   
4       ARID3B  238-R,263-V,509-L,350-G,402-R,491-I,166-P,147-...   
..         ...                                                ...   
390     ZNF827  772-P,906-T,770-Q,226-K,591-N,223-L,499-G,821-...   
391      ZNF85  231-K,250-K,318-E,447-I,278-I,329-L,385-L,425-...   
392    ZSCAN16  196-K,102-R,62-L,305-T,326-G,118-D,336-I,159-H...   
393    ZSCAN21  36-K,45-R,184-G,297-R,289-S,168-T,147-K,46-Q,6...   
394    ZSCAN31  53-P,238-R,59-R,138-V,254-L,104-P,278-R,173-Q,...   

     MutationSiteAndAA_MutationCount  
0                                326  
1                                428  
2         

In [152]:
gene_summary

,GeneSymbol,MutationSiteAndAA_MutationList,MutationSiteAndAA_MutationCount
0,AHR,"621-Q,570-V,396-R,257-S,466-S,467-I,480-P,487-...",326
1,AIRE,"257-R,83-K,139-R,228-G,1-M,80-V,401-P,432-A,44...",428
2,AR,"775-R,719-W,867-V,797-W,591-K,648-S,764-Y,884-...",464
3,ARID3A,"79-G,447-E,486-A,522-S,551-G,511-T,43-P,197-R,...",91
4,ARID3B,"238-R,263-V,509-L,350-G,402-R,491-I,166-P,147-...",58
...,...,...,...
390,ZNF827,"772-P,906-T,770-Q,226-K,591-N,223-L,499-G,821-...",109
391,ZNF85,"231-K,250-K,318-E,447-I,278-I,329-L,385-L,425-...",78
392,ZSCAN16,"196-K,102-R,62-L,305-T,326-G,118-D,336-I,159-H...",34
393,ZSCAN21,"36-K,45-R,184-G,297-R,289-S,168-T,147-K,46-Q,6...",65


In [148]:
gene_summary.to_csv('./clinvar/tf_var.csv')

## Organize variant disease information

In [47]:
var_dis = tf_pc_snv[['Name','GeneSymbol','ClinicalSignificance','PhenotypeIDS','PhenotypeList']]
var_dis

,Name,GeneSymbol,ClinicalSignificance,PhenotypeIDS,PhenotypeList
1352,NM_001109809.2(ZFP57):c.783C>A (p.Cys261Ter),ZFP57,Pathogenic,"MONDO:MONDO:0011073,MedGen:C1832386,OMIM:60141...","Diabetes mellitus, transient neonatal, 1"
1358,NM_001109809.5(ZFP57):c.1372C>G (p.His458Asp),ZFP57,Pathogenic,"MONDO:MONDO:0011073,MedGen:C1832386,OMIM:60141...","Diabetes mellitus, transient neonatal, 1"
1360,NM_001109809.5(ZFP57):c.743G>A (p.Arg248His),ZFP57,Pathogenic,"MONDO:MONDO:0011073,MedGen:C1832386,OMIM:60141...","Diabetes mellitus, transient neonatal, 1"
1362,NM_001109809.5(ZFP57):c.829C>A (p.His277Asn),ZFP57,Pathogenic,"MONDO:MONDO:0011073,MedGen:C1832386,OMIM:60141...","Diabetes mellitus, transient neonatal, 1"
3070,NM_018105.3(THAP1):c.241T>C (p.Phe81Leu),THAP1,Uncertain significance,"MONDO:MONDO:0011264,MedGen:C1414216,OMIM:60262...",Torsion dystonia 6
...,...,...,...,...,...
7859259,NM_000546.6(TP53):c.391A>C (p.Asn131His),TP53,Likely pathogenic,"MONDO:MONDO:0011450,MedGen:C2676676,OMIM:60437...","Breast-ovarian cancer, familial, susceptibilit..."
7859281,NM_006060.6(IKZF1):c.266G>A (p.Gly89Glu),IKZF1,Likely pathogenic,"MONDO:MONDO:0014810,MedGen:C4225173,OMIM:61687...",Pancytopenia due to IKZF1 mutations
7859317,NM_052854.4(CREB3L1):c.1042C>T (p.Gln348Ter),CREB3L1,Likely pathogenic,"MONDO:MONDO:0014544,MedGen:C4015610,OMIM:61622...",Osteogenesis imperfecta type 16
7859787,NM_006186.4(NR4A2):c.235T>C (p.Tyr79His),NR4A2,Uncertain significance,"MONDO:MONDO:0859257,MedGen:C5677001,OMIM:61991...",Intellectual developmental disorder with langu...


In [49]:
var_dis.to_csv('./clinvar/tf_var_disease.csv')

In [48]:
var_dis['PhenotypeIDS'].value_counts()

PhenotypeIDS
MedGen:CN169374                                                                                                                                                                                                                                                                                                                                                                                                                  17789
MedGen:C3661900                                                                                                                                                                                                                                                                                                                                                                                                                  11931
MeSH:D030342,MedGen:C0950123                                                                                                                 

### Organize mondo

In [17]:
import pronto
import csv
import time

# --- 0. Define custom mapping rules from ~43 categories to 7 super-categories ---
# This is the core logic of this task
CATEGORY_TO_SUPER_CATEGORY_MAP = {
    # Autoimmune
    'immune system disorder': 'Autoimmune',
    
    # Cancer
    'cancer or benign tumor': 'Cancer',
    
    # Cardiovascular
    'cardiovascular disorder': 'Cardiovascular',
    
    # Metabolic
    'metabolic disease': 'Metabolic',
    'mitochondrial disease': 'Metabolic', # Mitochondrial diseases are fundamentally energy metabolism issues
    
    # Neurological
    'nervous system disorder': 'Neurological',
    'psychiatric disorder': 'Neurological',    # Psychiatric disorders are grouped under Neurological
    'auditory system disorder': 'Neurological', # Auditory system is part of the nervous system
    'disorder of visual system': 'Neurological', # Visual system is part of the nervous system
    'sleep disorder': 'Neurological',          # Sleep disorders are functional issues of the nervous system
    
    # Pulmonary
    'respiratory system disorder': 'Pulmonary',
}


# --- 1. Initialize and Load Ontology ---
print("Starting task: Generating Mondo disease 5-column aggregated classification table...")
print("Loading Mondo ontology file (mondo.obo)...")
start_time = time.time()
try:
    mondo = pronto.Ontology('./clinvar/mondo.obo')
except FileNotFoundError:
    print("Error: 'mondo.obo' file not found in the current directory.")
    exit()
end_time = time.time()
print(f"Mondo ontology loaded! Time elapsed: {end_time - start_time:.2f} seconds.")


# --- 2. Automatically Identify Target Major Categories ---
try:
    root_disease_term = mondo['MONDO:0000001']
    # Find the 'human disease' term among the children of the root
    human_disease_term = next(child for child in root_disease_term.subclasses(distance=1) if child.name == 'human disease')
    # Get the direct subclasses of human disease (the major categories)
    major_categories = list(human_disease_term.subclasses(distance=1, with_self=False))
    print(f"\nSuccessfully identified {len(major_categories)} target major categories.")
except (KeyError, StopIteration):
    print("Error: Unable to find the structure 'MONDO:0000001' -> 'human disease' in the ontology.")
    exit()


# --- 3. Iterate through Major Categories, Find Descendants, and Create 5-Column Mapping ---
print("\nStarting traversal of each major category to aggregate into 7 super-categories...")
all_mappings = []
start_time = time.time()

for i, category_term in enumerate(major_categories):
    print(f"  ({i+1}/{len(major_categories)}) Processing category: '{category_term.name}'...")
    
    # Determine which super-category this major category belongs to; default to 'Other' if not in the map
    super_category_name = CATEGORY_TO_SUPER_CATEGORY_MAP.get(category_term.name, 'Other')
    
    # Get all descendants (including the category itself)
    descendants = category_term.subclasses(with_self=True)
    
    for descendant_term in descendants:
        all_mappings.append([
            descendant_term.name, 
            descendant_term.id, 
            category_term.name, 
            category_term.id,
            super_category_name # Add 5th column: Custom Super Category
        ])

end_time = time.time()
print(f"\nAll mapping relationships generated! Found {len(all_mappings)} records. Time elapsed: {end_time - start_time:.2f} seconds.")


# --- 4. Write All Mappings to CSV File ---
output_filename = 'mondo_aggregated_classification.csv'
print(f"\nWriting results to file: {output_filename}")

try:
    with open(output_filename, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow([
            'Disease_Term', 
            'Disease_Term_ID', 
            'Major_Category', 
            'Major_Category_ID',
            'Custom_Super_Category' # New header
        ])
        writer.writerows(all_mappings)
        
    print("\n🎉 Task successfully completed!")
    print(f"Successfully generated final aggregated classification file '{output_filename}'.")

except IOError:
    print(f"Error: Unable to write to file {output_filename}.")

开始执行任务：生成Mondo疾病五列聚合分类表...
正在加载Mondo本体文件 (mondo.obo)...


/tmp/ipykernel_1492626/2337637919.py:38: UnicodeWarning: unsound encoding, assuming utf-8 (99% confidence)
  mondo = pronto.Ontology('./clinvar/mondo.obo')


Mondo本体加载完毕！用时: 7.09 秒。

成功识别出 43 个目标大类。

开始遍历每个大类并聚合到7个超类中...
  (1/43) 正在处理大类: 'disorder of orbital region'...
  (2/43) 正在处理大类: 'psychiatric disorder'...
  (3/43) 正在处理大类: 'integumentary system disorder'...
  (4/43) 正在处理大类: 'musculoskeletal system disorder'...
  (5/43) 正在处理大类: 'urinary system disorder'...
  (6/43) 正在处理大类: 'syndromic disease'...
  (7/43) 正在处理大类: 'auditory system disorder'...
  (8/43) 正在处理大类: 'breast disorder'...
  (9/43) 正在处理大类: 'hereditary disease'...
  (10/43) 正在处理大类: 'connective tissue disorder'...
  (11/43) 正在处理大类: 'digestive system disorder'...
  (12/43) 正在处理大类: 'cardiovascular disorder'...
  (13/43) 正在处理大类: 'reproductive system disorder'...
  (14/43) 正在处理大类: 'immune system disorder'...
  (15/43) 正在处理大类: 'metabolic disease'...
  (16/43) 正在处理大类: 'nervous system disorder'...
  (17/43) 正在处理大类: 'respiratory system disorder'...
  (18/43) 正在处理大类: 'nutritional disorder'...
  (19/43) 正在处理大类: 'endocrine system disorder'...
  (20/43) 正在处理大类: 'infectious disease'...
  (21/43)

### Organize categories corresponding to mutated diseases

In [258]:
dis_id = pd.read_csv('./clinvar/mondo_aggregated_classification.csv')
dis_id

,Disease_Term,Disease_Term_ID,Major_Category,Major_Category_ID,Custom_Super_Category
0,disorder of orbital region,MONDO:0002022,disorder of orbital region,MONDO:0002022,Other
1,eye adnexa disorder,MONDO:0000462,disorder of orbital region,MONDO:0002022,Other
2,eye disorder,MONDO:0005328,disorder of orbital region,MONDO:0002022,Other
3,orbit neoplasm,MONDO:0024611,disorder of orbital region,MONDO:0002022,Other
4,orbital dermoid cyst,MONDO:0040700,disorder of orbital region,MONDO:0002022,Other
...,...,...,...,...,...
56830,disease related to hematopoietic stem cell tra...,MONDO:0700222,disease related to transplantation,MONDO:0700220,Other
56831,graft versus host disease,MONDO:0013730,disease related to transplantation,MONDO:0700220,Other
56832,engraftment syndrome,MONDO:0043358,disease related to transplantation,MONDO:0700220,Other
56833,acute graft versus host disease,MONDO:0020546,disease related to transplantation,MONDO:0700220,Other


In [259]:
var_dis

,Name,GeneSymbol,ClinicalSignificance,PhenotypeIDS,PhenotypeList,MONDO_ID
1352,NM_001109809.2(ZFP57):c.783C>A (p.Cys261Ter),ZFP57,Pathogenic,"MONDO:MONDO:0011073,MedGen:C1832386,OMIM:60141...","Diabetes mellitus, transient neonatal, 1",MONDO:0011073
1358,NM_001109809.5(ZFP57):c.1372C>G (p.His458Asp),ZFP57,Pathogenic,"MONDO:MONDO:0011073,MedGen:C1832386,OMIM:60141...","Diabetes mellitus, transient neonatal, 1",MONDO:0011073
1360,NM_001109809.5(ZFP57):c.743G>A (p.Arg248His),ZFP57,Pathogenic,"MONDO:MONDO:0011073,MedGen:C1832386,OMIM:60141...","Diabetes mellitus, transient neonatal, 1",MONDO:0011073
1362,NM_001109809.5(ZFP57):c.829C>A (p.His277Asn),ZFP57,Pathogenic,"MONDO:MONDO:0011073,MedGen:C1832386,OMIM:60141...","Diabetes mellitus, transient neonatal, 1",MONDO:0011073
3070,NM_018105.3(THAP1):c.241T>C (p.Phe81Leu),THAP1,Uncertain significance,"MONDO:MONDO:0011264,MedGen:C1414216,OMIM:60262...",Torsion dystonia 6,MONDO:0011264
...,...,...,...,...,...,...
7859259,NM_000546.6(TP53):c.391A>C (p.Asn131His),TP53,Likely pathogenic,"MONDO:MONDO:0011450,MedGen:C2676676,OMIM:60437...","Breast-ovarian cancer, familial, susceptibilit...",MONDO:0011450
7859281,NM_006060.6(IKZF1):c.266G>A (p.Gly89Glu),IKZF1,Likely pathogenic,"MONDO:MONDO:0014810,MedGen:C4225173,OMIM:61687...",Pancytopenia due to IKZF1 mutations,MONDO:0014810
7859317,NM_052854.4(CREB3L1):c.1042C>T (p.Gln348Ter),CREB3L1,Likely pathogenic,"MONDO:MONDO:0014544,MedGen:C4015610,OMIM:61622...",Osteogenesis imperfecta type 16,MONDO:0014544
7859787,NM_006186.4(NR4A2):c.235T>C (p.Tyr79His),NR4A2,Uncertain significance,"MONDO:MONDO:0859257,MedGen:C5677001,OMIM:61991...",Intellectual developmental disorder with langu...,MONDO:0859257


In [260]:
import pandas as pd
import numpy as np # Used for handling potential null values

# --- Assuming you have already loaded these two DataFrames ---
# var_dis = pd.read_csv('path/to/your/variant_data.csv')
# dis_id = pd.read_csv('./mondo_aggregated_classification.csv')


# --- Step 1: Prepare the "Lock" - Create a Unique Disease Classification Lookup Table ---
# Objective: Handle cases in dis_id where one disease ID corresponds to multiple classifications.
# We will merge all classification info for the same ID into one row, separated by semicolons.
print("Step 1: Preparing a unique disease classification lookup table...")

# Define aggregation functions: Deduplicate strings in each group and join them with '; '
agg_funcs = {
    'Disease_Term': 'first', # The name corresponding to the ID is unique, taking the first one is sufficient
    'Major_Category': lambda x: '; '.join(x.unique()),
    'Major_Category_ID': lambda x: '; '.join(x.unique()),
    'Custom_Super_Category': lambda x: '; '.join(x.unique())
}

# Group by Disease_Term_ID and aggregate
dis_lookup = dis_id.groupby('Disease_Term_ID').agg(agg_funcs).reset_index()

print("Lookup table example:")
print(dis_lookup[dis_lookup['Disease_Term'] == 'breast cancer'].to_markdown(index=False))
print("\n")


# --- Step 2 (Corrected Version): Extract and Clean MONDO IDs from var_dis ---
# Objective: Create a new column 'MONDO_ID' in var_dis and fix the double prefix issue.
print("Step 2 (Corrected Version): Extracting and cleaning MONDO IDs from PhenotypeIDS column...")

def extract_and_clean_mondo_id(phenotype_ids_str):
    # Check if input is a valid string
    if not isinstance(phenotype_ids_str, str):
        return np.nan # Return NaN for non-string inputs
    
    # Split the string and look for the ID starting with 'MONDO:'
    for id_part in phenotype_ids_str.split(','):
        if id_part.startswith('MONDO:'):
            # Found the part containing the MONDO ID
            clean_id = id_part.strip()
            
            # --- New Key Correction Step ---
            # If 'MONDO:MONDO:' exists, replace it with a single 'MONDO:'
            if clean_id.startswith('MONDO:MONDO:'):
                clean_id = clean_id.replace('MONDO:MONDO:', 'MONDO:', 1) # Only replace the first occurrence to avoid accidents
            
            return clean_id
    
    return np.nan # Return NaN if not found

# Apply the new, more robust function to the PhenotypeIDS column
var_dis['MONDO_ID'] = var_dis['PhenotypeIDS'].apply(extract_and_clean_mondo_id)

# View the results after cleaning and extraction
print("Example of var_dis table after cleaning and extracting MONDO IDs:")
print(var_dis[['PhenotypeIDS', 'MONDO_ID']].head().to_markdown(index=False))
print("\n")

# --- Step 3: Match and Merge ---
# Objective: Add classification information to the variant data.
print("Step 3: Merging classification info into the variant data table...")

# Use left merge to retain all original variant records
# left_on: Join key for the left table (var_dis)
# right_on: Join key for the right table (dis_lookup)
var_dis_classified = pd.merge(
    var_dis,
    dis_lookup,
    left_on='MONDO_ID',
    right_on='Disease_Term_ID',
    how='left' # 'left' ensures every row in var_dis is retained
)

# Cleanup: Drop temporary 'MONDO_ID' and redundant 'Disease_Term_ID' columns to make the table cleaner
var_dis_classified = var_dis_classified.drop(columns=['MONDO_ID'])

print("🎉 Merge complete! The structure and data examples of the final table are as follows:")

# Show info and first few rows of the final result
print(var_dis_classified.info())
print("\nFinal Result Preview:")
# For better display, we select only a few key columns for preview
display_cols = [
    'Name', 'GeneSymbol', 'ClinicalSignificance', 'PhenotypeList',
    'Disease_Term', 'Major_Category', 'Custom_Super_Category'
]
print(var_dis_classified[display_cols].head().to_markdown(index=False))

步骤1: 正在准备唯一的疾病分类查找表...
查找表示例:
| Disease_Term_ID   | Disease_Term   | Major_Category                          | Major_Category_ID            | Custom_Super_Category   |
|:------------------|:---------------|:----------------------------------------|:-----------------------------|:------------------------|
| MONDO:0007254     | breast cancer  | breast disorder; cancer or benign tumor | MONDO:0002657; MONDO:0045024 | Other; Cancer           |


步骤2 (修正版): 正在从 PhenotypeIDS 列中提取并清理 MONDO ID...
清理和提取 MONDO ID 后的 var_dis 表示例:
| PhenotypeIDS                                                   | MONDO_ID      |
|:---------------------------------------------------------------|:--------------|
| MONDO:MONDO:0011073,MedGen:C1832386,OMIM:601410,Orphanet:99886 | MONDO:0011073 |
| MONDO:MONDO:0011073,MedGen:C1832386,OMIM:601410,Orphanet:99886 | MONDO:0011073 |
| MONDO:MONDO:0011073,MedGen:C1832386,OMIM:601410,Orphanet:99886 | MONDO:0011073 |
| MONDO:MONDO:0011073,MedGen:C1832386,OMIM:601410,Orphanet:9

/tmp/ipykernel_1492626/2272048425.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  var_dis['MONDO_ID'] = var_dis['PhenotypeIDS'].apply(extract_and_clean_mondo_id)


In [261]:
var_dis_classified

,Name,GeneSymbol,ClinicalSignificance,PhenotypeIDS,PhenotypeList,Disease_Term_ID,Disease_Term,Major_Category,Major_Category_ID,Custom_Super_Category
0,NM_001109809.2(ZFP57):c.783C>A (p.Cys261Ter),ZFP57,Pathogenic,"MONDO:MONDO:0011073,MedGen:C1832386,OMIM:60141...","Diabetes mellitus, transient neonatal, 1",MONDO:0011073,"diabetes mellitus, transient neonatal, 1",hereditary disease; digestive system disorder;...,MONDO:0003847; MONDO:0004335; MONDO:0005066; M...,Other; Metabolic
1,NM_001109809.5(ZFP57):c.1372C>G (p.His458Asp),ZFP57,Pathogenic,"MONDO:MONDO:0011073,MedGen:C1832386,OMIM:60141...","Diabetes mellitus, transient neonatal, 1",MONDO:0011073,"diabetes mellitus, transient neonatal, 1",hereditary disease; digestive system disorder;...,MONDO:0003847; MONDO:0004335; MONDO:0005066; M...,Other; Metabolic
2,NM_001109809.5(ZFP57):c.743G>A (p.Arg248His),ZFP57,Pathogenic,"MONDO:MONDO:0011073,MedGen:C1832386,OMIM:60141...","Diabetes mellitus, transient neonatal, 1",MONDO:0011073,"diabetes mellitus, transient neonatal, 1",hereditary disease; digestive system disorder;...,MONDO:0003847; MONDO:0004335; MONDO:0005066; M...,Other; Metabolic
3,NM_001109809.5(ZFP57):c.829C>A (p.His277Asn),ZFP57,Pathogenic,"MONDO:MONDO:0011073,MedGen:C1832386,OMIM:60141...","Diabetes mellitus, transient neonatal, 1",MONDO:0011073,"diabetes mellitus, transient neonatal, 1",hereditary disease; digestive system disorder;...,MONDO:0003847; MONDO:0004335; MONDO:0005066; M...,Other; Metabolic
4,NM_018105.3(THAP1):c.241T>C (p.Phe81Leu),THAP1,Uncertain significance,"MONDO:MONDO:0011264,MedGen:C1414216,OMIM:60262...",Torsion dystonia 6,MONDO:0011264,torsion dystonia 6,hereditary disease; nervous system disorder,MONDO:0003847; MONDO:0005071,Other; Neurological
...,...,...,...,...,...,...,...,...,...,...
64112,NM_000546.6(TP53):c.391A>C (p.Asn131His),TP53,Likely pathogenic,"MONDO:MONDO:0011450,MedGen:C2676676,OMIM:60437...","Breast-ovarian cancer, familial, susceptibilit...",MONDO:0011450,"breast-ovarian cancer, familial, susceptibilit...",syndromic disease; hereditary disease; cancer ...,MONDO:0002254; MONDO:0003847; MONDO:0045024,Other; Cancer
64113,NM_006060.6(IKZF1):c.266G>A (p.Gly89Glu),IKZF1,Likely pathogenic,"MONDO:MONDO:0014810,MedGen:C4225173,OMIM:61687...",Pancytopenia due to IKZF1 mutations,MONDO:0014810,pancytopenia due to IKZF1 mutations,syndromic disease; hereditary disease; immune ...,MONDO:0002254; MONDO:0003847; MONDO:0005046; M...,Other; Autoimmune; Cancer
64114,NM_052854.4(CREB3L1):c.1042C>T (p.Gln348Ter),CREB3L1,Likely pathogenic,"MONDO:MONDO:0014544,MedGen:C4015610,OMIM:61622...",Osteogenesis imperfecta type 16,MONDO:0014544,osteogenesis imperfecta type 16,musculoskeletal system disorder; hereditary di...,MONDO:0002081; MONDO:0003847; MONDO:0021147,Other
64115,NM_006186.4(NR4A2):c.235T>C (p.Tyr79His),NR4A2,Uncertain significance,"MONDO:MONDO:0859257,MedGen:C5677001,OMIM:61991...",Intellectual developmental disorder with langu...,MONDO:0859257,intellectual developmental disorder with langu...,hereditary disease; metabolic disease; nervous...,MONDO:0003847; MONDO:0005066; MONDO:0005071,Other; Metabolic; Neurological


In [262]:
var_dis_classified['Custom_Super_Category'].value_counts()

Custom_Super_Category
Other; Cancer                           4336
Other                                   4317
Other; Autoimmune                       3952
Other; Neurological                     2124
Other; Cardiovascular                   1718
Other; Autoimmune; Cancer               1453
Other; Neurological; Cancer              802
Other; Cardiovascular; Autoimmune        727
Other; Metabolic                         525
Neurological; Other                      223
Neurological                             127
Cancer                                    89
Other; Metabolic; Neurological            52
Other; Cardiovascular; Cancer             19
Other; Pulmonary                           9
Autoimmune; Other; Cancer                  6
Cardiovascular                             5
Neurological; Cancer                       5
Pulmonary; Cancer                          4
Cardiovascular; Other                      4
Cardiovascular; Neurological; Cancer       2
Other; Pulmonary; Cancer         

In [263]:
var_dis_classified.to_csv('./clinvar/tf_var_disease_mondo.csv')

# Correspond to the mutation of the interaction interface

## Interaction

In [2]:
import pandas as pd

In [3]:
com_site = pd.read_csv('./result/com_339_sites.csv', index_col=0)[['TF_Pair', 'TF1_Union',	'TF1_Intersection',	'TF2_Union', 'TF2_Intersection']]
com_site

,TF_Pair,TF1_Union,TF1_Intersection,TF2_Union,TF2_Intersection
0,JDP2_JUND,"C139, D144, E103, E108, E124, E152, G14, H135,...","C139, E103, E108, E124, H135, I118, I129, I140...","A137, A143, A26, A311, A314, C336, D146, E133,...","A311, A314, C336, E297, G335, H331, I293, K299..."
1,EGR1_TBX21,"A102, A397, A462, C373, D254, D399, E103, E219...","C373, E393, F257, F396, H390, I252, I372, I387...","A140, A175, A230, A282, A304, A308, A321, A435...","A321, C284, D188, E281, F186, F324, G216, H271..."
2,EGR1_THAP1,"A397, C370, C373, C401, D399, F368, G364, G392...",C373,"C67, D15, E178, E182, E66, F22, H23, K16, K179...","K24, L71, N69"
3,EGR1_ZNF263,"C373, C543, D346, D353, E335, E343, E410, E541...",NaN,"C380, C383, C608, C636, D118, D638, E103, E107...",NaN
4,EGR1_ZNF341,"A184, A193, A306, A339, A508, C192, C340, C345...","C401, D399, D423, E335, E393, G402, H334, H390...","A144, A358, A360, A4, A554, A672, A8, A852, C3...","D635, F603, H607, H612, H683, I615, K601, K671..."
...,...,...,...,...,...
550,PPARA_THRA,"A109, A170, A186, A201, A431, A75, A97, C102, ...","A431, D387, D432, E402, F421, F423, H406, I420...","A117, A121, A133, A344, A379, A60, C380, C67, ...","A344, A379, D328, D373, E148, E339, E343, E350..."
552,RARA_THRA,"A389, C174, C336, C444, D338, D349, D383, E106...","A389, C336, D338, D349, D383, E176, E353, E357...","A117, A121, A379, C244, C67, D123, D166, D206,...","D328, D373, E339, E343, E350, H362, K366, K370..."
553,RORA_THRA,"A142, A271, A431, A479, A490, A493, C257, C472...","A479, C257, C472, F259, I471, Q458, S475, T468...","A298, A308, A379, A382, C244, C380, C392, D206...","A298, C380, D328, D373, E301, H362, K147, K366..."
554,RXRA_THRA,"A398, A402, A416, A424, D10, D140, D173, D379,...","A398, A402, A416, A424, D10, E228, E352, E390,...","A379, C244, C380, D123, D297, D328, D373, D48,...","A379, C244, D328, E339, E343, E350, E403, F239..."


In [4]:
import numpy as np

In [5]:
import numpy as np
import pandas as pd

# --- Step 1: One-step conversion from wide table to tidy long format (TF, Site) ---

# .reset_index() ensures 'TF_Pair' is a column, not an index
# .melt() converts the wide format to long format
# .dropna() immediately drops rows with no site information
# .assign() is a powerful tool for method chaining, creating multiple new columns simultaneously
tidy_df = com_site.reset_index().melt(
    id_vars=['TF_Pair'],
    value_vars=['TF1_Intersection', 'TF2_Intersection'],
    var_name='Source_Column',
    value_name='Site_String'
).dropna(subset=['Site_String']).assign(
    # Determine if it is TF1 or TF2 based on the source column name, and extract the corresponding TF name from TF_Pair
    TF = lambda dfx: np.where(
        dfx['Source_Column'].str.contains('TF1'), 
        dfx['TF_Pair'].str.split('_').str[0], 
        dfx['TF_Pair'].str.split('_').str[1]
    ),
    # Split the site string into a list by comma and optional whitespace
    Site = lambda dfx: dfx['Site_String'].str.split(',\s*')
).explode('Site')[['TF', 'Site']].drop_duplicates()


# --- Step 2: Complete final aggregation in a single line ---

# .groupby('TF') groups by Transcription Factor
# .agg() applies a lambda function: deduplicate (unique), sort (sorted), and then join the sites for each group
final_summary_df = tidy_df.groupby('TF')['Site'].agg(
    lambda sites: ', '.join(sorted(sites.unique()))
).reset_index(name='All_Sites')


# --- Display final results ---
print("🎉 Task completed! Final aggregated results below:")
print(final_summary_df.head())

🎉 任务完成！最终聚合结果如下:
      TF                                          All_Sites
0   ATF1  E242, E249, E258, E39, I256, K234, K263, K269,...
1   ATF2  A366, A384, A414, A469, A473, C418, D40, D417,...
2   ATF3  A100, C103, C153, D165, E117, E122, E138, E166...
3   ATF7  A364, A394, A446, A450, C398, E361, E377, E384...
4  BACH1  A628, A629, A643, C581, C621, C646, D132, D507...


<>:20: SyntaxWarning: invalid escape sequence '\s'
<>:20: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_2040670/165760455.py:20: SyntaxWarning: invalid escape sequence '\s'
  Site = lambda dfx: dfx['Site_String'].str.split(',\s*')


In [327]:
final_summary_df.to_csv('./test.csv')

## Variants

In [6]:
tf_var = pd.read_csv('./clinvar/tf_var_disease_mondo.csv',index_col=0)
tf_var

,Name,GeneSymbol,ClinicalSignificance,PhenotypeIDS,PhenotypeList,Disease_Term_ID,Disease_Term,Major_Category,Major_Category_ID,Custom_Super_Category
0,NM_001109809.2(ZFP57):c.783C>A (p.Cys261Ter),ZFP57,Pathogenic,"MONDO:MONDO:0011073,MedGen:C1832386,OMIM:60141...","Diabetes mellitus, transient neonatal, 1",MONDO:0011073,"diabetes mellitus, transient neonatal, 1",hereditary disease; digestive system disorder;...,MONDO:0003847; MONDO:0004335; MONDO:0005066; M...,Other; Metabolic
1,NM_001109809.5(ZFP57):c.1372C>G (p.His458Asp),ZFP57,Pathogenic,"MONDO:MONDO:0011073,MedGen:C1832386,OMIM:60141...","Diabetes mellitus, transient neonatal, 1",MONDO:0011073,"diabetes mellitus, transient neonatal, 1",hereditary disease; digestive system disorder;...,MONDO:0003847; MONDO:0004335; MONDO:0005066; M...,Other; Metabolic
2,NM_001109809.5(ZFP57):c.743G>A (p.Arg248His),ZFP57,Pathogenic,"MONDO:MONDO:0011073,MedGen:C1832386,OMIM:60141...","Diabetes mellitus, transient neonatal, 1",MONDO:0011073,"diabetes mellitus, transient neonatal, 1",hereditary disease; digestive system disorder;...,MONDO:0003847; MONDO:0004335; MONDO:0005066; M...,Other; Metabolic
3,NM_001109809.5(ZFP57):c.829C>A (p.His277Asn),ZFP57,Pathogenic,"MONDO:MONDO:0011073,MedGen:C1832386,OMIM:60141...","Diabetes mellitus, transient neonatal, 1",MONDO:0011073,"diabetes mellitus, transient neonatal, 1",hereditary disease; digestive system disorder;...,MONDO:0003847; MONDO:0004335; MONDO:0005066; M...,Other; Metabolic
4,NM_018105.3(THAP1):c.241T>C (p.Phe81Leu),THAP1,Uncertain significance,"MONDO:MONDO:0011264,MedGen:C1414216,OMIM:60262...",Torsion dystonia 6,MONDO:0011264,torsion dystonia 6,hereditary disease; nervous system disorder,MONDO:0003847; MONDO:0005071,Other; Neurological
...,...,...,...,...,...,...,...,...,...,...
64112,NM_000546.6(TP53):c.391A>C (p.Asn131His),TP53,Likely pathogenic,"MONDO:MONDO:0011450,MedGen:C2676676,OMIM:60437...","Breast-ovarian cancer, familial, susceptibilit...",MONDO:0011450,"breast-ovarian cancer, familial, susceptibilit...",syndromic disease; hereditary disease; cancer ...,MONDO:0002254; MONDO:0003847; MONDO:0045024,Other; Cancer
64113,NM_006060.6(IKZF1):c.266G>A (p.Gly89Glu),IKZF1,Likely pathogenic,"MONDO:MONDO:0014810,MedGen:C4225173,OMIM:61687...",Pancytopenia due to IKZF1 mutations,MONDO:0014810,pancytopenia due to IKZF1 mutations,syndromic disease; hereditary disease; immune ...,MONDO:0002254; MONDO:0003847; MONDO:0005046; M...,Other; Autoimmune; Cancer
64114,NM_052854.4(CREB3L1):c.1042C>T (p.Gln348Ter),CREB3L1,Likely pathogenic,"MONDO:MONDO:0014544,MedGen:C4015610,OMIM:61622...",Osteogenesis imperfecta type 16,MONDO:0014544,osteogenesis imperfecta type 16,musculoskeletal system disorder; hereditary di...,MONDO:0002081; MONDO:0003847; MONDO:0021147,Other
64115,NM_006186.4(NR4A2):c.235T>C (p.Tyr79His),NR4A2,Uncertain significance,"MONDO:MONDO:0859257,MedGen:C5677001,OMIM:61991...",Intellectual developmental disorder with langu...,MONDO:0859257,intellectual developmental disorder with langu...,hereditary disease; metabolic disease; nervous...,MONDO:0003847; MONDO:0005066; MONDO:0005071,Other; Metabolic; Neurological


In [7]:
tf_var['Custom_Super_Category'].value_counts()

Custom_Super_Category
Other; Cancer                           4336
Other                                   4317
Other; Autoimmune                       3952
Other; Neurological                     2124
Other; Cardiovascular                   1718
Other; Autoimmune; Cancer               1453
Other; Neurological; Cancer              802
Other; Cardiovascular; Autoimmune        727
Other; Metabolic                         525
Neurological; Other                      223
Neurological                             127
Cancer                                    89
Other; Metabolic; Neurological            52
Other; Cardiovascular; Cancer             19
Other; Pulmonary                           9
Autoimmune; Other; Cancer                  6
Cardiovascular                             5
Neurological; Cancer                       5
Pulmonary; Cancer                          4
Cardiovascular; Other                      4
Cardiovascular; Neurological; Cancer       2
Other; Pulmonary; Cancer         

In [8]:
import pandas as pd
import re  # Import regular expression module
import numpy as np  # Used for handling potential null/missing values

# --- Assuming 'tf_var' is the DataFrame you have already loaded ---
# tf_var = pd.read_csv('./clinvar/tf_var_disease_mondo.csv', index_col=0)


# --- Step 1: Define a dictionary mapping three-letter amino acid codes to one-letter codes ---
AA_MAP = {
    'Ala': 'A', 'Arg': 'R', 'Asn': 'N', 'Asp': 'D', 'Cys': 'C',
    'Glu': 'E', 'Gln': 'Q', 'Gly': 'G', 'His': 'H', 'Ile': 'I',
    'Leu': 'L', 'Lys': 'K', 'Met': 'M', 'Phe': 'F', 'Pro': 'P',
    'Ser': 'S', 'Thr': 'T', 'Trp': 'W', 'Tyr': 'Y', 'Val': 'V',
    'Ter': '*'  # 'Ter' stands for Termination (stop codon), usually represented by an asterisk
}


# --- Step 2: Write a function to extract and convert protein site information ---
def extract_protein_site(name_string):
    """
    Extracts simplified protein site information (e.g., 'C261') from HGVS nomenclature strings.
    """
    # First, check if input is a valid string to prevent issues with null values (NaN)
    if not isinstance(name_string, str):
        return np.nan

    # Regex explanation:
    # p\.         - Matches the literal "p."
    # ([A-Z][a-z]{2}) - 1st Capture Group: Matches one uppercase followed by two lowercase letters (three-letter code)
    # (\d+)       - 2nd Capture Group: Matches one or more digits (position number)
    pattern = re.compile(r'p\.([A-Z][a-z]{2})(\d+)')
    
    match = pattern.search(name_string)
    
    if match:
        three_letter_code = match.group(1) # Get content of group 1 (e.g., 'Cys')
        position = match.group(2)          # Get content of group 2 (e.g., '261')
        
        # Look up the corresponding one-letter code in the map
        one_letter_code = AA_MAP.get(three_letter_code)
        
        if one_letter_code:
            # If code found, format the final string
            return f"{one_letter_code}{position}"
            
    # Return NaN if no match found or three-letter code not in dictionary
    return np.nan


# --- Step 3: Apply function to 'Name' column to create new 'site' column ---
print("Extracting protein site info from 'Name' column and creating 'site' column...")

tf_var['site'] = tf_var['Name'].apply(extract_protein_site)

print("Extraction complete!")


# --- Step 4: Display results for verification ---
print("\nPreview of DataFrame with new 'site' column:")

# Select only key columns for better preview visualization
display_cols = ['Name', 'GeneSymbol', 'site']
print(tf_var[display_cols].head())

# Check for rows where extraction failed (site column is null)
print(f"\nNumber of rows where extraction failed (result is null): {tf_var['site'].isnull().sum()}")

正在从 'Name' 列提取蛋白质位点信息并创建 'site' 列...
提取完成！

带有新 'site' 列的 DataFrame 预览:
                                            Name GeneSymbol  site
0   NM_001109809.2(ZFP57):c.783C>A (p.Cys261Ter)      ZFP57  C261
1  NM_001109809.5(ZFP57):c.1372C>G (p.His458Asp)      ZFP57  H458
2   NM_001109809.5(ZFP57):c.743G>A (p.Arg248His)      ZFP57  R248
3   NM_001109809.5(ZFP57):c.829C>A (p.His277Asn)      ZFP57  H277
4       NM_018105.3(THAP1):c.241T>C (p.Phe81Leu)      THAP1   F81

提取失败 (结果为空) 的行数: 0


In [9]:
tf_var

,Name,GeneSymbol,ClinicalSignificance,PhenotypeIDS,PhenotypeList,Disease_Term_ID,Disease_Term,Major_Category,Major_Category_ID,Custom_Super_Category,site
0,NM_001109809.2(ZFP57):c.783C>A (p.Cys261Ter),ZFP57,Pathogenic,"MONDO:MONDO:0011073,MedGen:C1832386,OMIM:60141...","Diabetes mellitus, transient neonatal, 1",MONDO:0011073,"diabetes mellitus, transient neonatal, 1",hereditary disease; digestive system disorder;...,MONDO:0003847; MONDO:0004335; MONDO:0005066; M...,Other; Metabolic,C261
1,NM_001109809.5(ZFP57):c.1372C>G (p.His458Asp),ZFP57,Pathogenic,"MONDO:MONDO:0011073,MedGen:C1832386,OMIM:60141...","Diabetes mellitus, transient neonatal, 1",MONDO:0011073,"diabetes mellitus, transient neonatal, 1",hereditary disease; digestive system disorder;...,MONDO:0003847; MONDO:0004335; MONDO:0005066; M...,Other; Metabolic,H458
2,NM_001109809.5(ZFP57):c.743G>A (p.Arg248His),ZFP57,Pathogenic,"MONDO:MONDO:0011073,MedGen:C1832386,OMIM:60141...","Diabetes mellitus, transient neonatal, 1",MONDO:0011073,"diabetes mellitus, transient neonatal, 1",hereditary disease; digestive system disorder;...,MONDO:0003847; MONDO:0004335; MONDO:0005066; M...,Other; Metabolic,R248
3,NM_001109809.5(ZFP57):c.829C>A (p.His277Asn),ZFP57,Pathogenic,"MONDO:MONDO:0011073,MedGen:C1832386,OMIM:60141...","Diabetes mellitus, transient neonatal, 1",MONDO:0011073,"diabetes mellitus, transient neonatal, 1",hereditary disease; digestive system disorder;...,MONDO:0003847; MONDO:0004335; MONDO:0005066; M...,Other; Metabolic,H277
4,NM_018105.3(THAP1):c.241T>C (p.Phe81Leu),THAP1,Uncertain significance,"MONDO:MONDO:0011264,MedGen:C1414216,OMIM:60262...",Torsion dystonia 6,MONDO:0011264,torsion dystonia 6,hereditary disease; nervous system disorder,MONDO:0003847; MONDO:0005071,Other; Neurological,F81
...,...,...,...,...,...,...,...,...,...,...,...
64112,NM_000546.6(TP53):c.391A>C (p.Asn131His),TP53,Likely pathogenic,"MONDO:MONDO:0011450,MedGen:C2676676,OMIM:60437...","Breast-ovarian cancer, familial, susceptibilit...",MONDO:0011450,"breast-ovarian cancer, familial, susceptibilit...",syndromic disease; hereditary disease; cancer ...,MONDO:0002254; MONDO:0003847; MONDO:0045024,Other; Cancer,N131
64113,NM_006060.6(IKZF1):c.266G>A (p.Gly89Glu),IKZF1,Likely pathogenic,"MONDO:MONDO:0014810,MedGen:C4225173,OMIM:61687...",Pancytopenia due to IKZF1 mutations,MONDO:0014810,pancytopenia due to IKZF1 mutations,syndromic disease; hereditary disease; immune ...,MONDO:0002254; MONDO:0003847; MONDO:0005046; M...,Other; Autoimmune; Cancer,G89
64114,NM_052854.4(CREB3L1):c.1042C>T (p.Gln348Ter),CREB3L1,Likely pathogenic,"MONDO:MONDO:0014544,MedGen:C4015610,OMIM:61622...",Osteogenesis imperfecta type 16,MONDO:0014544,osteogenesis imperfecta type 16,musculoskeletal system disorder; hereditary di...,MONDO:0002081; MONDO:0003847; MONDO:0021147,Other,Q348
64115,NM_006186.4(NR4A2):c.235T>C (p.Tyr79His),NR4A2,Uncertain significance,"MONDO:MONDO:0859257,MedGen:C5677001,OMIM:61991...",Intellectual developmental disorder with langu...,MONDO:0859257,intellectual developmental disorder with langu...,hereditary disease; metabolic disease; nervous...,MONDO:0003847; MONDO:0005066; MONDO:0005071,Other; Metabolic; Neurological,Y79


In [332]:
pd.DataFrame(tf_var['GeneSymbol'].value_counts()).to_csv('./test.csv')

## Integrate

In [333]:
import pandas as pd
import numpy as np

# --- Assuming final_summary_df and tf_var are DataFrames you have already loaded ---
# final_summary_df = ...
# tf_var = ...
# And tf_var already has a column named 'site'

# --- Step 1: Create an efficient lookup dictionary (Same as before) ---
print("Step 1: Creating an efficient lookup dictionary from tf_var...")

tf_var['Major_Category'] = tf_var['Major_Category'].astype(str).fillna('')
tf_var['Custom_Super_Category'] = tf_var['Custom_Super_Category'].astype(str).fillna('')
lookup_dict = {}

for _, row in tf_var.dropna(subset=['GeneSymbol', 'site']).iterrows():
    key = (row['GeneSymbol'], row['site'])
    major_cats = {cat.strip() for cat in row['Major_Category'].split(';') if cat.strip()}
    super_cats = {cat.strip() for cat in row['Custom_Super_Category'].split(';') if cat.strip()}
    
    if key in lookup_dict:
        lookup_dict[key]['major'].update(major_cats)
        lookup_dict[key]['super'].update(super_cats)
    else:
        lookup_dict[key] = {'major': major_cats, 'super': super_cats}

print(f"Lookup dictionary created, containing {len(lookup_dict)} unique (Gene, Site) entries.\n")


# --- Step 2 (Enhanced): Define function to collect classifications and overlapping sites simultaneously ---
print("Step 2 (Enhanced): Iterating through summary table to collect classifications and overlapping sites...")

def find_matching_info(row, lookup):
    """
    For each row in the summary table, find corresponding classifications for all its sites
    and identify the specific sites that overlap.
    """
    tf_name = row['TF']
    sites_string = row['All_Sites']
    
    if not isinstance(sites_string, str):
        # Return three empty strings, corresponding to the three new columns
        return pd.Series(['', '', '']) 
        
    sites_list = {site.strip() for site in sites_string.split(',')}
    
    # Initialize three sets
    found_major_categories = set()
    found_super_categories = set()
    overlapping_sites = set() # <--- New set to store overlapping sites
    
    # Iterate through all sites for this TF
    for site in sites_list:
        match = lookup.get((tf_name, site))
        if match:
            # If a match is found, update classification sets
            found_major_categories.update(match['major'])
            found_super_categories.update(match['super'])
            # And add this specific site to the overlapping sites set
            overlapping_sites.add(site) # <--- New operation
            
    # Convert final sets to sorted, comma-separated strings
    major_str = ', '.join(sorted(list(found_major_categories)))
    super_str = ', '.join(sorted(list(found_super_categories)))
    overlap_str = ', '.join(sorted(list(overlapping_sites))) # <--- New string conversion
    
    # Return a Series containing three elements
    return pd.Series([major_str, super_str, overlap_str])


# --- Step 3: Apply the function and create three new columns ---
# Apply function row-wise and unpack the returned Series into three new columns
new_columns = ['Major_Category', 'Custom_Super_Category', 'Overlapping_Sites']
final_summary_df[new_columns] = final_summary_df.apply(
    lambda row: find_matching_info(row, lookup_dict),
    axis=1
)

print("🎉 Annotation complete!")


# --- Step 4: Display final results ---
print("\nPreview of final_summary_df with new classification and overlapping site columns:")

display_cols = ['TF', 'All_Sites', 'Major_Category', 'Custom_Super_Category', 'Overlapping_Sites']
# Using ZFP57 as an example again
print(final_summary_df[final_summary_df['TF'] == 'ZFP57'][display_cols].to_markdown(index=False))

# View the head of the full table
print("\nFull table head preview:")
print(final_summary_df[display_cols].head())

步骤1: 正在从 tf_var 创建一个高效的查找字典...
查找字典创建完成，包含 45106 个独特的 (基因, 位点) 条目。

步骤2 (增强版): 正在遍历 summary 表，收集分类和重叠位点...
🎉 注释完成！

带有新分类和重叠位点列的 final_summary_df 预览:
| TF   | All_Sites   | Major_Category   | Custom_Super_Category   | Overlapping_Sites   |
|------|-------------|------------------|-------------------------|---------------------|

完整表格头部预览:
      TF                                          All_Sites Major_Category  \
0   ATF1  E242, E249, E258, E39, I256, K234, K263, K269,...            nan   
1   ATF2  A366, A384, A414, A469, A473, C418, D40, D417,...            nan   
2   ATF3  A100, C103, C153, D165, E117, E122, E138, E166...            nan   
3   ATF7  A364, A394, A446, A450, C398, E361, E377, E384...                  
4  BACH1  A628, A629, A643, C581, C621, C646, D132, D507...            nan   

  Custom_Super_Category             Overlapping_Sites  
0                   nan               R244, R61, V238  
1                   nan                          A469  
2                   na

In [335]:
final_summary_df

,TF,All_Sites,Major_Category,Custom_Super_Category,Overlapping_Sites
0,ATF1,"E242, E249, E258, E39, I256, K234, K263, K269,...",nan,nan,"R244, R61, V238"
1,ATF2,"A366, A384, A414, A469, A473, C418, D40, D417,...",nan,nan,A469
2,ATF3,"A100, C103, C153, D165, E117, E122, E138, E166...",nan,nan,"K136, L146, N125, Q115, T111"
3,ATF7,"A364, A394, A446, A450, C398, E361, E377, E384...",,,
4,BACH1,"A628, A629, A643, C581, C621, C646, D132, D507...",nan,nan,"C581, L610, R575"
...,...,...,...,...,...
105,ZNF322,"D161, E159",nan,nan,E159
106,ZNF324,"C259, D271, E145, E254, E258, E282, F257, G150...",nan,nan,"D271, E254, E282, F257, G150, R260, S269"
107,ZNF341,"A2, A360, A554, D635, E10, E578, F359, F458, F...","hematologic disorder, hereditary disease, immu...","Autoimmune, Other, nan","A2, D635, E10, F458, F654, H368, K461, L129, M..."
108,ZNF350,"A28, C323, C332, C416, D223, E251, E315, E322,...",nan,nan,"E375, F215, N531, R276, S377, T199, T529, V524"


In [336]:
final_summary_df.to_csv('./common.csv')

## Export specific hematopoietic variants

In [12]:
import pandas as pd
import numpy as np
from collections import defaultdict

# --- Assuming final_summary_df and tf_var are DataFrames you have already loaded ---
# final_summary_df = ...
# tf_var = ...
# And tf_var already has a column named 'site'

# --- Step 1: Create an efficient lookup dictionary (Same as before) ---
print("Step 1: Creating an efficient lookup dictionary from tf_var...")

# Ensure category columns are strings and handle NaNs
tf_var['Major_Category'] = tf_var['Major_Category'].astype(str).fillna('')
tf_var['Custom_Super_Category'] = tf_var['Custom_Super_Category'].astype(str).fillna('')
lookup_dict = {}

# Iterate through tf_var to populate the dictionary
for _, row in tf_var.dropna(subset=['GeneSymbol', 'site']).iterrows():
    key = (row['GeneSymbol'], row['site'])
    # Split categories by semicolon and strip whitespace
    major_cats = {cat.strip() for cat in row['Major_Category'].split(';') if cat.strip()}
    super_cats = {cat.strip() for cat in row['Custom_Super_Category'].split(';') if cat.strip()}
    
    if key in lookup_dict:
        lookup_dict[key]['major'].update(major_cats)
        lookup_dict[key]['super'].update(super_cats)
    else:
        lookup_dict[key] = {'major': major_cats, 'super': super_cats}

print(f"Lookup dictionary created, containing {len(lookup_dict)} unique (Gene, Site) entries.\n")


# --- Step 2 (Enhanced): Define function to collect classifications and overlapping sites (Same as before) ---
print("Step 2 (Enhanced): Iterating through summary table to collect classifications and overlapping sites...")

def find_matching_info(row, lookup):
    tf_name = row['TF']
    sites_string = row['All_Sites']
    
    if not isinstance(sites_string, str):
        return pd.Series(['', '', '']) 
        
    sites_list = {site.strip() for site in sites_string.split(',')}
    
    found_major_categories = set()
    found_super_categories = set()
    overlapping_sites = set()
    
    for site in sites_list:
        match = lookup.get((tf_name, site))
        if match:
            found_major_categories.update(match['major'])
            found_super_categories.update(match['super'])
            overlapping_sites.add(site)
            
    major_str = ', '.join(sorted(list(found_major_categories)))
    super_str = ', '.join(sorted(list(found_super_categories)))
    overlap_str = ', '.join(sorted(list(overlapping_sites)))
    
    return pd.Series([major_str, super_str, overlap_str])


# --- Step 3: Apply the function and create three new columns (Same as before) ---
new_columns = ['Major_Category', 'Custom_Super_Category', 'Overlapping_Sites']
final_summary_df[new_columns] = final_summary_df.apply(
    lambda row: find_matching_info(row, lookup_dict),
    axis=1
)

print("🎉 Annotation complete!")


# ======================= [New Output Module - Precise Version] =======================
# --- New Step: Filter and output molecules and sites related to 'hematologic disorder' found in [Overlapping Sites] ---
print("\n--- Identifying molecules and sites related to 'hematologic disorder' from [Overlapping Sites] ---")

hematologic_hits = defaultdict(set)

# Iterate through the annotated final_summary_df
for _, row in final_summary_df.iterrows():
    tf = row['TF']
    overlapping_sites_str = row['Overlapping_Sites']
    
    # Ensure the Overlapping_Sites column is not empty
    if not overlapping_sites_str or pd.isna(overlapping_sites_str):
        continue
        
    # Split the overlapping sites string into a list
    overlapping_sites_list = {site.strip() for site in overlapping_sites_str.split(',')}
    
    # Iterate through [each overlapping site] of this molecule
    for site in overlapping_sites_list:
        # Use lookup_dict to check the classification of this specific (Molecule, Overlapping Site) pair
        match = lookup_dict.get((tf, site))
        if match and 'hematologic disorder' in match['major']:
            # If the classification includes 'hematologic disorder', record it
            hematologic_hits[tf].add(site)

print("Identification complete. The following are [Overlapping] molecules and sites related to 'hematologic disorder':")
if not hematologic_hits:
    print("No molecule/site combinations related to 'hematologic disorder' were found in [Overlapping Sites].")
else:
    # Sort by molecule name alphabetically and print results
    for tf, sites in sorted(hematologic_hits.items()):
        sites_str = ', '.join(sorted(list(sites)))
        print(f"Molecule (TF): {tf:<10} | Related Overlapping Sites: {sites_str}")
# =================================================================


# --- Step 4: Display final results (Same as before) ---
print("\nPreview of final_summary_df with new classification and overlapping site columns:")

display_cols = ['TF', 'All_Sites', 'Major_Category', 'Custom_Super_Category', 'Overlapping_Sites']
# Using ZFP57 as an example again
print(final_summary_df[final_summary_df['TF'] == 'ZFP57'][display_cols].to_markdown(index=False))

# View the head of the full table
print("\nFull table head preview:")
print(final_summary_df[display_cols].head())

步骤1: 正在从 tf_var 创建一个高效的查找字典...
查找字典创建完成，包含 45106 个独特的 (基因, 位点) 条目。

步骤2 (增强版): 正在遍历 summary 表，收集分类和重叠位点...
🎉 注释完成！

--- 正在从【重叠位点】中识别与 'hematologic disorder' 相关的分子及位点 ---
识别完成。以下是与 'hematologic disorder' 相关的【重叠】分子及位点：
分子 (TF): KLF1       | 相关重叠位点: M39
分子 (TF): RXRA       | 相关重叠位点: S224
分子 (TF): TCF3       | 相关重叠位点: L602
分子 (TF): TP53       | 相关重叠位点: S240
分子 (TF): ZNF341     | 相关重叠位点: Y549

带有新分类和重叠位点列的 final_summary_df 预览:
| TF   | All_Sites   | Major_Category   | Custom_Super_Category   | Overlapping_Sites   |
|------|-------------|------------------|-------------------------|---------------------|

完整表格头部预览:
      TF                                          All_Sites Major_Category  \
0   ATF1  E242, E249, E258, E39, I256, K234, K263, K269,...            nan   
1   ATF2  A366, A384, A414, A469, A473, C418, D40, D417,...            nan   
2   ATF3  A100, C103, C153, D165, E117, E122, E138, E166...            nan   
3   ATF7  A364, A394, A446, A450, C398, E361, E377, E384...            

## spe

In [340]:
spe_site = pd.read_csv('./result/spe_1888_sites.csv', index_col=0)[['TF_Pair','TF1_Intersection','TF2_Intersection']].sort_values('TF_Pair')
spe_site

,TF_Pair,TF1_Intersection,TF2_Intersection
4,AHR_EBF1,"F83, K80, L87, N111, P55, Y76","A382, E387, I400, L359, L379, L385, L389, Q397..."
5,AHR_HOXA10,"D152, R225",NaN
0,AHR_MEF2A,R225,NaN
9,AHR_MEF2C,"L67, R71",NaN
8,AHR_MEF2D,"D152, E160, F234, H39, L43, P35, R225",K521
...,...,...,...
1829,ZNF528_ZNF827,"A28, C24, D12, D35, E20, E21, E39, G48, G6, I1...","C1068, D921, D935, F1055, H900, V920, W925"
1830,ZNF554_ZNF784,NaN,NaN
1831,ZNF586_ZNF713,"A21, A35, D19, E27, E28, E46, F24, I51, L31, L...","A38, A52, D36, D40, E45, E63, F41, K54, L32, L..."
1832,ZNF740_ZNF8,"D170, H143, M27, Q140, R146, R168","D43, H60, L49, Q46, V30, V32, V53"


In [344]:
cell = pd.read_csv('../get/tf/tf_tf/specific_filtered_1888.csv', index_col=0).sort_values('tf_tf')
cell

,motif_motif,tf_tf,cell_type,score,tf1,tf2,tf1_exp,tf2_exp,tf1_ratio,tf2_ratio
51,AHR_EBF1,AHR_EBF1,b-cell_lineage,0.033615,AHR,EBF1,2.141253,3.226498,1.023211,1.682020
143,AHR_HD/18,AHR_HOXA10,b-cell_lineage,0.028071,AHR,HOXA10,2.141253,1.228779,1.023211,1.805473
253,AHR_MEF2,AHR_MEF2A,b-cell_lineage,0.074874,AHR,MEF2A,2.141253,2.537933,1.023211,1.010759
252,AHR_MEF2,AHR_MEF2C,b-cell_lineage,0.074874,AHR,MEF2C,2.141253,2.714232,1.023211,1.088819
251,AHR_MEF2,AHR_MEF2D,b-cell_lineage,0.074874,AHR,MEF2D,2.141253,1.932744,1.023211,1.043751
...,...,...,...,...,...,...,...,...,...,...
59339,ZNF354_ZNF528,ZNF528_ZNF827,b-cell_lineage,0.034771,ZNF528,ZNF827,0.998403,2.088145,1.059738,1.139400
59425,ZNF554_ZNF784,ZNF554_ZNF784,b-cell_lineage,0.054391,ZNF554,ZNF784,0.776673,0.542801,1.277811,1.009132
59438,ZNF586_ZNF713,ZNF586_ZNF713,pdc,0.014520,ZNF586,ZNF713,1.785301,1.524361,1.030761,1.092271
59030,ZIC/2_ZNF85,ZNF740_ZNF8,b-cell_lineage,0.025463,ZNF740,ZNF8,1.112305,1.288351,1.016900,1.141982


In [346]:
spe_site['cell_type'] = cell['cell_type'].tolist()
spe_site

,TF_Pair,TF1_Intersection,TF2_Intersection,cell_type
4,AHR_EBF1,"F83, K80, L87, N111, P55, Y76","A382, E387, I400, L359, L379, L385, L389, Q397...",b-cell_lineage
5,AHR_HOXA10,"D152, R225",NaN,b-cell_lineage
0,AHR_MEF2A,R225,NaN,b-cell_lineage
9,AHR_MEF2C,"L67, R71",NaN,b-cell_lineage
8,AHR_MEF2D,"D152, E160, F234, H39, L43, P35, R225",K521,b-cell_lineage
...,...,...,...,...
1829,ZNF528_ZNF827,"A28, C24, D12, D35, E20, E21, E39, G48, G6, I1...","C1068, D921, D935, F1055, H900, V920, W925",b-cell_lineage
1830,ZNF554_ZNF784,NaN,NaN,b-cell_lineage
1831,ZNF586_ZNF713,"A21, A35, D19, E27, E28, E46, F24, I51, L31, L...","A38, A52, D36, D40, E45, E63, F41, K54, L32, L...",pdc
1832,ZNF740_ZNF8,"D170, H143, M27, Q140, R146, R168","D43, H60, L49, Q46, V30, V32, V53",b-cell_lineage


In [347]:
list_of_dataframes = [group for _, group in spe_site.groupby('cell_type')]

if list_of_dataframes:
    print("(cell_type: {})：".format(list_of_dataframes[0]['cell_type'].iloc[0]))
    print(list_of_dataframes[0].head())

拆分后的第一个 DataFrame (cell_type: b-cell_lineage)：
      TF_Pair                       TF1_Intersection  \
4    AHR_EBF1          F83, K80, L87, N111, P55, Y76   
5  AHR_HOXA10                             D152, R225   
0   AHR_MEF2A                                   R225   
9   AHR_MEF2C                               L67, R71   
8   AHR_MEF2D  D152, E160, F234, H39, L43, P35, R225   

                                    TF2_Intersection       cell_type  
4  A382, E387, I400, L359, L379, L385, L389, Q397...  b-cell_lineage  
5                                                NaN  b-cell_lineage  
0                                                NaN  b-cell_lineage  
9                                                NaN  b-cell_lineage  
8                                               K521  b-cell_lineage  


In [348]:
list_of_dataframes

[            TF_Pair                                   TF1_Intersection  \
 4          AHR_EBF1                      F83, K80, L87, N111, P55, Y76   
 5        AHR_HOXA10                                         D152, R225   
 0         AHR_MEF2A                                               R225   
 9         AHR_MEF2C                                           L67, R71   
 8         AHR_MEF2D              D152, E160, F234, H39, L43, P35, R225   
 ...             ...                                                ...   
 1828  ZNF528_ZNF768  D12, E39, F17, I15, I49, L32, L47, L8, M37, N4...   
 1829  ZNF528_ZNF827  A28, C24, D12, D35, E20, E21, E39, G48, G6, I1...   
 1830  ZNF554_ZNF784                                                NaN   
 1832    ZNF740_ZNF8                  D170, H143, M27, Q140, R146, R168   
 1833   ZNF740_ZNF85                                           F93, Q35   
 
                                        TF2_Intersection       cell_type  
 4     A382, E387, I400

In [349]:
len(list_of_dataframes)

14

In [361]:
new_columns = ['Major_Category', 'Custom_Super_Category', 'Overlapping_Sites']

for i in range(14):
    print(i)
    tidy_df = list_of_dataframes[i].reset_index().melt(
    id_vars=['TF_Pair'],
    value_vars=['TF1_Intersection', 'TF2_Intersection'],
    var_name='Source_Column',
    value_name='Site_String'
    ).dropna(subset=['Site_String']).assign(
    TF = lambda dfx: np.where(
        dfx['Source_Column'].str.contains('TF1'), 
        dfx['TF_Pair'].str.split('_').str[0], 
        dfx['TF_Pair'].str.split('_').str[1]
    ),
    Site = lambda dfx: dfx['Site_String'].str.split(',\s*')
    ).explode('Site')[['TF', 'Site']].drop_duplicates()

    final_summary_df = tidy_df.groupby('TF')['Site'].agg(
        lambda sites: ', '.join(sorted(sites.unique()))
    ).reset_index(name='All_Sites')
    final_summary_df[new_columns] = final_summary_df.apply(
            lambda row: find_matching_info(row, lookup_dict),
            axis=1
    )
    final_summary_df.to_csv('./result/spe_final/'+list_of_dataframes[i]['cell_type'].tolist()[0]+'.csv')

<>:18: SyntaxWarning: invalid escape sequence '\s'
<>:18: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_1492626/1724453253.py:18: SyntaxWarning: invalid escape sequence '\s'
  Site = lambda dfx: dfx['Site_String'].str.split(',\s*')


0
1
2
3
4
5
6
7
8
9
10
11
12
13


In [362]:
spe_site['cell_type'].value_counts()

cell_type
b-cell_lineage    660
macrophages       535
dc                194
cycling_cells     126
mast_cells         90
pdc                78
early_mk           69
plasma_cells       36
erythroid          31
t_cells            26
ilc                24
b_cells             7
hsc_mpp             7
monocytes           5
Name: count, dtype: int64

In [363]:
spe_site[spe_site['cell_type'] == 'monocytes']

,TF_Pair,TF1_Intersection,TF2_Intersection,cell_type
220,CEBPB_RXRA,NaN,NaN,monocytes
221,CEBPD_RXRA,NaN,"E453, E456",monocytes
228,CEBPG_RXRA,"H125, L121",NaN,monocytes
340,DBP_RXRA,"A27, G30, G37, L28, L29, L31, L34, L35, Q36, R...","D295, E281, E453, E456, F277, F289, F450, K284...",monocytes
1371,NFIL3_RXRA,NaN,NaN,monocytes


# Visual

In [1]:
import pandas as pd

## Original draw

In [2]:
filtered_df = pd.read_csv('./tf/tf_tf/common_filtered_339.csv', index_col=0)
filtered_df

,tf_tf,tf1,tf2,score,motif_motif,tf1_sum,tf1_non_zero,tf2_sum,tf2_non_zero
1597,ATF1_ATF7,ATF1,ATF7,10.277222,CREB/ATF/1_CREB/ATF/2,22.700978,14,25.669878,14
1599,ATF1_CREB3,ATF1,CREB3,10.277222,CREB/ATF/1_CREB/ATF/2,22.700978,14,12.737690,14
1600,ATF1_CREB5,ATF1,CREB5,10.277222,CREB/ATF/1_CREB/ATF/2,22.700978,14,19.327195,14
1605,ATF1_JDP2,ATF1,JDP2,10.277222,CREB/ATF/1_CREB/ATF/2,22.700978,14,14.033428,14
1606,ATF1_MAFB,ATF1,MAFB,10.277222,CREB/ATF/1_CREB/ATF/2,22.700978,14,11.760669,14
...,...,...,...,...,...,...,...,...,...
32134,ZBTB17_ZNF770,ZBTB17,ZNF770,14.069301,GC-tract_KLF/SP/2,19.660017,14,20.516318,14
32137,ZBTB18_ZEB1,ZBTB18,ZEB1,10.912522,Ebox/CACCTG_Ebox/CAGCTG,18.160063,14,33.031295,14
32269,ZNF263_ZNF281,ZNF263,ZNF281,14.069301,GC-tract_KLF/SP/2,18.387812,14,19.246573,14
32282,ZNF281_ZNF341,ZNF281,ZNF341,14.069301,GC-tract_KLF/SP/2,19.246573,14,13.358270,14


In [3]:
# Get the unique TFs from 'tf1' and 'tf2' columns
unique_tfs = pd.unique(filtered_df[['tf1', 'tf2']].values.ravel('K'))

# Count the number of unique TFs
num_unique_tfs = len(unique_tfs)
num_unique_tfs

116

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# Set font and PDF output parameters
plt.rcParams['pdf.fonttype'] = 42      # Ensure text is embedded as TrueType fonts
plt.rcParams['ps.fonttype'] = 42       # If PostScript output is required

# Construct the graph
G = nx.from_pandas_edgelist(filtered_df, "tf1", "tf2", edge_attr="score")

# Calculate node centrality
centrality = nx.degree_centrality(G)

# Normalize node sizes (reduce range to avoid overlap)
node_sizes = np.interp(
    list(centrality.values()),
    (min(centrality.values()), max(centrality.values())),
    (100, 800)
)

# Edge color mapping
scores = [d['score'] for (_, _, d) in G.edges(data=True)]
norm = plt.Normalize(vmin=min(scores), vmax=max(scores))
edge_colors = cm.plasma(norm(scores))

# Improve layout: increase k value or use kamada_kawai
pos = nx.spring_layout(G, k=2.0, seed=42)
# pos = nx.kamada_kawai_layout(G)  # Alternative layout option

fig, ax = plt.subplots(figsize=(12, 10))

# Draw nodes
nx.draw_networkx_nodes(
    G, pos,
    node_size=node_sizes,
    node_color="skyblue",
    alpha=0.9,
    edgecolors="black",
    ax=ax
)

# Draw edges
nx.draw_networkx_edges(
    G, pos,
    edge_color=edge_colors,
    width=np.interp(scores, (min(scores), max(scores)), (0.5, 2)),
    alpha=0.3,
    ax=ax
)

# Labels (Show top 50, add offset to avoid overlapping nodes)
top_nodes = sorted(centrality, key=centrality.get, reverse=True)[:50]
for n in top_nodes:
    x, y = pos[n]
    ax.text(
        x, y+0.05,  # Shift upwards
        n,
        fontsize=9,
        ha="center", va="center",
        bbox=dict(facecolor="white", alpha=0.6, edgecolor="none", pad=1)  # White background to prevent overlap with edges
    )

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=cm.plasma, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label("Edge Score")

ax.set_title("Improved PPI Network (Reduced Overlap)", fontsize=14)
ax.axis("off")
# plt.savefig("ppi_network.pdf", format="pdf", bbox_inches="tight")

plt.show()

In [6]:
dis = pd.read_csv('../af3/result/com_final.csv', index_col=0)
dis['Overlapping_Sites_Count'] = dis['Overlapping_Sites'].str.split(', ').str.len().fillna(0).astype(int)
dis

,TF,All_Sites,Major_Category,Custom_Super_Category,Overlapping_Sites,Overlapping_Sites_Count
0,ATF1,"E242, E249, E258, E39, I256, K234, K263, K269,...",NaN,NaN,"R244, R61, V238",3
1,ATF2,"A366, A384, A414, A469, A473, C418, D40, D417,...",NaN,NaN,A469,1
2,ATF3,"A100, C103, C153, D165, E117, E122, E138, E166...",NaN,NaN,"K136, L146, N125, Q115, T111",5
3,ATF7,"A364, A394, A446, A450, C398, E361, E377, E384...",NaN,NaN,NaN,0
4,BACH1,"A628, A629, A643, C581, C621, C646, D132, D507...",NaN,NaN,"C581, L610, R575",3
...,...,...,...,...,...,...
105,ZNF322,"D161, E159",NaN,NaN,E159,1
106,ZNF324,"C259, D271, E145, E254, E258, E282, F257, G150...",NaN,NaN,"D271, E254, E282, F257, G150, R260, S269",7
107,ZNF341,"A2, A360, A554, D635, E10, E578, F359, F458, F...","hematologic disorder, hereditary disease, immu...","Autoimmune, Other, nan","A2, D635, E10, F458, F654, H368, K461, L129, M...",24
108,ZNF350,"A28, C323, C332, C416, D223, E251, E315, E322,...",NaN,NaN,"E375, F215, N531, R276, S377, T199, T529, V524",8


In [7]:
set(unique_tfs) - set(dis['TF'])

{'ERG', 'FLI1', 'NFIX', 'RUNX3', 'STAT3', 'TGIF2'}

## Add disease information

In [8]:
import pandas as pd
from collections import Counter
import numpy as np
def parse_categories(category_string):
    if pd.isna(category_string) or category_string.lower() == 'nan':
        return Counter()
    categories = [cat.strip() for cat in category_string.split(',')]
    valid_categories = [cat for cat in categories if cat.lower() != 'nan' and cat]
    return Counter(valid_categories)

dis['Category_Counts'] = dis['Major_Category'].apply(parse_categories)

In [9]:
dis

,TF,All_Sites,Major_Category,Custom_Super_Category,Overlapping_Sites,Overlapping_Sites_Count,Category_Counts
0,ATF1,"E242, E249, E258, E39, I256, K234, K263, K269,...",NaN,NaN,"R244, R61, V238",3,{}
1,ATF2,"A366, A384, A414, A469, A473, C418, D40, D417,...",NaN,NaN,A469,1,{}
2,ATF3,"A100, C103, C153, D165, E117, E122, E138, E166...",NaN,NaN,"K136, L146, N125, Q115, T111",5,{}
3,ATF7,"A364, A394, A446, A450, C398, E361, E377, E384...",NaN,NaN,NaN,0,{}
4,BACH1,"A628, A629, A643, C581, C621, C646, D132, D507...",NaN,NaN,"C581, L610, R575",3,{}
...,...,...,...,...,...,...,...
105,ZNF322,"D161, E159",NaN,NaN,E159,1,{}
106,ZNF324,"C259, D271, E145, E254, E258, E282, F257, G150...",NaN,NaN,"D271, E254, E282, F257, G150, R260, S269",7,{}
107,ZNF341,"A2, A360, A554, D635, E10, E578, F359, F458, F...","hematologic disorder, hereditary disease, immu...","Autoimmune, Other, nan","A2, D635, E10, F458, F654, H368, K461, L129, M...",24,"{'hematologic disorder': 1, 'hereditary diseas..."
108,ZNF350,"A28, C323, C332, C416, D223, E251, E315, E322,...",NaN,NaN,"E375, F215, N531, R276, S377, T199, T529, V524",8,{}


In [10]:
dis.to_csv('./tf/tf_tf/com_tf_dis.csv')

In [11]:
dis['Overlapping_Sites_Count'].min()

0

In [12]:
dis['Overlapping_Sites_Count'].max()

32

In [124]:
all_categories = set()
for counter in dis['Category_Counts']:
    all_categories.update(counter.keys())

colors = cm.get_cmap('tab20', len(all_categories)) 
category_color_map = {category: colors(i) for i, category in enumerate(all_categories)}

default_node_color = 'lightgrey'

/tmp/ipykernel_209841/3032408599.py:7: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colors = cm.get_cmap('tab20', len(all_categories)) # 'tab20' 是一个适合分类数据的色板


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.patches as mpatches  # Used for the legend

# Assuming 'dis' and 'filtered_df' are already defined
# ... (Your data loading and preprocessing code) ...

# ==============================================================================
# Steps 1 & 2: Data Preprocessing and Color Mapping (Kept unchanged)
def parse_categories(category_string):
    if pd.isna(category_string) or category_string.lower() == 'nan':
        return Counter()
    # Split by comma and strip whitespace
    categories = [cat.strip() for cat in category_string.split(',')]
    # Filter out empty strings or 'nan' strings
    valid_categories = [cat for cat in categories if cat.lower() != 'nan' and cat]
    return Counter(valid_categories)

# Apply parsing to the dataframe
dis['Category_Counts'] = dis['Major_Category'].apply(parse_categories)

# Collect all unique categories
all_categories = set()
for counter in dis['Category_Counts']:
    all_categories.update(counter.keys())

# Generate color map
if all_categories:
    colors = cm.get_cmap('tab20', len(all_categories))
    category_color_map = {category: colors(i) for i, category in enumerate(all_categories)}
else:
    category_color_map = {}
default_node_color = 'lightgrey'
# ==============================================================================

# Set fonts to ensure compatibility with PDF output (embedding TrueType fonts)
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# Construct the graph
G = nx.from_pandas_edgelist(filtered_df, "tf1", "tf2", edge_attr="score")

# --- Node Size/Radius Settings (Kept unchanged) ---
# Map TFs to their overlapping sites count
tf_to_count_map = dis.set_index('TF')['Overlapping_Sites_Count']
# Align with graph nodes, fill missing with 0
raw_node_sizes = tf_to_count_map.reindex(G.nodes(), fill_value=0)

# Calculate radius interpolation
min_val, max_val = raw_node_sizes.min(), raw_node_sizes.max()
if min_val == max_val:
    node_radii = pd.Series([0.05 if size > 0 else 0 for size in raw_node_sizes], index=raw_node_sizes.index)
else:
    radii_values = np.interp(raw_node_sizes, (min_val, max_val), (0.02, 0.1))
    node_radii = pd.Series(radii_values, index=raw_node_sizes.index)

# --- Plotting Settings ---
# Calculate layout
pos = nx.spring_layout(G, k=2.0, seed=42)
fig, ax = plt.subplots(figsize=(16, 12))

# --- Draw Edges ---
scores = [d['score'] for (_, _, d) in G.edges(data=True)]
nx.draw_networkx_edges(
    G, pos,
    edge_color='#1976d2',
    width=np.interp(scores, (min(scores), max(scores)), (0.5, 2)),
    alpha=0.3,
    ax=ax
)

# --- Draw Nodes as Pie Charts (Kept unchanged) ---
# Create a mapping for easy access inside the loop
tf_to_categories = dis.set_index('TF')['Category_Counts']

for node in G.nodes():
    x, y = pos[node]
    radius = node_radii.get(node, 0.01)
    categories = tf_to_categories.get(node)

    # Determine pie chart data
    if not categories:
        pie_labels = ['default']
        pie_sizes = [1]
        pie_colors = [default_node_color]
    else:
        pie_labels = list(categories.keys())
        pie_sizes = list(categories.values())
        pie_colors = [category_color_map[label] for label in pie_labels]

    # Draw pie chart using inset_axes
    pie_ax = ax.inset_axes([x - radius, y - radius, 2 * radius, 2 * radius],
                           transform=ax.transData)
    pie_ax.pie(pie_sizes, colors=pie_colors, wedgeprops={'edgecolor': 'black', 'linewidth': 0.5})

# --- Draw Labels (Kept unchanged) ---
# Select top 50 largest nodes and any node that has a category assigned
top_50_set = set(raw_node_sizes.nlargest(50).index)
nodes_with_category_set = set(dis[pd.notna(dis['Custom_Super_Category'])]['TF'])
final_nodes_to_label = top_50_set.union(nodes_with_category_set)

for n in final_nodes_to_label:
    if n in pos:
        x, y = pos[n]
        ax.text(
            x, y + node_radii.get(n, 0.02) * 1.1, # Shift text slightly above the node
            n,
            fontsize=10,
            ha="center", va="bottom",
            color='black',
            zorder=10,
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1)
        )

# ======================= [Key Modification: Sorted Legend] =======================
# --- Create and Add Legend ---
if category_color_map:
    # 1. Get (key, value) pairs from the dictionary and sort by key (category label)
    sorted_legend_items = sorted(category_color_map.items(), key=lambda item: item[0])
    
    # 2. Create list of legend_patches based on sorted results
    legend_patches = [mpatches.Patch(color=color, label=label) for label, color in sorted_legend_items]
    
    # 3. Create the legend
    ax.legend(handles=legend_patches, title="Categories", bbox_to_anchor=(1.05, 1), loc='upper left')
# =====================================================================

ax.set_title("PPI Network (Nodes as Pie Charts by Category)", fontsize=16)
ax.axis("off")
fig.tight_layout()

plt.savefig("ppi_network.pdf", format="pdf", bbox_inches="tight")
plt.show()

## Check if the node needs to be removed

In [1]:
import pandas as pd

In [2]:
dis = pd.read_csv('../af3/result/com_final.csv', index_col=0)
dis['Overlapping_Sites_Count'] = dis['Overlapping_Sites'].str.split(', ').str.len().fillna(0).astype(int)
dis = dis[['TF', 'Overlapping_Sites_Count']]
dis

,TF,Overlapping_Sites_Count
0,ATF1,3
1,ATF2,1
2,ATF3,5
3,ATF7,0
4,BACH1,3
...,...,...
105,ZNF322,1
106,ZNF324,7
107,ZNF341,24
108,ZNF350,8


In [3]:
new_tfs = {
    'TF': ['ERG', 'FLI1', 'NFIX', 'RUNX3', 'STAT3', 'TGIF2'],
    'Overlapping_Sites_Count': [0, 0, 0, 0, 0, 0]
}
new_rows = pd.DataFrame(new_tfs)

In [4]:
dis_updated = pd.concat([dis, new_rows], ignore_index=True)

In [5]:
dis_updated

,TF,Overlapping_Sites_Count
0,ATF1,3
1,ATF2,1
2,ATF3,5
3,ATF7,0
4,BACH1,3
...,...,...
111,FLI1,0
112,NFIX,0
113,RUNX3,0
114,STAT3,0


In [6]:
filtered_df = pd.read_csv('./tf/tf_tf/common_filtered_339.csv', index_col=0)
filtered_df

,tf_tf,tf1,tf2,score,motif_motif,tf1_sum,tf1_non_zero,tf2_sum,tf2_non_zero
1597,ATF1_ATF7,ATF1,ATF7,10.277222,CREB/ATF/1_CREB/ATF/2,22.700978,14,25.669878,14
1599,ATF1_CREB3,ATF1,CREB3,10.277222,CREB/ATF/1_CREB/ATF/2,22.700978,14,12.737690,14
1600,ATF1_CREB5,ATF1,CREB5,10.277222,CREB/ATF/1_CREB/ATF/2,22.700978,14,19.327195,14
1605,ATF1_JDP2,ATF1,JDP2,10.277222,CREB/ATF/1_CREB/ATF/2,22.700978,14,14.033428,14
1606,ATF1_MAFB,ATF1,MAFB,10.277222,CREB/ATF/1_CREB/ATF/2,22.700978,14,11.760669,14
...,...,...,...,...,...,...,...,...,...
32134,ZBTB17_ZNF770,ZBTB17,ZNF770,14.069301,GC-tract_KLF/SP/2,19.660017,14,20.516318,14
32137,ZBTB18_ZEB1,ZBTB18,ZEB1,10.912522,Ebox/CACCTG_Ebox/CAGCTG,18.160063,14,33.031295,14
32269,ZNF263_ZNF281,ZNF263,ZNF281,14.069301,GC-tract_KLF/SP/2,18.387812,14,19.246573,14
32282,ZNF281_ZNF341,ZNF281,ZNF341,14.069301,GC-tract_KLF/SP/2,19.246573,14,13.358270,14


In [7]:
import pandas as pd
import numpy as np # Import numpy for data processing

# --- Step 1: Create an efficient TF -> Count mapping ---
# Set the 'TF' column as the index, then select the 'Overlapping_Sites_Count' column to get a Series.
# This acts like a dictionary; you can retrieve the count value directly using the TF name, which is very fast.
tf_to_count_map = dis_updated.set_index('TF')['Overlapping_Sites_Count']

print("\n--- Created lookup mapping (partial) ---")
print(tf_to_count_map.head())


# --- Step 2: Define a function to calculate the sum of interaction partners' counts ---
def get_interaction_partners_sum(current_tf, interaction_df, lookup_map):
    """
    Calculates the sum of Overlapping_Sites_Count for all interaction partners of a given TF.
    
    Parameters:
    - current_tf: The name of the current TF to calculate (str)
    - interaction_df: DataFrame containing interaction relationships (filtered_df)
    - lookup_map: Mapping of TF names to count values (pd.Series)
    """
    # Find partners where current_tf appears in the 'tf1' column (partners are in 'tf2')
    partners1 = interaction_df[interaction_df['tf1'] == current_tf]['tf2']
    
    # Find partners where current_tf appears in the 'tf2' column (partners are in 'tf1')
    partners2 = interaction_df[interaction_df['tf2'] == current_tf]['tf1']
    
    # Combine both sets of partners and remove duplicates
    all_partners = pd.concat([partners1, partners2]).unique()
    
    # If there are no partners, return 0
    if len(all_partners) == 0:
        return 0
        
    # Use .get(partner, 0) to safely retrieve the count from the map. 
    # If the partner is not in the map (e.g., it wasn't measured), it counts as 0.
    # Then sum the count values of all partners.
    total_sum = sum(lookup_map.get(partner, 0) for partner in all_partners)
    
    return total_sum

# --- Step 3: Apply the function to each row of dis_updated ---
# Call our function for every TF in the 'TF' column and store the result in a new column
dis_updated['Interaction_Partners_Count_Sum'] = dis_updated['TF'].apply(
    get_interaction_partners_sum,
    args=(filtered_df, tf_to_count_map) # Pass additional arguments
)


# --- Final Result ---
print("\n--- Final result after adding the new column ---")
print(dis_updated)


--- 创建的查找映射 (部分) ---
TF
ATF1     3
ATF2     1
ATF3     5
ATF7     0
BACH1    3
Name: Overlapping_Sites_Count, dtype: int64

--- 添加新列后的最终结果 ---
        TF  Overlapping_Sites_Count  Interaction_Partners_Count_Sum
0     ATF1                        3                              26
1     ATF2                        1                              26
2     ATF3                        5                              26
3     ATF7                        0                              26
4    BACH1                        3                              71
..     ...                      ...                             ...
111   FLI1                        0                               4
112   NFIX                        0                               8
113  RUNX3                        0                               0
114  STAT3                        0                              22
115  TGIF2                        0                               1

[116 rows x 3 columns]


In [8]:
dis_updated

,TF,Overlapping_Sites_Count,Interaction_Partners_Count_Sum
0,ATF1,3,26
1,ATF2,1,26
2,ATF3,5,26
3,ATF7,0,26
4,BACH1,3,71
...,...,...,...
111,FLI1,0,4
112,NFIX,0,8
113,RUNX3,0,0
114,STAT3,0,22


In [10]:
dis_updated[dis_updated['Overlapping_Sites_Count'] == 0]

,TF,Overlapping_Sites_Count,Interaction_Partners_Count_Sum
3,ATF7,0,26
9,CREM,0,26
11,ELF2,0,4
12,ELK4,0,4
14,ETS1,0,4
16,ETV5,0,4
23,IRF2,0,24
29,JUNB,0,26
33,KLF3,0,79
36,KLF6,0,79


In [11]:
dis_updated[dis_updated['Interaction_Partners_Count_Sum'] == 0]

,TF,Overlapping_Sites_Count,Interaction_Partners_Count_Sum
66,RUNX2,5,0
105,ZNF322,1,0
113,RUNX3,0,0


In [9]:
condition1 = dis_updated['Overlapping_Sites_Count'] == 0

condition2 = dis_updated['Interaction_Partners_Count_Sum'] == 0

zero_rows = dis_updated[condition1 & condition2]

print("\n--- 筛选出的两列全为0的行 ---")
print(zero_rows)


--- 筛选出的两列全为0的行 ---
        TF  Overlapping_Sites_Count  Interaction_Partners_Count_Sum
113  RUNX3                        0                               0
